In [ ]:
# Alpha Connectome, sweep 10a: one model per instrument group. One A100 run (~3.5 h), self-contained (the whole
# pipeline is embedded below). Needs from earlier sweeps in Drive: runs/sweep9/s9_h128 (best student, the start of
# the fine-tunes), runs/cache/t_9.npy (teacher 9), runs/cache/mask_train.npy, runs/mask_model/split.json.
#   Phase A (-> runs/sweep10a): s9_h128 fine-tuned 6 passes at lr 5e-4, half teacher 9 / half true targets, on all
#            groups (ft_all, the control), on each group alone (ft_g1..ft_g4), and as a group-conditioned model
#            (ft_film: FiLM inputs + 4 heads). Group 4 alone from scratch on the true targets only (no teacher: the
#            teacher learned group 4 together with the other groups), 5 passes, h128 and h192+dropout: the same
#            recipe as the sweep 9 teachers t9_h128_s0..s3 / t9_h192_drop_s0..s1, so their group-4 scores compare.
#   Phase B: per-group scores of everything (incl. those sweep 9 teachers), per group the best option on validation
#            half A (single model, blend with s9_h128, or weights halfway to s9_h128), routed score on half B (true
#            and detected groups), export the routed submission (runs/sweep10a_routed), zips, speed test.
# Re-runnable: finished models are skipped. Keep this tab open while it runs.
# rclone version: runs in any Colab account and reads/writes YOUR Drive (My Drive/wunder-connectome/runs) through
# rclone, so the Colab account's own Drive is not used. Once, on your Mac:  brew install rclone  then
#   rclone authorize "drive"
# sign in with YOUR Google account in the browser that opens, and copy everything it prints after "--->".
# Paste it when this cell asks (or store it as a Colab secret RCLONE_DRIVE_TOKEN: key icon on the left, notebook
# access on). It downloads only the inputs it needs (~2 GB), works on this machine's disk, and uploads its results
# every 10 minutes and at the end. The token can read your whole Drive: it stays on this runtime (deleted when the
# runtime ends); revoke it any time at myaccount.google.com/connections ("rclone").
import os, subprocess, sys, json, glob, shutil, time, threading

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    assert p.wait() == 0, 'command failed: ' + cmd[:80]

# ---- your own Google Drive through rclone (the Drive of the account running Colab is not used)
REMOTE = 'gd:wunder-connectome/runs'  # = My Drive/wunder-connectome/runs of the account you authorized on your Mac
OUT_DIR = '/content/runs'             # local working copy: what this sweep needs (downloaded) and makes (uploaded)
RC = 'rclone --transfers 8 --checkers 16 --drive-chunk-size 64M --retries 5 --low-level-retries 20 --stats-one-line --stats 60s'
if not shutil.which('rclone'):
    sh('cd /tmp && curl -sSLO https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -oq rclone-current-linux-amd64.zip '
       '&& cp rclone-v*-linux-amd64/rclone /usr/local/bin/ && rclone version | head -1')

def parse_token(s):
    """The text `rclone authorize "drive"` prints (a JSON token, or the newer base64 blob), with or without the
    'Paste the following ...' / '<---End paste' lines around it."""
    import base64
    s = s.strip()
    if '--->' in s: s = s.split('--->', 1)[1]
    s = s.split('<---')[0].strip()
    try:
        t = json.loads(s)
    except ValueError:
        b = s.replace('+', '-').replace('/', '_')
        t = json.loads(base64.urlsafe_b64decode(b + '=' * (-len(b) % 4)))
    if isinstance(t, dict) and 'token' in t and 'access_token' not in t:
        t = t['token']
        t = json.loads(t) if isinstance(t, str) else t
    assert isinstance(t, dict) and 'refresh_token' in t, 'that is not the token printed by rclone authorize "drive"'
    return json.dumps(t)

if REMOTE.startswith('gd:') and 'gd:' not in subprocess.run('rclone listremotes', shell=True, capture_output=True, text=True).stdout:
    tok = ''
    try:
        from google.colab import userdata  # Colab secret (key icon on the left), if you made one
        tok = userdata.get('RCLONE_DRIVE_TOKEN') or ''
    except Exception:
        pass
    if not tok:
        import getpass
        tok = getpass.getpass('Paste what  rclone authorize "drive"  printed on your Mac, then press Enter: ')
    conf = subprocess.run('rclone config file', shell=True, capture_output=True, text=True).stdout.strip().splitlines()[-1]
    os.makedirs(os.path.dirname(conf), exist_ok=True)
    with open(conf, 'a') as f:
        f.write(f'\n[gd]\ntype = drive\nscope = drive\ntoken = {parse_token(tok)}\n')
    os.chmod(conf, 0o600)
    del tok
sh(f'rclone about {REMOTE} || true; echo "in {REMOTE}:"; rclone lsf --dirs-only {REMOTE} | tr "\\n" " "; echo')

PULL = ['sweep9/summary.jsonl', 'sweep9/s9_h128/**', 'cache/t_9.npy', 'cache/mask_train.npy', 'mask_model/split.json',
        *[f'sweep9_teachers/{n}/best.pt' for n in ('t9_h128_s0', 't9_h128_s1', 't9_h128_s2', 't9_h128_s3',
                                                     't9_h192_drop_s0', 't9_h192_drop_s1')],
        'sweep10a/**', 'sweep10a_routed/**']  # the last two: what an earlier run of this notebook made (it resumes)
PUSH = ['sweep10a/**', 'sweep10a_routed/**', 'zips/sweep10a*']
inc = lambda pats: ' '.join(f"--include '{p}'" for p in pats)
os.makedirs(OUT_DIR, exist_ok=True)
print('downloading the inputs from your Drive', flush=True)
sh(f'{RC} copy {REMOTE} {OUT_DIR} {inc(PULL)} && du -sh {OUT_DIR}')

_sync_lock = threading.Lock()
def sync_up(quiet=False):
    """Upload what this sweep made (new or changed files only) to your Drive."""
    with _sync_lock:
        r = subprocess.run(f'{RC} copy {OUT_DIR} {REMOTE} {inc(PUSH)}', shell=True, capture_output=True, text=True)
    if r.returncode:
        print('UPLOAD TO DRIVE FAILED (retried at the next upload):', r.stderr[-600:], flush=True)
    elif not quiet:
        print(f'uploaded to Drive ({REMOTE})', flush=True)
    return r.returncode == 0

def drive_writable():
    """Upload what is done, and check the Drive still takes writes (storage full / token revoked) before training."""
    sync_up()
    r = subprocess.run(f'echo ok | rclone rcat {REMOTE}/.write_test && rclone deletefile {REMOTE}/.write_test',
                       shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('cannot write to your Drive through rclone (storage full? token revoked?): ' + r.stderr[-600:])
    print('Drive (rclone) is writable', flush=True)

def _upload_every(sec):
    while True:
        time.sleep(sec)
        sync_up(quiet=True)

drive_writable()
if not any(t.name == 'drive-upload' for t in threading.enumerate()):  # every 10 min, so a dropped runtime loses little
    threading.Thread(target=_upload_every, args=(600,), name='drive-upload', daemon=True).start()

sh('nvidia-smi --query-gpu=name,memory.total --format=csv; echo "CPUs: $(nproc)"; free -g | head -2')
DATA = r"""mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets"""
sh(DATA)
sh('pip -q install onnx onnxruntime')
os.chdir('/content/wnn')

# the pipeline code (pipeline/*.py from the Mac, 2026-11-04)
FILES = {'pipeline/__init__.py': '',
 'pipeline/data.py': '"""Fast sequence loading from the competition parquet files.\n'
                     '\n'
                     'Each row group is one 20k-step sequence. Readers keep one ParquetFile per thread\n'
                     '(re-opening parses a large footer) and write straight into a preallocated batch array.\n'
                     '"""\n'
                     'import threading\n'
                     'from concurrent.futures import ThreadPoolExecutor\n'
                     '\n'
                     'import numpy as np\n'
                     'import pyarrow.parquet as pq\n'
                     '\n'
                     'from utils import FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS, WARMUP\n'
                     '\n'
                     'N_FEATURES = len(FEATURE_COLUMNS)\n'
                     '\n'
                     '\n'
                     'class SeqStore:\n'
                     '    def __init__(self, path: str, n_threads: int = 4):\n'
                     '        self.path = path\n'
                     '        pf = pq.ParquetFile(path)\n'
                     '        self.n_seqs = pf.num_row_groups\n'
                     '        self.has_mask = "is_scored" in pf.schema_arrow.names\n'
                     '        self.cols = list(FEATURE_COLUMNS) + list(TARGET_COLUMNS) + (["is_scored"] if '
                     'self.has_mask else [])\n'
                     '        self._local = threading.local()\n'
                     '        self._pool = ThreadPoolExecutor(n_threads)\n'
                     '        self._cache = None\n'
                     '\n'
                     '    def enable_cache(self):\n'
                     '        """Keep every sequence in RAM after its first read (~9 MB each)."""\n'
                     '        self._cache = {}\n'
                     '\n'
                     '    def _pf(self):\n'
                     '        if not hasattr(self._local, "pf"):\n'
                     '            self._local.pf = pq.ParquetFile(self.path)\n'
                     '        return self._local.pf\n'
                     '\n'
                     '    def _load_into(self, i, g, X, Y, M):\n'
                     '        if self._cache is not None and g in self._cache:\n'
                     '            X[i], Y[i], M[i] = self._cache[g]\n'
                     '            return\n'
                     '        df = self._pf().read_row_group(int(g), columns=self.cols, '
                     'use_threads=False).to_pandas()\n'
                     '        a = df.to_numpy(np.float32)\n'
                     '        X[i] = a[:, :N_FEATURES]\n'
                     '        Y[i] = a[:, N_FEATURES:N_FEATURES + 2]\n'
                     '        need = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                     '        M[i] = need & (a[:, -1] > 0.5) if self.has_mask else need\n'
                     '        if self._cache is not None:\n'
                     '            self._cache[g] = (X[i].copy(), Y[i].copy(), M[i].copy())\n'
                     '\n'
                     '    def load(self, ids):\n'
                     '        """-> X [B,T,112] f32, Y [B,T,2] f32, M [B,T] bool (scored rows; all required rows if no '
                     'mask)."""\n'
                     '        B = len(ids)\n'
                     '        X = np.empty((B, SEQUENCE_LENGTH, N_FEATURES), np.float32)\n'
                     '        Y = np.empty((B, SEQUENCE_LENGTH, 2), np.float32)\n'
                     '        M = np.empty((B, SEQUENCE_LENGTH), bool)\n'
                     '        list(self._pool.map(lambda ig: self._load_into(ig[0], ig[1], X, Y, M), enumerate(ids)))\n'
                     '        return X, Y, M\n'
                     '\n'
                     '    def batches(self, ids, batch_size):\n'
                     '        """Yield loaded batches, prefetching the next one in the background."""\n'
                     '        chunks = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]\n'
                     '        if not chunks:\n'
                     '            return\n'
                     '        prefetch = ThreadPoolExecutor(1)\n'
                     '        fut = prefetch.submit(self.load, chunks[0])\n'
                     '        for k in range(len(chunks)):\n'
                     '            hold = [fut.result()]\n'
                     '            fut = prefetch.submit(self.load, chunks[k + 1]) if k + 1 < len(chunks) else None\n'
                     '            yield hold.pop()  # the generator keeps no reference, so the consumer can free the '
                     'batch\n'
                     '        prefetch.shutdown()\n'
                     '\n'
                     '\n'
                     'def split_train(n_seqs: int, n_holdout: int, seed: int = 0):\n'
                     '    """Deterministic split of train row groups into (train_ids, holdout_ids)."""\n'
                     '    perm = np.random.default_rng(seed).permutation(n_seqs)\n'
                     '    return np.sort(perm[n_holdout:]), np.sort(perm[:n_holdout])\n',
 'pipeline/ensemble.py': '"""Score models alone and averaged, pick an ensemble on the train holdout, optionally export '
                         'it.\n'
                         '\n'
                         '    python -m pipeline.ensemble --ckpts "runs/*/best.pt" --budgets 1 1.5 2 --export '
                         'runs/ensembles\n'
                         '\n'
                         'Every model is run once over the holdout and validation data (shared loading); combinations '
                         'are then\n'
                         'scored from the stored predictions. Members are rescaled to a common prediction spread '
                         'before\n'
                         'averaging (scales measured on the holdout). Selection uses the holdout only, so the '
                         'validation\n'
                         'numbers stay an honest check of the chosen combination.\n'
                         '\n'
                         'Cost = multiply-adds per row relative to the baseline (models.macs). Real run time also has '
                         'a\n'
                         "fixed per-row overhead, so an ensemble's measured time ratio is lower than its cost; check\n"
                         'candidates with pipeline.speed. One greedy ensemble is built and exported per --budgets '
                         'value.\n'
                         '"""\n'
                         'import argparse\n'
                         'import glob\n'
                         'import json\n'
                         'import os\n'
                         'import shutil\n'
                         'import time\n'
                         '\n'
                         'import numpy as np\n'
                         '\n'
                         'from utils import SEQUENCE_LENGTH, WARMUP, weighted_pearson\n'
                         '\n'
                         'from .data import SeqStore, split_train\n'
                         'from .fast_eval import load_ckpt, net_inputs, pick_device, predict_batch\n'
                         'from .model import BASELINE_MACS, EnsembleNet, export_onnx, feat_kind, load_baseline, macs\n'
                         '\n'
                         'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                         'NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                         '\n'
                         '\n'
                         'def load_member(path):\n'
                         '    if path == "baseline":\n'
                         '        return "baseline", load_baseline("baseline/baseline.onnx")\n'
                         '    return os.path.basename(os.path.dirname(path)), load_ckpt(path)\n'
                         '\n'
                         '\n'
                         'def collect(nets, store, ids, device, masks):\n'
                         '    """-> Y {mask: [N,2]}, P {mask: [K,N,2]} for masks \'all\' (required rows) and/or '
                         '\'scored\'."""\n'
                         '    ys = {m: [] for m in masks}\n'
                         '    ps = {m: [[] for _ in nets] for m in masks}\n'
                         '    for X, Y, M in store.batches(list(ids), 128):\n'
                         '        cache = {}  # extras once per batch and kind\n'
                         '        sel = {"all": np.broadcast_to(NEED, M.shape), "scored": M}\n'
                         '        for m in masks:\n'
                         '            ys[m].append(Y[sel[m]])\n'
                         '        for k, net in enumerate(nets):\n'
                         '            P = predict_batch(net, net_inputs(net, X, cache), device)\n'
                         '            for m in masks:\n'
                         '                ps[m][k].append(P[sel[m]])\n'
                         '    return ({m: np.concatenate(ys[m]) for m in masks},\n'
                         '            {m: np.stack([np.concatenate(p) for p in ps[m]]) for m in masks})\n'
                         '\n'
                         '\n'
                         'def wp(y, p):\n'
                         '    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))\n'
                         '\n'
                         '\n'
                         'def combine(P, scales, idx):\n'
                         '    return (P[idx] * scales[idx][:, None, :]).mean(0)\n'
                         '\n'
                         '\n'
                         'def main():\n'
                         '    ap = argparse.ArgumentParser()\n'
                         '    ap.add_argument("--ckpts", nargs="+", required=True, help="best.pt paths/globs, or '
                         '\'baseline\'")\n'
                         '    ap.add_argument("--holdout", type=int, default=500)\n'
                         '    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs '
                         '(default all)")\n'
                         '    ap.add_argument("--max-members", type=int, default=5)\n'
                         '    ap.add_argument("--budgets", type=float, nargs="+", default=[99.0],\n'
                         '                    help="max total cost (x baseline multiply-adds); one greedy ensemble per '
                         'budget")\n'
                         '    ap.add_argument("--export", help="root folder: writes <root>/cost<budget>/submission per '
                         'budget")\n'
                         '    ap.add_argument("--device", default="auto")\n'
                         '    ap.add_argument("--threads", type=int, default=4)\n'
                         '    ap.add_argument("--valid-split", help="split.json from mask_model: report on validation '
                         'half B")\n'
                         '    ap.add_argument("--select-on", choices=["holdout", "validA"], default="holdout",\n'
                         '                    help="what greedy selection maximises: holdout all-rows WP, or '
                         'validation half A scored rows "\n'
                         '                         "(needs --valid-split; right for mask-weighted models, keeps half B '
                         'honest)")\n'
                         '    args = ap.parse_args()\n'
                         '    assert args.select_on == "holdout" or args.valid_split, "--select-on validA needs '
                         '--valid-split"\n'
                         '\n'
                         '    paths = []\n'
                         '    for c in args.ckpts:\n'
                         '        paths += [c] if c == "baseline" else sorted(glob.glob(c))\n'
                         '    names, nets = zip(*[load_member(p) for p in paths])\n'
                         '    device = pick_device(args.device)\n'
                         '    nets = [n.eval().to(device) for n in nets]\n'
                         '    print(f"{len(nets)} models: {\', \'.join(names)}", flush=True)\n'
                         '\n'
                         '    t = time.time()\n'
                         '    train = SeqStore("datasets/train.parquet", args.threads)\n'
                         '    valid = SeqStore("datasets/valid.parquet", args.threads)\n'
                         '    _, hold_ids = split_train(train.n_seqs, args.holdout)\n'
                         '    Yh, Ph = collect(nets, train, hold_ids, device, ["all"])\n'
                         '    if args.valid_split:\n'
                         '        split = json.load(open(args.valid_split))\n'
                         '        Ya, Pa = collect(nets, valid, split["A"], device, ["scored"])\n'
                         '        Yv, Pv = collect(nets, valid, split["B"], device, ["scored", "all"])  # report on '
                         'half B\n'
                         '    else:\n'
                         '        Yv, Pv = collect(nets, valid, range(args.valid_seqs or valid.n_seqs), device, '
                         '["scored", "all"])\n'
                         '    print(f"predictions collected in {time.time() - t:.0f}s", flush=True)\n'
                         '    SEL = args.select_on\n'
                         '\n'
                         '    sd = Ph["all"].std(axis=1)  # [K, 2]\n'
                         '    scales = sd.mean(0, keepdims=True) / sd\n'
                         '\n'
                         '    def sel_score(idx):\n'
                         '        if SEL == "validA":\n'
                         '            return wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n'
                         '        return wp(Yh["all"], combine(Ph["all"], scales, idx))\n'
                         '\n'
                         '    def score(idx):  # valid_* = half B when --valid-split is given\n'
                         '        out = {"holdout": wp(Yh["all"], combine(Ph["all"], scales, idx)),\n'
                         '               "valid_scored": wp(Yv["scored"], combine(Pv["scored"], scales, idx)),\n'
                         '               "valid_all": wp(Yv["all"], combine(Pv["all"], scales, idx))}\n'
                         '        if args.valid_split:\n'
                         '            out["validA"] = wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n'
                         '        return out\n'
                         '\n'
                         '    cost = np.array([macs(n.config) / BASELINE_MACS for n in nets])\n'
                         '    rows = []\n'
                         '    singles = [(k, score([k])) for k in range(len(nets))]\n'
                         '    for k, sc in singles:\n'
                         '        rows.append({"combo": names[k], "n": 1, "cost": cost[k], **sc})\n'
                         '    order = [k for k, sc in sorted(singles, key=lambda ks: -ks[1][SEL])]\n'
                         '    for top in range(2, min(len(nets), args.max_members) + 1):\n'
                         '        rows.append({"combo": f"top{top} by {SEL}", "n": top, "cost": '
                         'cost[order[:top]].sum(),\n'
                         '                     "members": [names[k] for k in order[:top]], **score(order[:top])})\n'
                         '\n'
                         '    def greedy(budget):\n'
                         '        """Forward selection on SEL (no repeats) within a total cost budget."""\n'
                         '        ok = [k for k in order if cost[k] <= budget]\n'
                         '        if not ok:\n'
                         '            return []\n'
                         '        chosen, best = [ok[0]], singles[ok[0]][1][SEL]\n'
                         '        while len(chosen) < args.max_members:\n'
                         '            cand = [(k, sel_score(chosen + [k])) for k in range(len(nets))\n'
                         '                    if k not in chosen and cost[chosen + [k]].sum() <= budget]\n'
                         '            if not cand:\n'
                         '                break\n'
                         '            k, sc = max(cand, key=lambda c: c[1])\n'
                         '            if sc <= best:\n'
                         '                break\n'
                         '            chosen.append(k)\n'
                         '            best = sc\n'
                         '        return chosen\n'
                         '\n'
                         '    picks = {}\n'
                         '    for b in args.budgets:\n'
                         '        chosen = greedy(b)\n'
                         '        if chosen:\n'
                         '            picks[b] = chosen\n'
                         '            rows.append({"combo": f"greedy, cost <= {b:g}", "n": len(chosen), "cost": '
                         'cost[chosen].sum(),\n'
                         '                         "members": [names[k] for k in chosen], **score(chosen)})\n'
                         '\n'
                         '    vl = "B" if args.valid_split else ""\n'
                         '    # diversity: members that disagree with the best single model are the ones an average '
                         '(or a teacher) gains from\n'
                         '    best_k = order[0]\n'
                         '    print(f"\\nprediction correlation with {names[best_k]} (valid{vl} scored rows, mean of '
                         't0/t1):")\n'
                         '    for k in order:\n'
                         '        r = np.mean([np.corrcoef(Pv["scored"][k][:, j], Pv["scored"][best_k][:, j])[0, 1] '
                         'for j in range(2)])\n'
                         '        print(f"  {names[k][:40]:40s} {r:.4f}")\n'
                         '    print(f"\\n{\'combo\':30s} {\'n\':>2s} {\'cost\':>5s} {\'holdout\':>8s} '
                         '{\'valid\'+vl+\'_sc\':>9s} {\'valid\'+vl+\'_all\':>10s}"\n'
                         '          + (f" {\'validA_sc\':>9s}" if args.valid_split else ""))\n'
                         '    for r in rows:\n'
                         '        print(f"{r[\'combo\'][:30]:30s} {r[\'n\']:2d} {r[\'cost\']:5.2f} '
                         '{r[\'holdout\']:8.5f} {r[\'valid_scored\']:9.5f} "\n'
                         '              f"{r[\'valid_all\']:10.5f}" + (f" {r[\'validA\']:9.5f}" if "validA" in r else '
                         '"")\n'
                         '              + (f"   {\'+\'.join(r[\'members\'])}" if "members" in r else ""))\n'
                         '\n'
                         '    if args.export:\n'
                         '        for b, chosen in picks.items():\n'
                         '            out = os.path.join(args.export, f"cost{b:g}", "submission")\n'
                         '            os.makedirs(out, exist_ok=True)\n'
                         '            net = nets[chosen[0]] if len(chosen) == 1 else EnsembleNet([nets[k] for k in '
                         'chosen], scales[chosen])\n'
                         '            if len({feat_kind(nets[k]) for k in chosen} - {None}) <= 1:\n'
                         '                export_onnx(net, os.path.join(out, "model.onnx"))\n'
                         '                shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(out, '
                         '"solution.py"))\n'
                         '                shutil.copy(os.path.join(HERE, "features.py"), os.path.join(out, '
                         '"wnn_features.py"))\n'
                         '            else:\n'
                         '                print(f"cost<={b:g}: members use different extra features, no ONNX '
                         '(ensemble.json only, e.g. for a teacher)")\n'
                         '            row = next(r for r in rows if r["combo"] == f"greedy, cost <= {b:g}")\n'
                         '            json.dump({"members": [paths[k] for k in chosen], "scales": '
                         'scales[chosen].tolist(),\n'
                         '                       "scores": {k: (float(v) if isinstance(v, (float, np.floating)) else '
                         'v) for k, v in row.items()}},\n'
                         '                      open(os.path.join(out, "ensemble.json"), "w"), indent=1)\n'
                         '            print(f"exported cost<={b:g}: {len(chosen)} model(s) -> {out}")\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    main()\n',
 'pipeline/fast_eval.py': '"""Batched evaluation: runs whole sequences through the network at once (seconds instead '
                          'of\n'
                          'the ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.\n'
                          '\n'
                          '    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052\n'
                          '    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500\n'
                          '"""\n'
                          'import argparse\n'
                          'import json\n'
                          'import time\n'
                          '\n'
                          'import numpy as np\n'
                          'import torch\n'
                          '\n'
                          'from utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator\n'
                          '\n'
                          'from .data import SeqStore, split_train\n'
                          'from .groups import label as group_label\n'
                          'from .model import build_net, feat_kind, load_baseline\n'
                          '\n'
                          'NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                          '\n'
                          '\n'
                          'def pick_device(name="auto"):\n'
                          '    if name != "auto":\n'
                          '        return torch.device(name)\n'
                          '    if torch.cuda.is_available():\n'
                          '        return torch.device("cuda")\n'
                          '    if torch.backends.mps.is_available():\n'
                          '        return torch.device("mps")\n'
                          '    return torch.device("cpu")\n'
                          '\n'
                          '\n'
                          'def with_extras(X, kind="act"):\n'
                          '    """Append the extra causal features of `kind` (pipeline/features.py) to raw inputs '
                          '[B,T,112]."""\n'
                          '    from .features import offline\n'
                          '    return np.concatenate([X, offline(X[..., :112], kind)], -1)\n'
                          '\n'
                          '\n'
                          'def net_inputs(net, X, cache):\n'
                          '    """Raw batch X [B,T,112] -> what `net` takes; `cache` (a dict per batch) shares the '
                          'extras between models."""\n'
                          '    kind = feat_kind(net)\n'
                          '    if kind and kind not in cache:\n'
                          '        cache[kind] = with_extras(X, kind)\n'
                          '    return cache[kind] if kind else X\n'
                          '\n'
                          '\n'
                          '@torch.no_grad()\n'
                          'def predict_batch(net, X, device, chunk=2000):\n'
                          '    if X.shape[-1] == 112:\n'
                          '        X = net_inputs(net, X, {})\n'
                          '    B, T, _ = X.shape\n'
                          '    out = np.empty((B, T, 2), np.float32)\n'
                          '    h = xp = None\n'
                          '    for s in range(0, T, chunk):\n'
                          '        x = torch.from_numpy(X[:, s:s + chunk]).to(device)\n'
                          '        p, h = net(x, h, xp)\n'
                          '        xp = x[:, -1]\n'
                          '        out[:, s:s + chunk] = p.float().cpu().numpy()\n'
                          '    return out\n'
                          '\n'
                          '\n'
                          'def evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):\n'
                          '    """Score several models in one pass over the data. Per model:\n'
                          "    {'scored': WP on scored rows, 'all': WP on all required rows, ...} (equal without a "
                          'mask)."""\n'
                          '    for net in nets:\n'
                          '        net.eval()\n'
                          '    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]\n'
                          '    acc_g = [[GlobalAccumulator() for _ in range(4)] for _ in nets]  # scored rows per '
                          'instrument group\n'
                          '    for X, Y, M in store.batches(list(ids), batch_size):\n'
                          '        cache = {}  # extras once per batch and kind, shared by every model that uses them\n'
                          '        groups = group_label(Y)\n'
                          '        for net, (acc, acc_all), ag in zip(nets, accs, acc_g):\n'
                          '            P = predict_batch(net, net_inputs(net, X, cache), device)\n'
                          '            for b in range(len(X)):\n'
                          '                acc.add(Y[b], P[b], M[b])\n'
                          '                ag[groups[b]].add(Y[b], P[b], M[b])\n'
                          '                if store.has_mask:\n'
                          '                    acc_all.add(Y[b], P[b], NEED)\n'
                          '    out = []\n'
                          '    for (acc, acc_all), ag in zip(accs, acc_g):\n'
                          '        r = acc.result()\n'
                          '        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": '
                          'r["blocks"]}\n'
                          '        for g, a in enumerate(ag):  # nan when the eval set has no sequence of that group\n'
                          '            res[f"scored_g{g}"] = a.result()["weighted_pearson"] if a.selected_rows else '
                          'float("nan")\n'
                          '        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else '
                          'res["scored"]\n'
                          '        out.append(res)\n'
                          '    return out\n'
                          '\n'
                          '\n'
                          'def evaluate(net, store: SeqStore, ids, device, batch_size=128):\n'
                          '    return evaluate_many([net], store, ids, device, batch_size)[0]\n'
                          '\n'
                          '\n'
                          'def load_ckpt(path):\n'
                          '    ck = torch.load(path, map_location="cpu", weights_only=False)\n'
                          '    net = build_net(ck["config"])\n'
                          '    net.load_state_dict(ck["state_dict"])\n'
                          '    return net\n'
                          '\n'
                          '\n'
                          'def main():\n'
                          '    ap = argparse.ArgumentParser()\n'
                          '    g = ap.add_mutually_exclusive_group(required=True)\n'
                          '    g.add_argument("--baseline", action="store_true")\n'
                          '    g.add_argument("--ckpt")\n'
                          '    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs '
                          '(default all)")\n'
                          '    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train '
                          'holdout")\n'
                          '    ap.add_argument("--valid-split", help="split.json from mask_model: score validation '
                          'half B instead")\n'
                          '    ap.add_argument("--device", default="auto")\n'
                          '    args = ap.parse_args()\n'
                          '\n'
                          '    device = pick_device(args.device)\n'
                          '    net = (load_baseline("baseline/baseline.onnx") if args.baseline else '
                          'load_ckpt(args.ckpt)).to(device)\n'
                          '    out = {}\n'
                          '    t = time.time()\n'
                          '    valid = SeqStore("datasets/valid.parquet")\n'
                          '    ids = json.load(open(args.valid_split))["B"] if args.valid_split else '
                          'list(range(valid.n_seqs))\n'
                          '    out["valid_B" if args.valid_split else "valid"] = evaluate(net, valid, '
                          'ids[:args.valid_seqs or len(ids)], device)\n'
                          '    if args.holdout:\n'
                          '        train = SeqStore("datasets/train.parquet")\n'
                          '        _, hold = split_train(train.n_seqs, args.holdout)\n'
                          '        out["holdout"] = evaluate(net, train, hold, device)\n'
                          '    out["seconds"] = round(time.time() - t, 1)\n'
                          '    print(json.dumps(out, indent=1))\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    main()\n',
 'pipeline/features.py': '"""Extra causal input features, computed identically offline (training, whole sequences) '
                         'and\n'
                         'online (inference, one row at a time):\n'
                         '\n'
                         "* activity of each instrument's prices / volumes / trades: mean |x_t - x_{t-1}| over the "
                         'group\n'
                         '  (0 on the first row of a sequence), now and as EMAs;\n'
                         '* EMAs of the extra features a2..a5 (a3 = the strongest scoring-mask signal).\n'
                         '\n'
                         'EMA_k(t) = EMA_k(t-1) + (x_t - EMA_k(t-1)) / span_k, started at x_0. All values standardised '
                         'with\n'
                         'fixed constants (MEAN / STD, measured once on training data). Width: N_EXTRA = 36.\n'
                         '\n'
                         'Feature sets ("kinds"): "act" = the 36 above; "norm" = every one of the 112 inputs as a '
                         'z-score against its\n'
                         'own running mean and std since the start of the sequence (per-sequence level and scale, '
                         'causal);\n'
                         '"act+norm" = both, in that order; "grp" = the instrument-group detector\'s 4 probabilities '
                         '(below).\n'
                         "A model's kind follows from its input width (kind_for_width).\n"
                         '\n'
                         'Instrument groups: every sequence comes from one of 4 instruments, told apart by the tick '
                         'size of its targets\n'
                         '(TICKS; pipeline/groups.py labels them from the targets). At inference the group is detected '
                         'from the inputs:\n'
                         'multinomial logistic regression on the mean and std of the 112 inputs over the first '
                         'DET_ROWS rows (the\n'
                         'never-scored warm-up), constants below (fitted by `python -m pipeline.groups --fit`). The '
                         '"grp" feature is that\n'
                         'detector applied to the rows seen so far, frozen from row DET_ROWS - 1 on.\n'
                         'This file is copied next to solution.py as wnn_features.py, so it must only need numpy at '
                         'inference.\n'
                         '"""\n'
                         'import numpy as np\n'
                         '\n'
                         'GROUPS = [(0, 22), (22, 44), (44, 52), (52, 74), (74, 96), (96, 104)]  # i0 p, i0 v, i0 '
                         'trades, i1 p, i1 v, i1 trades\n'
                         'LEVEL_COLS = [106, 107, 108, 109]                                      # a2, a3, a4, a5\n'
                         'SPANS = [20, 200, 2000]\n'
                         'N_EXTRA = len(GROUPS) * (1 + len(SPANS)) + len(LEVEL_COLS) * len(SPANS)  # 36\n'
                         'NORM_FLOOR = 0.02  # added to the running std (inputs are globally standardised), so early '
                         'rows / flat columns stay finite\n'
                         'NORM_CLIP = 8.0\n'
                         'KINDS = {"act": N_EXTRA, "norm": 112, "act+norm": N_EXTRA + 112, "grp": 4}\n'
                         'TICKS = [0.12543, 0.24133, 0.46854, 0.65687]  # tick size of each group, in target units\n'
                         'DET_ROWS = 99\n'
                         '\n'
                         '\n'
                         'def width(kind):\n'
                         '    """Number of extra inputs of a feature kind (\'\' or None = none)."""\n'
                         '    return KINDS[kind] if kind else 0\n'
                         '\n'
                         '\n'
                         'def kind_for_width(n):\n'
                         '    return next(k for k, w in KINDS.items() if w == n)\n'
                         '\n'
                         '# standardisation constants: `python -m pipeline.features` on 256 training sequences\n'
                         'MEAN = [\n'
                         '    0.02899, 0.080176, 0.344639, 0.013477, 0.046691, 0.409072,\n'
                         '    0.028905, 0.079919, 0.344147, 0.013465, 0.046752, 0.408771,\n'
                         '    0.028743, 0.079457, 0.341971, 0.013384, 0.046461, 0.406134,\n'
                         '    0.026218, 0.07248, 0.31095, 0.012185, 0.04226, 0.36902,\n'
                         '    -0.197401, -0.235902, 0.050653, -0.37099, -0.197616, -0.236261,\n'
                         '    0.050583, -0.3714, -0.199425, -0.240088, 0.051767, -0.375679]\n'
                         'STD = [\n'
                         '    0.061435, 0.144017, 0.647244, 0.045508, 0.138009, 0.751662,\n'
                         '    0.035306, 0.083746, 0.311466, 0.019796, 0.072021, 0.43189,\n'
                         '    0.028954, 0.070935, 0.235276, 0.013908, 0.058679, 0.335773,\n'
                         '    0.02191, 0.058177, 0.200191, 0.010707, 0.048553, 0.27719,\n'
                         '    0.911831, 0.966895, 0.3914, 1.13453, 0.729865, 0.841434,\n'
                         '    0.244381, 0.879383, 0.598998, 0.688904, 0.264152, 0.809816]\n'
                         '\n'
                         '\n'
                         'def _raw_offline(X):\n'
                         '    """X [B,T,112] float32 -> unstandardised extras [B,T,36] float64 (whole sequences, exact '
                         'EMA)."""\n'
                         '    from scipy.signal import lfilter\n'
                         '\n'
                         '    X = X.astype(np.float64)\n'
                         '    d = np.abs(np.diff(X, axis=1, prepend=X[:, :1]))\n'
                         '    act = np.stack([d[..., a:b].mean(-1) for a, b in GROUPS], -1)      # [B,T,6]\n'
                         '    lev = X[..., LEVEL_COLS]                                           # [B,T,4]\n'
                         '    out = [act]\n'
                         '    for v in (act, lev):\n'
                         '        for span in SPANS:\n'
                         '            a = 1.0 / span\n'
                         '            zi = (1 - a) * v[:, :1]                                     # [B,1,F]: so that '
                         'EMA(0) = x_0\n'
                         '            y, _ = lfilter([a], [1, -(1 - a)], v, axis=1, zi=zi)\n'
                         '            out.append(y)\n'
                         '    # order: act, act_ema20, act_ema200, act_ema2000, lev_ema20, lev_ema200, lev_ema2000\n'
                         '    return np.concatenate(out, -1)\n'
                         '\n'
                         '\n'
                         'def _norm_offline(X):\n'
                         '    """X [B,T,112] -> running z-scores [B,T,112] float64. Values are taken relative to the '
                         'first row, so the\n'
                         '    float64 running sums stay exact enough; the online version adds them up in the same '
                         'order."""\n'
                         '    xs = X.astype(np.float64)\n'
                         '    xs -= xs[:, :1]\n'
                         '    n = np.arange(1, X.shape[1] + 1, dtype=np.float64)[None, :, None]\n'
                         '    mean = np.cumsum(xs, 1) / n\n'
                         '    var = np.maximum(np.cumsum(xs * xs, 1) / n - mean * mean, 0.0)\n'
                         '    return np.clip((xs - mean) / np.sqrt(var + NORM_FLOOR ** 2), -NORM_CLIP, NORM_CLIP)\n'
                         '\n'
                         '\n'
                         'def offline(X, kind="act", chunk=16):\n'
                         '    """Extras [B,T,width(kind)] float32 for whole sequences. Computed `chunk` sequences at a '
                         'time:\n'
                         '    the float64 intermediates of a whole 256-sequence batch would need ~10 GB of RAM."""\n'
                         '    out = np.empty(X.shape[:2] + (width(kind),), np.float32)\n'
                         '    for i in range(0, len(X), chunk):\n'
                         '        x = X[i:i + chunk]\n'
                         '        out[i:i + chunk] = np.concatenate([_OFFLINE[part](x) for part in kind.split("+")], '
                         '-1)\n'
                         '    return out\n'
                         '\n'
                         '\n'
                         'def _det_probs(mean, std):\n'
                         '    """Detector: [..., 112] mean and std -> [..., 4] group probabilities."""\n'
                         '    z = np.clip((np.concatenate([mean, std], -1) - DET_MEAN) / DET_SCALE, -10, 10)\n'
                         '    logits = z @ np.asarray(DET_COEF).T + DET_INTERCEPT\n'
                         '    e = np.exp(logits - logits.max(-1, keepdims=True))\n'
                         '    return e / e.sum(-1, keepdims=True)\n'
                         '\n'
                         '\n'
                         'def _grp_offline(X):\n'
                         '    """X [B,T,112] -> [B,T,4]: detector on rows 0..t (frozen from row DET_ROWS - 1 on). Sums '
                         'are taken relative\n'
                         '    to the first row (exact enough in float64), in the same order as _GrpOnline."""\n'
                         '    n_rows = min(DET_ROWS, X.shape[1])\n'
                         '    x0 = X[:, :1].astype(np.float64)\n'
                         '    xs = X[:, :n_rows].astype(np.float64) - x0\n'
                         '    n = np.arange(1, n_rows + 1, dtype=np.float64)[None, :, None]\n'
                         '    m = np.cumsum(xs, 1) / n\n'
                         '    std = np.sqrt(np.maximum(np.cumsum(xs * xs, 1) / n - m * m, 0.0))\n'
                         '    p = _det_probs(m + x0, std)\n'
                         '    return np.concatenate([p, np.repeat(p[:, -1:], X.shape[1] - n_rows, 1)], 1)\n'
                         '\n'
                         '\n'
                         'def detect(X):\n'
                         '    """Group probabilities [B,4] of whole sequences X [B,T>=DET_ROWS,112] (what the online '
                         'detector has after the\n'
                         '    warm-up)."""\n'
                         '    return _grp_offline(X[:, :DET_ROWS, :112])[:, -1]\n'
                         '\n'
                         '\n'
                         '_OFFLINE = {"act": lambda x: (_raw_offline(x) - MEAN) / STD, "norm": _norm_offline, "grp": '
                         '_grp_offline}\n'
                         '\n'
                         '\n'
                         'class Online:\n'
                         '    """Row-by-row version of `offline(X, kind)`; reset() at every new sequence."""\n'
                         '\n'
                         '    def __init__(self, kind="act"):\n'
                         '        self.parts = [{"act": _ActOnline, "norm": _NormOnline, "grp": _GrpOnline}[p]() for p '
                         'in kind.split("+")]\n'
                         '\n'
                         '    def reset(self):\n'
                         '        for p in self.parts:\n'
                         '            p.reset()\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        if len(self.parts) == 1:\n'
                         '            return self.parts[0](x)\n'
                         '        return np.concatenate([p(x) for p in self.parts])\n'
                         '\n'
                         '\n'
                         'class _NormOnline:\n'
                         '    def __init__(self):\n'
                         '        self.reset()\n'
                         '\n'
                         '    def reset(self):\n'
                         '        self.x0 = None\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        xs = np.asarray(x, dtype=np.float64)\n'
                         '        if self.x0 is None:\n'
                         '            self.x0, self.n = xs.copy(), 0\n'
                         '            self.s1, self.s2 = np.zeros_like(xs), np.zeros_like(xs)\n'
                         '        xs = xs - self.x0\n'
                         '        self.n += 1\n'
                         '        self.s1 += xs\n'
                         '        self.s2 += xs * xs\n'
                         '        mean = self.s1 / self.n\n'
                         '        var = np.maximum(self.s2 / self.n - mean * mean, 0.0)\n'
                         '        return np.clip((xs - mean) / np.sqrt(var + NORM_FLOOR ** 2), -NORM_CLIP, '
                         'NORM_CLIP).astype(np.float32)\n'
                         '\n'
                         '\n'
                         'class _GrpOnline:\n'
                         '    def __init__(self):\n'
                         '        self.reset()\n'
                         '\n'
                         '    def reset(self):\n'
                         '        self.x0, self.p = None, None\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        if self.p is not None and self.n >= DET_ROWS:\n'
                         '            return self.p\n'
                         '        xs = np.asarray(x, dtype=np.float64)\n'
                         '        if self.x0 is None:\n'
                         '            self.x0, self.n = xs.copy(), 0\n'
                         '            self.s1, self.s2 = np.zeros_like(xs), np.zeros_like(xs)\n'
                         '        xs = xs - self.x0\n'
                         '        self.n += 1\n'
                         '        self.s1 += xs\n'
                         '        self.s2 += xs * xs\n'
                         '        m = self.s1 / self.n\n'
                         '        std = np.sqrt(np.maximum(self.s2 / self.n - m * m, 0.0))\n'
                         '        self.p = _det_probs(m + self.x0, std).astype(np.float32)\n'
                         '        return self.p\n'
                         '\n'
                         '\n'
                         'class _ActOnline:\n'
                         '    """Row-by-row "act" features. Kept to a handful of numpy calls per row: one matrix '
                         'product for all group\n'
                         '    activities, one array for all EMAs."""\n'
                         '\n'
                         '    def __init__(self):\n'
                         '        n_g, n_l = len(GROUPS), len(LEVEL_COLS)\n'
                         '        self.G = np.zeros((112, n_g))                                   # |dx| @ G = '
                         'per-group mean\n'
                         '        for k, (a, b) in enumerate(GROUPS):\n'
                         '            self.G[a:b, k] = 1.0 / (b - a)\n'
                         '        self.alpha = np.array([1.0 / s for s in SPANS])[:, None]       # [3,1]\n'
                         '        self.mean = np.asarray(MEAN, dtype=np.float64)\n'
                         '        self.inv_std = 1.0 / np.asarray(STD, dtype=np.float64)\n'
                         '        self.n_g = n_g\n'
                         '        self.raw = np.empty(N_EXTRA)\n'
                         '        self.lev_idx = np.array(LEVEL_COLS)\n'
                         '        self.reset()\n'
                         '\n'
                         '    def reset(self):\n'
                         '        self.prev = None\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        x = np.asarray(x, dtype=np.float64)\n'
                         '        n_g = self.n_g\n'
                         '        if self.prev is None:\n'
                         '            act = np.zeros(n_g)\n'
                         '            self.ema = np.empty((len(SPANS), n_g + len(LEVEL_COLS)))     # [3, 6 activities '
                         '+ 4 levels]\n'
                         '            self.ema[:, :n_g] = 0.0\n'
                         '            self.ema[:, n_g:] = x[self.lev_idx]\n'
                         '        else:\n'
                         '            act = np.abs(x - self.prev) @ self.G\n'
                         '            v = np.concatenate((act, x[self.lev_idx]))\n'
                         '            self.ema += (v - self.ema) * self.alpha\n'
                         '        self.prev = x\n'
                         '        raw = self.raw\n'
                         '        raw[:n_g] = act\n'
                         '        raw[n_g:n_g * 4] = self.ema[:, :n_g].ravel()\n'
                         '        raw[n_g * 4:] = self.ema[:, n_g:].ravel()\n'
                         '        return ((raw - self.mean) * self.inv_std).astype(np.float32)\n'
                         '\n'
                         '\n'
                         '# ---- detector constants: written by `python -m pipeline.groups --fit` (between the '
                         'markers)\n'
                         '# DETECTOR BEGIN\n'
                         '# fitted on 10607 train sequences, C=0.1\n'
                         'DET_MEAN = np.array([-1.59535255e+00,  1.63937562e-01,  4.03085799e-01, -1.81277606e+00, '
                         '-1.19969050e-01,  8.43520226e-01,\n'
                         ' -3.40671044e-01, -1.09158076e-01, -8.37366759e-01, -4.48835398e-01,  1.06189585e+00, '
                         '-6.85288185e-01,\n'
                         '  2.87731529e-01,  5.48141617e-01, -1.17836287e+00, -5.63898967e-01,  1.47864301e+00,  '
                         '6.27175202e-02,\n'
                         '  6.71396523e-01,  1.98145113e+00,  1.16776456e+00, -1.03889401e+00,  1.44108506e+00,  '
                         '4.33337268e-02,\n'
                         ' -8.64151001e-01, -1.15018821e+00,  1.04276185e+00,  7.59418162e-01,  7.06694884e-01,  '
                         '2.03953903e-01,\n'
                         '  8.61796385e-01, -9.22076682e-01,  5.44038998e-01, -4.41659863e-01, -1.11754706e-01,  '
                         '2.73270439e-01,\n'
                         ' -6.64272350e-01, -9.25669164e-01, -4.55401890e-01, -1.27692274e+00, -4.89713715e-01,  '
                         '5.91891009e-01,\n'
                         '  5.99315388e-01, -3.02537838e-01, -1.03478717e+00,  1.81330828e-01, -2.96665752e-01,  '
                         '1.01899855e+00,\n'
                         '  1.05784380e-02,  1.31115616e-01, -9.49132597e-02,  1.93386497e-02, -1.61337188e+00,  '
                         '1.63376450e-01,\n'
                         '  4.04910844e-01, -1.82565154e+00, -1.25227004e-01,  8.51889888e-01, -3.43260387e-01, '
                         '-1.15359562e-01,\n'
                         ' -8.47957655e-01, -4.48910375e-01,  1.07297450e+00, -6.93131859e-01,  2.84256805e-01,  '
                         '5.46364687e-01,\n'
                         ' -1.18750218e+00, -5.73375375e-01,  1.48065280e+00,  5.89709490e-02,  6.75105896e-01,  '
                         '1.99145191e+00,\n'
                         '  1.17744676e+00, -1.04947466e+00,  1.17554864e+00,  1.50503512e-01, -5.50595941e-01, '
                         '-8.51430110e-01,\n'
                         '  8.41467994e-01,  5.88216065e-01,  6.17221806e-01,  2.15346899e-01,  7.30981377e-01, '
                         '-6.42725101e-01,\n'
                         '  4.58566200e-01, -2.18616703e-01,  5.44414713e-03,  2.50614805e-01, -4.13833765e-01, '
                         '-6.35885734e-01,\n'
                         ' -2.52624667e-01, -9.49852483e-01, -2.81142723e-01,  4.99401048e-01,  4.67699939e-01, '
                         '-1.71981254e-01,\n'
                         ' -9.08156117e-01,  2.05368742e-01, -2.39169833e-01,  9.75858150e-01, -3.24933442e-02,  '
                         '1.53243273e-01,\n'
                         ' -1.92474842e-01, -1.24090404e-02, -7.18255069e-02, -1.69877398e-03, -1.74854923e-01, '
                         '-2.13870326e-01,\n'
                         '  4.36704498e-02, -4.81706112e-01, -5.25935897e-01, -1.45604903e+00,  2.05084783e-01,  '
                         '2.31440408e-02,\n'
                         '  2.40014263e-02,  1.57895490e-01,  4.55674597e-02,  5.00090239e-02,  5.79619602e-02,  '
                         '4.12759917e-02,\n'
                         '  6.90550233e-02,  7.93925758e-02,  9.52695191e-02,  7.82329909e-02,  2.11434279e-02,  '
                         '2.85367319e-02,\n'
                         '  9.83022023e-02,  6.29402133e-02,  7.62092803e-02,  2.15328474e-02,  2.62167406e-02,  '
                         '1.25124050e-01,\n'
                         '  5.21963092e-02,  1.00057097e-01,  1.86157720e-01,  3.35519691e-01,  2.23779508e-01,  '
                         '1.48180128e-01,\n'
                         '  2.16147306e-01,  3.11234301e-01,  3.60033362e-01,  3.12074308e-01,  3.19230690e-01,  '
                         '1.82976195e-01,\n'
                         '  3.32364217e-01,  2.59936491e-01,  3.03370098e-01,  3.09539834e-01,  2.32874776e-01,  '
                         '2.18028934e-01,\n'
                         '  2.85102162e-01,  1.17188962e-01,  2.51314838e-01,  3.17859398e-01,  3.34101715e-01,  '
                         '3.14383486e-01,\n'
                         '  4.08035163e-01,  4.29568054e-01,  5.90178935e-01,  5.82911766e-01,  5.41739656e-01,  '
                         '4.91742213e-01,\n'
                         '  4.92852562e-01,  5.50734899e-01,  1.56852163e-01,  1.84953916e-02,  1.82723891e-02,  '
                         '1.21320508e-01,\n'
                         '  3.60976802e-02,  4.30955958e-02,  4.84187626e-02,  3.20977362e-02,  6.18232106e-02,  '
                         '7.06204308e-02,\n'
                         '  7.82157482e-02,  7.19206974e-02,  1.65155609e-02,  2.27072980e-02,  9.41187039e-02,  '
                         '5.33592632e-02,\n'
                         '  6.06367193e-02,  1.81973787e-02,  2.08803306e-02,  9.45063605e-02,  4.79432629e-02,  '
                         '8.54293198e-02,\n'
                         '  1.67178076e-01,  2.55740881e-01,  1.97097392e-01,  1.31073114e-01,  1.79019879e-01,  '
                         '2.46133689e-01,\n'
                         '  2.70744346e-01,  2.37897310e-01,  2.51467424e-01,  1.62748337e-01,  2.45318416e-01,  '
                         '2.28419949e-01,\n'
                         '  2.42855721e-01,  2.50083644e-01,  2.06801576e-01,  1.88978193e-01,  2.37615552e-01,  '
                         '1.02170008e-01,\n'
                         '  2.18791559e-01,  2.52187402e-01,  2.58986649e-01,  2.17628730e-01,  4.34882332e-01,  '
                         '4.00493578e-01,\n'
                         '  5.89931824e-01,  5.13624229e-01,  6.57317110e-01,  5.85242951e-01,  5.71303224e-01,  '
                         '6.67883089e-01,\n'
                         '  8.13577299e-01,  9.27598690e-01,  4.21959530e-01,  2.28934601e-01,  8.61319968e-01,  '
                         '1.42823982e+00,\n'
                         '  1.29361079e+00,  2.23962612e+00])\n'
                         'DET_SCALE = np.array([0.23462969, 0.05281278, 0.06101852, 0.20802024, 0.05964505, '
                         '0.05424397, 0.12929275, 0.07036838, 0.12350859,\n'
                         ' 0.20374899, 0.13378152, 0.16766007, 0.02413448, 0.06706873, 0.16710252, 0.11499964, '
                         '0.11399146, 0.04760914,\n'
                         ' 0.04501736, 0.13455716, 0.07225701, 0.16045676, 0.54731579, 0.57483207, 0.57474981, '
                         '0.60195044, 0.67769287,\n'
                         ' 0.61400203, 0.56762336, 0.61803861, 0.52873307, 0.57134864, 0.42066881, 0.59021637, '
                         '0.48376387, 0.62807704,\n'
                         ' 0.50096094, 0.5830616 , 0.52076263, 0.59104345, 0.56566998, 0.58187808, 0.57210594, '
                         '0.50033788, 0.35603686,\n'
                         ' 0.22350006, 0.24361362, 0.21945572, 0.07593479, 0.18774505, 0.18106074, 0.06121268, '
                         '0.29402455, 0.06276322,\n'
                         ' 0.06215742, 0.25909647, 0.07515532, 0.0867776 , 0.13309886, 0.09369351, 0.15127848, '
                         '0.19451607, 0.16200885,\n'
                         ' 0.17911685, 0.03585553, 0.0602639 , 0.19624264, 0.13432232, 0.12925879, 0.05160824, '
                         '0.05988254, 0.16734647,\n'
                         ' 0.09222144, 0.18781576, 0.48464068, 0.57163337, 0.54529189, 0.58061983, 0.66983751, '
                         '0.60456234, 0.54882803,\n'
                         ' 0.6330925 , 0.51720629, 0.5272223 , 0.4303853 , 0.61744233, 0.49027828, 0.60782953, '
                         '0.48777469, 0.5741794 ,\n'
                         ' 0.52482785, 0.53006457, 0.56863114, 0.56652425, 0.56429183, 0.5356931 , 0.49842223, '
                         '0.28668026, 0.32647706,\n'
                         ' 0.28639178, 0.13142896, 0.30663764, 0.29397789, 0.09759341, 0.4236772 , 0.02253475, '
                         '0.83945313, 0.92671085,\n'
                         ' 0.32167995, 1.05165359, 1.23021321, 1.59182536, 0.1435951 , 0.03060348, 0.03814225, '
                         '0.1096325 , 0.04382494,\n'
                         ' 0.04291313, 0.07183977, 0.04041075, 0.07996096, 0.11449634, 0.07083898, 0.10251676, '
                         '0.02629863, 0.04253964,\n'
                         ' 0.11175028, 0.06693124, 0.07000156, 0.02759462, 0.03485356, 0.08545819, 0.0435561 , '
                         '0.0970709 , 0.21189175,\n'
                         ' 0.32257068, 0.27016862, 0.17946469, 0.22778989, 0.31526521, 0.36850409, 0.30264478, '
                         '0.32986104, 0.20333083,\n'
                         ' 0.3108909 , 0.27342728, 0.34996624, 0.31570081, 0.23441863, 0.25413537, 0.28998372, '
                         '0.11432975, 0.26440827,\n'
                         ' 0.32489567, 0.32467444, 0.27772564, 0.23607744, 0.25483621, 0.31906157, 0.30529563, '
                         '0.39974673, 0.35733758,\n'
                         ' 0.35792046, 0.40604228, 0.14717697, 0.02647632, 0.0345923 , 0.11341893, 0.04863167, '
                         '0.04915736, 0.06945584,\n'
                         ' 0.03848595, 0.08415118, 0.11267223, 0.07627381, 0.10545146, 0.02925999, 0.03790776, '
                         '0.11974524, 0.07038802,\n'
                         ' 0.07031732, 0.03017965, 0.0335906 , 0.08653637, 0.04893367, 0.10377079, 0.24301147, '
                         '0.33227991, 0.28943555,\n'
                         ' 0.21120577, 0.25192701, 0.32772709, 0.34378264, 0.30937642, 0.33563524, 0.2217285 , '
                         '0.31122928, 0.30579758,\n'
                         ' 0.34281525, 0.32501308, 0.2720828 , 0.27377108, 0.30997018, 0.12581008, 0.29218768, '
                         '0.32796618, 0.31963937,\n'
                         ' 0.27059136, 0.33394253, 0.31843402, 0.43888701, 0.36367054, 0.49479018, 0.43914711, '
                         '0.42377031, 0.50241912,\n'
                         ' 0.25272388, 0.24538326, 0.23301068, 0.16456892, 0.26862666, 0.87147386, 0.77279737, '
                         '0.83000672])\n'
                         'DET_COEF = np.array([[ 1.44479697e-01, -1.64358179e-01, -1.52571845e-01,  5.75430557e-01,  '
                         '2.20901951e-01,  8.21133390e-02,\n'
                         '  -2.18651816e-01, -2.18654020e-01,  1.12053662e-01, -3.81425047e-01,  1.95529787e-01, '
                         '-2.11098968e-01,\n'
                         '   8.79791180e-02,  1.51138646e-01,  5.29970682e-02, -4.84812596e-02, -1.41077996e-02,  '
                         '5.55565966e-02,\n'
                         '  -8.84453245e-02, -2.50902609e-02, -2.09240165e-01, -5.18782278e-02, -2.99151845e-01,  '
                         '2.88423288e-02,\n'
                         '   1.50547285e-01,  2.16620721e-01, -2.40650595e-01, -1.41367831e-01, -1.10335376e-01,  '
                         '1.31006802e-02,\n'
                         '  -1.21200988e-01,  1.05688715e-01, -1.29251851e-01,  1.22175242e-01,  3.78048977e-02,  '
                         '1.97849548e-02,\n'
                         '   1.34391775e-01,  1.86949535e-01,  1.01900213e-01,  2.98247716e-01,  7.11854803e-02, '
                         '-7.27777631e-02,\n'
                         '  -1.14707434e-01,  4.93357467e-03,  1.42855585e-01, -1.83010738e-01,  6.96340276e-04,  '
                         '7.12449868e-03,\n'
                         '   1.44469336e-01,  1.04841389e-01,  9.83428236e-02,  1.91036235e-01,  9.85541184e-02, '
                         '-5.11420385e-02,\n'
                         '   9.20900824e-03,  4.76144902e-01,  3.36885961e-01,  1.05809030e-01, -1.45920516e-01, '
                         '-1.18820035e-01,\n'
                         '   1.25175873e-01, -2.18860414e-01,  1.51207751e-01, -1.74216834e-01,  8.47288802e-02,  '
                         '1.59671784e-01,\n'
                         '   1.25375290e-01, -5.43469858e-02, -9.34192700e-02,  4.50866109e-02, -6.80221478e-02,  '
                         '2.02170835e-02,\n'
                         '  -1.48670087e-01, -8.90221736e-03,  7.07451968e-03,  6.08329242e-02, -9.89761522e-02, '
                         '-1.68969610e-01,\n'
                         '   3.97761503e-02, -7.96669462e-02, -3.50539496e-02,  8.60435009e-02,  5.85096188e-02, '
                         '-2.85650572e-01,\n'
                         '  -5.14243489e-02,  4.03720069e-02, -6.89435951e-02, -3.68313408e-03, -1.47130999e-01, '
                         '-2.11751693e-01,\n'
                         '  -9.79998027e-02, -1.29148508e-01, -1.52566316e-01,  5.71366135e-02, -1.25502329e-02, '
                         '-2.50101579e-01,\n'
                         '   2.64924972e-01, -7.17630926e-02,  8.44943865e-02, -1.49847389e-01,  5.99541623e-02,  '
                         '9.86615670e-02,\n'
                         '  -7.72150463e-02,  2.45185889e-02, -8.71854223e-02,  1.01406273e-02, -8.95791892e-02, '
                         '-6.85347378e-02,\n'
                         '  -7.54704094e-02, -4.36590262e-02, -8.85882734e-03, -1.56669674e-01,  2.66725826e-02,  '
                         '6.88758840e-02,\n'
                         '  -3.35301117e-03,  6.69215392e-02, -2.14267890e-02, -4.04088846e-02, -8.18203383e-02, '
                         '-1.42221313e-01,\n'
                         '   5.44432935e-02,  9.83644149e-02, -9.12675895e-02,  5.20523159e-02,  4.88388961e-03, '
                         '-1.32424308e-01,\n'
                         '   6.90427887e-02, -8.65147688e-02,  1.10212525e-01,  5.51077287e-02,  1.09209597e-02, '
                         '-4.29282227e-02,\n'
                         '   1.93980963e-02,  7.10635284e-02, -3.33032293e-02,  6.06413093e-03, -1.95769806e-02, '
                         '-1.01256139e-02,\n'
                         '  -1.86308876e-02,  3.77227023e-03, -1.64587313e-02,  1.02732105e-02, -2.22513614e-02, '
                         '-3.85513013e-02,\n'
                         '   5.34163580e-03, -2.20620170e-02, -1.10860458e-03, -2.46460946e-02, -4.62158382e-03, '
                         '-3.72201470e-02,\n'
                         '  -3.16591716e-02, -3.80139798e-02, -3.70995201e-02, -4.50013902e-02, -3.35786427e-02, '
                         '-3.99735594e-02,\n'
                         '  -6.63733202e-02, -5.74525474e-02, -5.18475806e-02, -5.23859574e-02, -9.46703407e-02, '
                         '-1.00370672e-01,\n'
                         '  -1.03678486e-01, -9.74310178e-02, -4.33731020e-02, -4.07959261e-02, -6.29588307e-02, '
                         '-1.55383725e-02,\n'
                         '  -1.13200635e-01, -4.92916426e-02,  3.23028926e-02, -1.45211459e-01,  4.02851021e-02,  '
                         '4.70942306e-02,\n'
                         '  -6.44409300e-02,  2.15837045e-02, -3.37255804e-02, -5.84011671e-02,  2.98224658e-02, '
                         '-2.22717234e-02,\n'
                         '   8.33223981e-02, -2.03203361e-02,  4.95576399e-02, -6.05985083e-02, -5.26427813e-02,  '
                         '3.94896233e-02,\n'
                         '   6.94276252e-02,  1.08082258e-01,  3.71971311e-02,  1.38446160e-02,  2.49733057e-02,  '
                         '1.03282954e-01,\n'
                         '   5.66735993e-02,  8.08842436e-02,  7.67186128e-02, -6.03027660e-03,  1.54506038e-01,  '
                         '9.38090710e-02,\n'
                         '   1.26956336e-01,  4.62279234e-02,  9.58882954e-02, -4.45218146e-03,  6.66624175e-02,  '
                         '1.34979258e-02,\n'
                         '   2.89165980e-02,  7.34016005e-02,  6.34045497e-02,  3.35286714e-02, -6.41968902e-02, '
                         '-8.89814216e-02,\n'
                         '  -5.76684781e-02, -6.27165362e-02,  6.23868102e-02,  1.79484046e-02,  2.23352094e-02,  '
                         '6.17914618e-02,\n'
                         '  -1.69213216e-01, -5.90617849e-04, -3.98616616e-02, -1.01853812e-01,  2.86245257e-02, '
                         '-2.59716169e-02,\n'
                         '   7.28687881e-02,  1.09387421e-01],\n'
                         ' [ 2.59587874e-02, -1.06120901e-01,  2.02383191e-02,  6.09481154e-01,  3.56511875e-01, '
                         '-1.96496502e-02,\n'
                         '  -6.17156534e-02, -4.21433373e-01,  1.22257675e-01,  6.46026751e-03,  2.04773540e-01,  '
                         '4.25566488e-02,\n'
                         '  -1.80094105e-02, -5.35045659e-03,  9.96642825e-02,  1.41047129e-01, -1.90732749e-01, '
                         '-4.52005410e-02,\n'
                         '  -1.54485380e-01,  1.66566576e-01, -2.85526770e-01,  4.94474736e-02, -2.03282738e-02,  '
                         '5.74376161e-02,\n'
                         '  -1.24562582e-01, -9.88580134e-02,  1.11192327e-01, -7.84732992e-02, -8.15550492e-02,  '
                         '4.18542830e-02,\n'
                         '   1.59216027e-01, -1.25800816e-01,  3.02005987e-02, -5.32735107e-02, -5.71630942e-02, '
                         '-1.71615039e-01,\n'
                         '  -2.34200052e-01, -2.11712109e-01, -2.16067990e-01,  1.17425795e-03, -2.39685782e-01,  '
                         '3.32399596e-02,\n'
                         '  -1.16185232e-02, -2.05985284e-01,  7.40815869e-02, -8.22808327e-02,  1.38903435e-01, '
                         '-2.17637922e-02,\n'
                         '  -7.07495611e-03,  4.48525480e-02, -1.91969336e-01, -1.49798892e-01, -6.77960264e-02, '
                         '-3.14732087e-02,\n'
                         '   1.07800051e-01,  3.85975648e-01,  5.73956611e-02,  5.92802603e-03, -2.01001364e-02, '
                         '-2.15841361e-01,\n'
                         '   4.98634941e-02, -3.39807148e-02,  1.98007586e-01,  1.82457597e-02, -4.75656123e-02, '
                         '-2.92678522e-02,\n'
                         '   8.23600333e-04,  7.32464152e-02, -7.97740218e-02, -9.24070041e-02,  8.27349501e-02,  '
                         '1.98203467e-01,\n'
                         '  -5.14951025e-02, -3.18380435e-02, -5.07070052e-01,  5.56183130e-02,  2.22970350e-01,  '
                         '2.62344547e-01,\n'
                         '  -2.48723004e-01, -1.33782726e-01, -2.01161466e-01, -3.31843740e-02, -1.74302165e-01,  '
                         '3.24687037e-01,\n'
                         '  -1.51435507e-01,  1.49450726e-01,  1.44858458e-02, -4.89984165e-02,  1.33276643e-01,  '
                         '3.16388460e-01,\n'
                         '   4.21340873e-02,  3.93558744e-01,  7.97247378e-02, -1.62938397e-01, -9.29839774e-02, '
                         '-1.64519329e-02,\n'
                         '   9.14335188e-02, -8.43165786e-03,  1.74079827e-02,  5.36379066e-02,  1.71885999e-02, '
                         '-5.66065705e-02,\n'
                         '   1.11705828e-01,  1.36317054e-02, -8.54568591e-02, -7.68643909e-03, -2.77575750e-02, '
                         '-5.56316188e-02,\n'
                         '   3.38001418e-02,  1.58475629e-01,  6.30370570e-02, -4.05305749e-01, -1.83508165e-01,  '
                         '5.83570587e-02,\n'
                         '   2.90374151e-02, -6.19168413e-02, -1.20268844e-01,  1.12757539e-01, -2.75082628e-02, '
                         '-1.94103904e-01,\n'
                         '   6.39746541e-02,  7.14306699e-02,  1.97292295e-02,  8.57449699e-02,  6.57927095e-02,  '
                         '8.85267424e-03,\n'
                         '   1.96877487e-01, -8.54897076e-02,  3.88341927e-02,  1.43120929e-01,  6.44123519e-02, '
                         '-2.31404857e-01,\n'
                         '   1.26649123e-01,  6.85109813e-02, -3.06111098e-02,  9.29346515e-02, -4.11614003e-02, '
                         '-1.39295522e-01,\n'
                         '   5.45699438e-02,  7.85263206e-02,  3.99306054e-02,  1.75838858e-01,  7.23533111e-02, '
                         '-1.38571078e-02,\n'
                         '   1.43281460e-01,  1.65532762e-01,  1.07858596e-01,  1.48659802e-01,  1.23181705e-01, '
                         '-6.60969764e-02,\n'
                         '   8.25873161e-03,  5.66020008e-02, -3.14962507e-02,  1.22082624e-01,  7.38368435e-02,  '
                         '4.96098138e-02,\n'
                         '  -1.14835934e-01, -7.78677893e-02,  5.81307572e-03,  1.21963566e-01,  5.01465386e-02,  '
                         '2.25612616e-02,\n'
                         '  -4.40300196e-03,  3.90187289e-02,  8.05786610e-03,  1.39047093e-01,  4.75592134e-02,  '
                         '1.28329718e-01,\n'
                         '  -4.11761803e-02,  5.18365332e-02,  6.05197710e-03, -1.81255944e-01,  5.49574992e-02,  '
                         '8.31622812e-02,\n'
                         '   2.00332035e-02,  6.12984856e-02,  2.92227311e-03,  3.92125190e-02,  6.20729353e-02, '
                         '-3.99569204e-02,\n'
                         '   9.07777095e-02,  9.54492925e-02,  5.32092074e-03,  2.08970516e-02,  8.45985651e-02,  '
                         '7.39462208e-02,\n'
                         '  -4.28603755e-02,  4.97346059e-02,  5.03966171e-02, -3.36093456e-02, -2.89720549e-02,  '
                         '6.66120277e-03,\n'
                         '   2.81433474e-02,  3.22502726e-02, -4.10977791e-03,  4.50083295e-02,  6.94735436e-03, '
                         '-2.51129050e-03,\n'
                         '   2.26238559e-02,  4.66985062e-03,  1.91300718e-03, -8.42119110e-03,  2.94895906e-02, '
                         '-2.04917632e-02,\n'
                         '   2.63356389e-02,  5.34660965e-02,  2.75919095e-02,  1.23198719e-02, -8.58655867e-02, '
                         '-2.02513257e-01,\n'
                         '  -1.33711548e-01, -9.68402954e-02, -1.38713417e-01, -1.45857580e-01, -1.35215816e-01, '
                         '-1.34680329e-01,\n'
                         '  -5.26418279e-02, -4.68949811e-02,  1.63411646e-02, -8.32177834e-02,  2.54205990e-02,  '
                         '3.23940208e-01,\n'
                         '  -1.10667384e-02,  3.50745583e-02],\n'
                         ' [-2.79826439e-01, -5.71418224e-01, -6.13395668e-01, -5.30249034e-01, -2.43414758e-01, '
                         '-9.86611039e-02,\n'
                         '   2.39921635e-01,  1.90238490e-01, -1.08920853e-01,  3.62215151e-01, -2.57219624e-01,  '
                         '2.51662414e-02,\n'
                         '  -8.07708784e-02,  5.24858199e-01, -5.74324666e-02,  1.48705902e-01, -1.08444522e-01, '
                         '-2.20171002e-01,\n'
                         '   1.99037803e-01, -1.83039859e-02,  1.74165101e-01,  6.92445847e-02, -1.65769727e-01, '
                         '-1.36236461e-02,\n'
                         '   1.13412481e-02,  4.77741222e-02, -9.79494342e-02,  4.43499533e-04, -7.52092565e-02, '
                         '-9.51076669e-02,\n'
                         '  -5.02123895e-02,  2.54803235e-01, -1.79997084e-02,  9.00943541e-02,  4.94208428e-02,  '
                         '8.21781005e-02,\n'
                         '   4.05626253e-01,  4.14476092e-01,  1.26885652e-01, -3.53350404e-02,  2.71494438e-01, '
                         '-1.04493604e-01,\n'
                         '   6.77137006e-02,  3.77597079e-01, -3.62774954e-01, -2.65348918e-01, -1.55008165e-02, '
                         '-2.27854561e-01,\n'
                         '  -5.71925775e-02,  1.01822553e-01,  9.60502216e-02, -4.05318597e-03, -1.15259140e-01, '
                         '-5.34948476e-01,\n'
                         '  -6.93388245e-01, -3.33723583e-01, -3.08901273e-01, -2.28014170e-01,  2.38197162e-01,  '
                         '2.40555252e-01,\n'
                         '  -2.03268956e-03,  3.09811529e-01, -2.80797068e-01, -2.82947440e-02, -6.18658064e-02,  '
                         '6.41821750e-01,\n'
                         '   1.62156209e-02,  2.72082708e-01,  2.82334258e-02, -4.13980187e-02,  1.55753791e-01, '
                         '-2.07310618e-01,\n'
                         '  -1.59401206e-01,  6.95699438e-02,  2.47020457e-01, -1.02416338e-02, -7.00599862e-02, '
                         '-4.25141903e-01,\n'
                         '   2.36583844e-01,  1.28221058e-01,  1.00811724e-01, -3.05764494e-02,  1.95701159e-01, '
                         '-3.81822568e-01,\n'
                         '   2.63028669e-03, -9.49229900e-02, -7.76596165e-02,  4.49582924e-02, -3.25599961e-02, '
                         '-9.14219924e-02,\n'
                         '  -8.19515255e-02, -6.81476059e-01, -8.45347954e-02,  7.01887229e-02,  1.36891588e-01,  '
                         '1.62897904e-01,\n'
                         '  -2.10835898e-01, -2.40652625e-01, -2.41827329e-02, -2.42594707e-01,  3.13389717e-02,  '
                         '1.94715279e-01,\n'
                         '  -2.15456826e-02,  6.19421710e-02, -3.16730721e-01,  1.68185392e-02,  2.71268581e-02, '
                         '-2.10370624e-02,\n'
                         '   1.18061242e-01, -4.22842104e-01,  2.01235432e-01,  1.16029645e-01,  2.93417603e-01, '
                         '-4.96036377e-01,\n'
                         '  -2.36008928e-01, -1.98235855e-02,  5.44168634e-02, -5.26354778e-02,  8.84468333e-02,  '
                         '2.30566967e-01,\n'
                         '  -6.35992253e-02, -2.92063983e-02,  7.37659016e-02, -1.58677980e-01, -1.24767431e-01, '
                         '-1.06383339e-01,\n'
                         '   2.04991782e-01,  1.11977500e-02, -9.06260500e-02,  1.21120946e-02, -1.46242547e-01, '
                         '-6.73790950e-02,\n'
                         '  -4.78840532e-02,  7.27125883e-02,  1.95305584e-01, -1.03500997e-02, -7.94824889e-02,  '
                         '2.63291732e-01,\n'
                         '  -9.28672084e-02, -5.55778488e-03, -1.37624822e-01, -2.51282026e-01,  5.59570628e-03,  '
                         '1.80496296e-01,\n'
                         '   2.03391695e-03, -3.65718458e-02,  1.27558723e-01, -1.45476885e-01, -3.51795330e-02,  '
                         '4.40788259e-02,\n'
                         '   6.36656690e-02,  1.56338410e-01,  1.06286042e-01, -7.69800362e-02, -1.18515011e-01, '
                         '-2.45215561e-02,\n'
                         '   1.05867601e-02, -2.79641906e-01, -7.93636368e-02,  2.53157458e-01, -6.46164959e-02, '
                         '-1.25854170e-01,\n'
                         '  -6.96131548e-02, -6.65150933e-02,  1.79526974e-01, -2.41447887e-01, -1.30315414e-01,  '
                         '2.45210143e-02,\n'
                         '   6.57412575e-02, -2.72712317e-01, -6.88905151e-03,  1.06641734e-01, -7.86736584e-02, '
                         '-3.14881601e-02,\n'
                         '  -2.80362214e-02, -2.99354813e-02, -1.58549586e-01, -1.80358410e-01,  4.37984690e-02,  '
                         '6.60094152e-02,\n'
                         '  -2.54839076e-03, -1.14274202e-01, -7.71349513e-02,  2.69933392e-02, -3.43420436e-01,  '
                         '1.22689164e-01,\n'
                         '   8.33075814e-03,  3.54318680e-02,  1.36686648e-01,  1.47954252e-01, -2.51263010e-01, '
                         '-1.15241595e-01,\n'
                         '  -3.29195101e-02, -3.99990547e-03,  5.76762959e-02,  3.31028356e-02, -1.19436276e-01,  '
                         '1.54756476e-02,\n'
                         '  -9.43363871e-02, -2.88561839e-02, -2.70642015e-02,  6.26685955e-02,  1.30469001e-01,  '
                         '1.25869571e-01,\n'
                         '   7.30081914e-03, -6.99574641e-02, -6.54243119e-02,  4.45858028e-02,  6.74098847e-02,  '
                         '1.45605805e-02,\n'
                         '   1.65137365e-01,  5.54309317e-01,  1.45098217e-01,  5.88320279e-02,  1.39794960e-01,  '
                         '1.78447853e-01,\n'
                         '  -1.86123911e-01,  5.78066939e-02,  1.08614670e-01,  9.31524223e-02, -3.80524833e-02, '
                         '-2.26617156e-01,\n'
                         '  -2.12479658e-01,  1.41036687e-01],\n'
                         ' [ 1.09387954e-01,  8.41897305e-01,  7.45729194e-01, -6.54662677e-01, -3.33999067e-01,  '
                         '3.61974151e-02,\n'
                         '   4.04458336e-02,  4.49848902e-01, -1.25390484e-01,  1.27496276e-02, -1.43083704e-01,  '
                         '1.43376078e-01,\n'
                         '   1.08011710e-02, -6.70646389e-01, -9.52288841e-02, -2.41271771e-01,  3.13285071e-01,  '
                         '2.09814946e-01,\n'
                         '   4.38929010e-02, -1.23172329e-01,  3.20601834e-01, -6.68138305e-02,  4.85249846e-01, '
                         '-7.26562988e-02,\n'
                         '  -3.73259506e-02, -1.65536829e-01,  2.27407703e-01,  2.19397631e-01,  2.67099682e-01,  '
                         '4.01527037e-02,\n'
                         '   1.21973502e-02, -2.34691134e-01,  1.17050961e-01, -1.58996085e-01, -3.00626463e-02,  '
                         '6.96519841e-02,\n'
                         '  -3.05817976e-01, -3.89713519e-01, -1.27178747e-02, -2.64086933e-01, -1.02994136e-01,  '
                         '1.44031408e-01,\n'
                         '   5.86122570e-02, -1.76545369e-01,  1.45837783e-01,  5.30640489e-01, -1.24098958e-01,  '
                         '2.42493854e-01,\n'
                         '  -8.02018028e-02, -2.51516490e-01, -2.42370942e-03, -3.71841567e-02,  8.45010482e-02,  '
                         '6.17563723e-01,\n'
                         '   5.76379186e-01, -5.28396966e-01, -8.53803494e-02,  1.16277114e-01, -7.21765095e-02,  '
                         '9.41061438e-02,\n'
                         '  -1.73006678e-01, -5.69703999e-02, -6.84182685e-02,  1.84265818e-01,  2.47025385e-02, '
                         '-7.72225682e-01,\n'
                         '  -1.42414511e-01, -2.90982137e-01,  1.44959866e-01,  8.87184120e-02, -1.70466593e-01, '
                         '-1.11099329e-02,\n'
                         '   3.59566395e-01, -2.88296829e-02,  2.52975076e-01, -1.06209603e-01, -5.39342117e-02,  '
                         '3.31766966e-01,\n'
                         '  -2.76369909e-02,  8.52286145e-02,  1.35403692e-01, -2.22826776e-02, -7.99086130e-02,  '
                         '3.42786103e-01,\n'
                         '   2.00229569e-01, -9.48997432e-02,  1.32117366e-01,  7.72325818e-03,  4.64143525e-02, '
                         '-1.32147747e-02,\n'
                         '   1.37817241e-01,  4.17065823e-01,  1.57376373e-01,  3.56130605e-02, -3.13573772e-02,  '
                         '1.03655608e-01,\n'
                         '  -1.45522592e-01,  3.20847375e-01, -7.77196364e-02,  3.38804190e-01, -1.08481734e-01, '
                         '-2.36770275e-01,\n'
                         '  -1.29450995e-02, -1.00092465e-01,  4.89373003e-01, -1.92727274e-02,  9.02099061e-02,  '
                         '1.45203419e-01,\n'
                         '  -7.63909741e-02,  3.08025501e-01, -2.55413662e-01,  4.45945778e-01, -1.36582021e-01,  '
                         '3.68803434e-01,\n'
                         '   2.10324524e-01,  1.48188876e-02,  8.72787693e-02, -1.97131764e-02,  2.08817678e-02,  '
                         '1.05758250e-01,\n'
                         '  -5.48187224e-02, -1.40588686e-01, -2.22754161e-03,  2.08806945e-02,  5.40908322e-02,  '
                         '2.29954973e-01,\n'
                         '  -4.70912057e-01,  1.60806726e-01, -5.84206673e-02, -2.10340752e-01,  7.09092352e-02,  '
                         '3.41712174e-01,\n'
                         '  -9.81631658e-02, -2.12287098e-01, -1.31391245e-01, -8.86486827e-02,  1.40220870e-01, '
                         '-1.13870596e-01,\n'
                         '   5.69281522e-02, -7.67408060e-02,  1.14152947e-01,  6.51699566e-02, -5.56976560e-02, '
                         '-1.28087887e-01,\n'
                         '  -1.50657013e-01, -1.06898899e-01, -2.34308715e-01,  2.14631773e-02, -8.33805879e-02,  '
                         '5.92382974e-02,\n'
                         '  -4.02652290e-02, -1.74926430e-01, -3.76902715e-02, -1.01197516e-04,  7.82568101e-02,  '
                         '1.48853018e-02,\n'
                         '   1.70622495e-01,  4.14962243e-01,  1.25398142e-01, -3.22735066e-01,  1.09140298e-01,  '
                         '2.03663581e-01,\n'
                         '   1.77694642e-01,  1.24927382e-01, -1.44211738e-01,  1.43196719e-01,  1.45715032e-01, '
                         '-1.37312360e-01,\n'
                         '   8.86355579e-02,  2.70167427e-01, -3.14658182e-02,  2.19825669e-01, -1.65689429e-02, '
                         '-9.87683517e-02,\n'
                         '   7.24439479e-02, -5.29467087e-02,  1.89352893e-01,  1.99547058e-01, -1.35693870e-01, '
                         '-3.78077132e-03,\n'
                         '  -1.71551717e-01,  3.91452452e-02,  2.22563907e-02,  1.27081175e-02,  3.11464653e-01, '
                         '-2.36125008e-01,\n'
                         '  -3.48980079e-02, -1.93248732e-01, -2.24280397e-01, -1.28189522e-01,  2.55261759e-01,  '
                         '5.29743766e-03,\n'
                         '  -5.18974366e-02, -1.09134611e-01, -1.30285131e-01, -7.20808885e-02, -4.20171157e-02, '
                         '-1.06773428e-01,\n'
                         '  -5.52438045e-02, -2.20415901e-02, -7.07371010e-02, -4.97952230e-02, -2.26621009e-01, '
                         '-1.18875734e-01,\n'
                         '  -6.25530560e-02, -5.69102329e-02, -2.55721473e-02, -9.04343460e-02,  8.26525922e-02,  '
                         '2.76934098e-01,\n'
                         '   2.62426612e-02, -3.94752486e-01, -6.87716104e-02,  6.90771479e-02, -2.69143527e-02, '
                         '-1.05558986e-01,\n'
                         '   4.07978955e-01, -1.03210950e-02, -8.50941726e-02,  9.19191735e-02, -1.59926414e-02, '
                         '-7.13514349e-02,\n'
                         '   1.50677608e-01, -2.85498667e-01]])\n'
                         'DET_INTERCEPT = np.array([-1.95277847, -1.43725674,  1.95477532,  1.43525989])\n'
                         '# DETECTOR END\n'
                         '\n'
                         '\n'
                         'def compute_stats(n_seqs=256, data="datasets/train.parquet"):\n'
                         '    from .data import SeqStore\n'
                         '\n'
                         '    st = SeqStore(data)\n'
                         '    ids = list(range(0, st.n_seqs, st.n_seqs // n_seqs))[:n_seqs]\n'
                         '    R = []\n'
                         '    for i in range(0, len(ids), 32):\n'
                         '        X, _, _ = st.load(ids[i:i + 32])\n'
                         '        R.append(_raw_offline(X)[:, 99:].reshape(-1, N_EXTRA)[::7])\n'
                         '    R = np.concatenate(R)\n'
                         '    return R.mean(0), R.std(0) + 1e-6\n'
                         '\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    m, s = compute_stats()\n'
                         '    np.set_printoptions(precision=6, suppress=True, linewidth=150)\n'
                         '    print("MEAN = " + repr([round(float(v), 6) for v in m]))\n'
                         '    print("STD = " + repr([round(float(v), 6) for v in s]))\n',
 'pipeline/groups.py': '"""Instrument groups (tick sizes): labels from the targets, and fitting the input-based '
                       'detector.\n'
                       '\n'
                       '    python -m pipeline.groups --fit        # fit the detector on train, check it on valid, '
                       'write it into features.py\n'
                       '    python -m pipeline.groups --check      # accuracy of the detector in features.py on valid\n'
                       '\n'
                       "Every target of a sequence is a whole multiple of its instrument's tick (features.TICKS), so "
                       'the smallest\n'
                       'non-zero |target| names the group (training and evaluation only; at inference features.detect '
                       '/ the "grp"\n'
                       'feature work from the inputs).\n'
                       '"""\n'
                       'import argparse\n'
                       'import os\n'
                       'import re\n'
                       '\n'
                       'import numpy as np\n'
                       'import pyarrow.parquet as pq\n'
                       '\n'
                       'from utils import FEATURE_COLUMNS\n'
                       '\n'
                       'from . import features\n'
                       'from .features import DET_ROWS, TICKS\n'
                       '\n'
                       'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                       '\n'
                       '\n'
                       'def label(Y):\n'
                       '    """Y [B,T,2] (numpy) -> group index [B] (0..3)."""\n'
                       '    a = np.abs(Y[..., 0]).astype(np.float64)\n'
                       '    tick = np.where(a > 1e-6, a, np.inf).min(1)\n'
                       '    return np.abs(tick[:, None] - np.asarray(TICKS)[None]).argmin(1)\n'
                       '\n'
                       '\n'
                       'def warmup_stats(path, ids=None):\n'
                       '    """(mean, std) [N,112] of the first DET_ROWS rows, as the detector computes them, and the '
                       'group labels [N]."""\n'
                       '    pf = pq.ParquetFile(path)\n'
                       '    ids = range(pf.num_row_groups) if ids is None else ids\n'
                       '    X, G = [], []\n'
                       '    for g in ids:\n'
                       '        t = pf.read_row_group(int(g), columns=list(FEATURE_COLUMNS) + ["t0", '
                       '"t1"]).to_pandas().to_numpy(np.float32)\n'
                       '        X.append(t[:DET_ROWS, :112].copy())  # a copy, so the whole sequence can be freed\n'
                       '        G.append(label(t[None, :, 112:])[0])\n'
                       '    X = np.stack(X).astype(np.float64)\n'
                       '    x0 = X[:, :1]\n'
                       '    xs = X - x0\n'
                       '    m = xs.mean(1)\n'
                       '    std = np.sqrt(np.maximum((xs * xs).mean(1) - m * m, 0.0))\n'
                       '    return m + x0[:, 0], std, np.array(G)\n'
                       '\n'
                       '\n'
                       'def accuracy(mean, std, G):\n'
                       '    P = features._det_probs(mean, std)\n'
                       '    pred = P.argmax(1)\n'
                       '    cm = np.zeros((4, 4), int)\n'
                       '    np.add.at(cm, (G, pred), 1)\n'
                       '    return (pred == G).mean(), P.max(1), pred, cm\n'
                       '\n'
                       '\n'
                       'def fit(C=0.1):\n'
                       '    from sklearn.linear_model import LogisticRegression\n'
                       '\n'
                       '    mt, st, gt = warmup_stats("datasets/train.parquet")\n'
                       '    Z = np.concatenate([mt, st], 1)\n'
                       '    mu, sc = Z.mean(0), Z.std(0)\n'
                       '    sc = np.where(sc > 1e-6, sc, 1.0)\n'
                       '    clf = LogisticRegression(C=C, max_iter=10000).fit(np.clip((Z - mu) / sc, -10, 10), gt)\n'
                       '    fmt = lambda a: np.array2string(np.asarray(a), precision=8, separator=", ", threshold=10 '
                       '** 6, max_line_width=116)\n'
                       '    block = ("# DETECTOR BEGIN\\n"\n'
                       '             f"# fitted on {len(gt)} train sequences, C={C}\\n"\n'
                       '             f"DET_MEAN = np.array({fmt(mu)})\\nDET_SCALE = np.array({fmt(sc)})\\n"\n'
                       '             f"DET_COEF = np.array({fmt(clf.coef_)})\\nDET_INTERCEPT = '
                       'np.array({fmt(clf.intercept_)})\\n"\n'
                       '             "# DETECTOR END")\n'
                       '    path = os.path.join(HERE, "features.py")\n'
                       '    src = open(path).read()\n'
                       '    src = re.sub(r"# DETECTOR BEGIN.*?# DETECTOR END", lambda _: block, src, flags=re.S)\n'
                       '    open(path, "w").write(src)\n'
                       '    print(f"detector written to {path} ({len(gt)} train sequences, train groups '
                       '{np.bincount(gt)})")\n'
                       '    exec(compile(block, "detector", "exec"), vars(features))  # use the new constants for the '
                       'check below\n'
                       '    a, conf, _, _ = accuracy(mt, st, gt)\n'
                       '    print(f"train accuracy {a:.4f}")\n'
                       '\n'
                       '\n'
                       'def check():\n'
                       '    mv, sv, gv = warmup_stats("datasets/valid.parquet")\n'
                       '    a, conf, pred, cm = accuracy(mv, sv, gv)\n'
                       '    wrong = pred != gv\n'
                       '    print(f"valid accuracy {a:.4f} ({int(wrong.sum())} of {len(gv)} wrong); confidence of the '
                       'wrong ones: "\n'
                       '          f"{np.round(np.sort(conf[wrong]), 3).tolist()}; correct ones below 0.99: '
                       '{int((conf[~wrong] < 0.99).sum())}")\n'
                       '    print(cm)\n'
                       '\n'
                       '\n'
                       'if __name__ == "__main__":\n'
                       '    ap = argparse.ArgumentParser()\n'
                       '    ap.add_argument("--fit", action="store_true")\n'
                       '    ap.add_argument("--check", action="store_true")\n'
                       '    ap.add_argument("--C", type=float, default=0.1)\n'
                       '    args = ap.parse_args()\n'
                       '    if args.fit:\n'
                       '        fit(args.C)\n'
                       '    if args.fit or args.check:\n'
                       '        check()\n',
 'pipeline/mask_model.py': '"""Scoring-mask model: predicts which rows the metric scores (is_scored) from causal '
                           'inputs only,\n'
                           'and turns it into per-row training weights.\n'
                           '\n'
                           '    python -m pipeline.mask_model fit   --out runs/mask_model          # train on '
                           'validation half A, AUC on half B\n'
                           '    python -m pipeline.mask_model score --model runs/mask_model --out mask_train.npy   # '
                           'p(scored) for train rows\n'
                           '\n'
                           "Validation is split once into halves A and B (split.json). The classifier sees only A's "
                           'masks, so\n'
                           'models can be compared on B without the mask information leaking into the comparison.\n'
                           'Output of `score`: float16 .npy [n_train_seqs, 20000], p(scored) per row (0 on warm-up '
                           'rows),\n'
                           'read by the trainer with Common.row_weights and ExpConfig.mask_weight.\n'
                           '"""\n'
                           'import argparse\n'
                           'import json\n'
                           'import os\n'
                           'import time\n'
                           '\n'
                           'import joblib\n'
                           'import numpy as np\n'
                           '\n'
                           'from utils import SEQUENCE_LENGTH, WARMUP\n'
                           '\n'
                           'from .data import SeqStore\n'
                           '\n'
                           'GROUPS = [slice(0, 11), slice(11, 22), slice(22, 33), slice(33, 44), slice(44, 48), '
                           'slice(48, 52),\n'
                           '          slice(52, 63), slice(63, 74), slice(74, 85), slice(85, 96), slice(96, 100), '
                           'slice(100, 104)]\n'
                           '\n'
                           '\n'
                           'def _trailing_mean(v, w):\n'
                           '    """Mean over the last min(w, t+1) rows, for every row t. v [B,T,F] -> [B,T,F]."""\n'
                           '    c = np.cumsum(v, 1, dtype=np.float64)\n'
                           '    c = np.concatenate([np.zeros_like(c[:, :1]), c], 1)\n'
                           '    t = np.arange(v.shape[1])\n'
                           '    lo = np.maximum(0, t + 1 - w)\n'
                           '    return ((c[:, t + 1] - c[:, lo]) / (t + 1 - lo)[None, :, None]).astype(np.float32)\n'
                           '\n'
                           '\n'
                           'def features(X):\n'
                           '    """Causal per-row features for rows WARMUP..T-1. X [B,T,112] -> [B, T-WARMUP, '
                           '142]."""\n'
                           '    d = np.abs(np.diff(X, axis=1, prepend=X[:, :1]))                     # |x_t - '
                           'x_{t-1}|, 0 at t=0\n'
                           '    dg = np.stack([d[..., g].mean(-1) for g in GROUPS], -1)               # activity per '
                           'book/trade group\n'
                           '    a = X[..., 106:109]                                                    # a2, a3, a4\n'
                           '    parts = [X, dg, _trailing_mean(a, 20), _trailing_mean(a, 200), _trailing_mean(dg, '
                           '20)]\n'
                           '    return np.concatenate(parts, -1)[:, WARMUP:]\n'
                           '\n'
                           '\n'
                           'def split_valid(n_seqs, seed=5):\n'
                           '    perm = np.random.default_rng(seed).permutation(n_seqs)\n'
                           '    half = n_seqs // 2\n'
                           '    return sorted(int(i) for i in perm[:half]), sorted(int(i) for i in perm[half:])\n'
                           '\n'
                           '\n'
                           'def predict_rows(model, X, chunk=8):\n'
                           '    """p(scored) [B, T] for a batch of sequences (0 on warm-up rows)."""\n'
                           '    out = np.zeros(X.shape[:2], np.float32)\n'
                           '    for i in range(0, len(X), chunk):\n'
                           '        f = features(X[i:i + chunk])\n'
                           '        out[i:i + chunk, WARMUP:] = model.predict_proba(f.reshape(-1, f.shape[-1]))[:, '
                           '1].reshape(f.shape[:2])\n'
                           '    return out\n'
                           '\n'
                           '\n'
                           'def fit(args):\n'
                           '    from sklearn.ensemble import HistGradientBoostingClassifier\n'
                           '    from sklearn.metrics import average_precision_score, roc_auc_score\n'
                           '\n'
                           '    rng = np.random.default_rng(0)\n'
                           '    st = SeqStore(os.path.join(args.data_dir, "valid.parquet"), args.threads)\n'
                           '    A, B = split_valid(st.n_seqs)\n'
                           '    os.makedirs(args.out, exist_ok=True)\n'
                           '    json.dump({"A": A, "B": B, "seed": 5}, open(os.path.join(args.out, "split.json"), '
                           '"w"))\n'
                           '\n'
                           '    t = time.time()\n'
                           '    F, Y = [], []\n'
                           '    for i in range(0, len(A), 32):  # sample rows of half A for training\n'
                           '        X, _, M = st.load(A[i:i + 32])\n'
                           '        f, m = features(X), M[:, WARMUP:]\n'
                           '        sel = rng.random(m.shape) < args.sample\n'
                           '        F.append(f[sel]); Y.append(m[sel])\n'
                           '    F, Y = np.concatenate(F), np.concatenate(Y)\n'
                           '    print(f"training rows {len(Y):,} from {len(A)} sequences (half A), load {time.time() - '
                           't:.0f}s", flush=True)\n'
                           '\n'
                           '    t = time.time()\n'
                           '    model = HistGradientBoostingClassifier(max_iter=args.trees, learning_rate=0.12, '
                           'max_leaf_nodes=63,\n'
                           '                                           early_stopping=False, random_state=0).fit(F, '
                           'Y)\n'
                           '    print(f"fit {time.time() - t:.0f}s", flush=True)\n'
                           '    joblib.dump(model, os.path.join(args.out, "model.joblib"))\n'
                           '\n'
                           '    # honest check on half B (sequences the classifier never saw)\n'
                           '    P, M = [], []\n'
                           '    for i in range(0, min(len(B), args.eval_seqs), 32):\n'
                           '        X, _, m = st.load(B[i:i + 32])\n'
                           '        P.append(predict_rows(model, X)[:, WARMUP:].ravel()); M.append(m[:, '
                           'WARMUP:].ravel())\n'
                           '    P, M = np.concatenate(P), np.concatenate(M)\n'
                           '    top10 = M[P >= np.quantile(P, 0.9)].sum() / M.sum()\n'
                           '    rep = {"auc_B": float(roc_auc_score(M, P)), "avg_precision_B": '
                           'float(average_precision_score(M, P)),\n'
                           '           "base_rate_B": float(M.mean()), "top10pct_share_of_scored": float(top10), '
                           '"eval_seqs": int(min(len(B), args.eval_seqs))}\n'
                           '    json.dump(rep, open(os.path.join(args.out, "report.json"), "w"), indent=1)\n'
                           '    print(json.dumps(rep, indent=1))\n'
                           '\n'
                           '\n'
                           'def score(args):\n'
                           '    model = joblib.load(os.path.join(args.model, "model.joblib"))\n'
                           '    st = SeqStore(os.path.join(args.data_dir, "train.parquet"), args.threads)\n'
                           '    n = args.limit or st.n_seqs\n'
                           '    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.float16, '
                           'shape=(st.n_seqs, SEQUENCE_LENGTH))\n'
                           '    if args.limit:\n'
                           '        out[n:] = 0\n'
                           '    t, done = time.time(), 0\n'
                           '    for X, _, _ in st.batches(list(range(n)), 64):\n'
                           '        out[done:done + len(X)] = predict_rows(model, X).astype(np.float16)\n'
                           '        done += len(X)\n'
                           '        if done % 640 == 0 or done == n:\n'
                           '            print(f"{done}/{n} sequences, {time.time() - t:.0f}s", flush=True)\n'
                           '    out.flush()\n'
                           '    print(f"mean p(scored) on required rows: {float(np.asarray(out[:n:50, WARMUP:], '
                           'np.float32).mean()):.4f} -> {args.out}")\n'
                           '\n'
                           '\n'
                           'def main():\n'
                           '    ap = argparse.ArgumentParser()\n'
                           '    ap.add_argument("cmd", choices=["fit", "score"])\n'
                           '    ap.add_argument("--out", required=True)\n'
                           '    ap.add_argument("--model", help="folder written by fit (for score)")\n'
                           '    ap.add_argument("--data-dir", default="datasets")\n'
                           '    ap.add_argument("--sample", type=float, default=0.1, help="fraction of half-A rows '
                           'used for fitting")\n'
                           '    ap.add_argument("--trees", type=int, default=150)\n'
                           '    ap.add_argument("--eval-seqs", type=int, default=400)\n'
                           '    ap.add_argument("--limit", type=int, default=None, help="score only the first N train '
                           'sequences (tests)")\n'
                           '    ap.add_argument("--threads", type=int, default=8)\n'
                           '    args = ap.parse_args()\n'
                           '    fit(args) if args.cmd == "fit" else score(args)\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    main()\n',
 'pipeline/model.py': '"""Stateful sequence models (GRU/LSTM, minGRU, causal TCN), metric-aligned losses, ONNX step '
                      'export and\n'
                      'baseline import. Every model has forward(x, h, x_prev) -> (pred, new_state) for chunked '
                      'training/evaluation;\n'
                      'minGRU and TCN also have stream_step() for the one-row ONNX graph (see '
                      'research/architectures_cnn_rnn_transformer.md)."""\n'
                      'import math\n'
                      '\n'
                      'import numpy as np\n'
                      'import torch\n'
                      'import torch.nn.functional as F\n'
                      'from torch import nn\n'
                      '\n'
                      'from .features import kind_for_width\n'
                      '\n'
                      'N_FEATURES = 112\n'
                      'CLIP = 2.0\n'
                      '\n'
                      '\n'
                      'def _model_inputs(x, x_prev, use_diff, n_extra):\n'
                      '    """The 112 features, then their row-to-row diffs (use_diff), then the extra causal features '
                      '(n_extra)."""\n'
                      '    base = x[..., :N_FEATURES]\n'
                      '    parts = [base]\n'
                      '    if use_diff:\n'
                      '        first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n'
                      '        prev = torch.cat([first, base[:, :-1]], 1)\n'
                      '        parts.append(base - prev)\n'
                      '    if n_extra:\n'
                      '        assert x.shape[-1] >= N_FEATURES + n_extra, "model needs the extra features as input"\n'
                      '        parts.append(x[..., N_FEATURES:N_FEATURES + n_extra])\n'
                      '    return torch.cat(parts, -1) if len(parts) > 1 else base\n'
                      '\n'
                      '\n'
                      'class GRUNet(nn.Module):\n'
                      '    """Stacked GRU (or LSTM with cell="lstm") + linear head. The recurrent module keeps the\n'
                      '    attribute name `gru` for either cell so older checkpoints load unchanged.\n'
                      '    n_aux > 0 adds a training-only head for auxiliary targets (trainer.aux_targets) on the same '
                      'hidden states:\n'
                      '    forward() keeps the last top-layer output in `last_out` for it; the exported graph only has '
                      '`head`."""\n'
                      '\n'
                      '    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0, cell="gru", '
                      'n_extra=0, feat_kind=None,\n'
                      '                 n_aux=0):\n'
                      '        super().__init__()\n'
                      '        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout, '
                      'cell=cell, n_extra=n_extra)\n'
                      '        self.config.update({k: v for k, v in dict(feat_kind=feat_kind, n_aux=n_aux).items() if '
                      'v})\n'
                      '        self.use_diff = use_diff\n'
                      '        self.cell = cell\n'
                      '        self.n_extra = n_extra  # extra causal features (pipeline/features.py), appended after '
                      'the 112 inputs\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        rnn = {"gru": nn.GRU, "lstm": nn.LSTM}[cell]\n'
                      '        self.gru = rnn(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 '
                      'else 0.0)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '        self.aux_head = nn.Linear(hidden, n_aux) if n_aux else None\n'
                      '        self.last_out = None\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h [L,B,H] or None; x_prev [B,>=112] = last row of the '
                      'previous\n'
                      '        chunk (None at sequence start, where the first diff is zero). Input order to the GRU:\n'
                      '        the 112 features, then their diffs (use_diff), then the extras (n_extra)."""\n'
                      '        base = x[..., :N_FEATURES]\n'
                      '        parts = [base]\n'
                      '        if self.use_diff:\n'
                      '            first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n'
                      '            prev = torch.cat([first, base[:, :-1]], 1)\n'
                      '            parts.append(base - prev)\n'
                      '        if self.n_extra:\n'
                      '            assert x.shape[-1] >= N_FEATURES + self.n_extra, "model needs the extra features as '
                      'input"\n'
                      '            parts.append(x[..., N_FEATURES:N_FEATURES + self.n_extra])\n'
                      '        out, h = self.gru(torch.cat(parts, -1) if len(parts) > 1 else base, h)\n'
                      '        if self.aux_head is not None:\n'
                      '            self.last_out = out\n'
                      '        return self.head(out), h\n'
                      '\n'
                      '\n'
                      'def _g(x):\n'
                      '    """minGRU candidate activation: x + 0.5 for x >= 0, sigmoid(x) below, so the candidate is '
                      'always positive."""\n'
                      '    return torch.where(x >= 0, x + 0.5, torch.sigmoid(x))\n'
                      '\n'
                      '\n'
                      'def _log_g(x):\n'
                      '    return torch.where(x >= 0, torch.log(F.relu(x) + 0.5), -F.softplus(-x))\n'
                      '\n'
                      '\n'
                      'class MinGRUNet(nn.Module):\n'
                      '    """Stack of minGRU layers (Feng et al. 2024, "Were RNNs All We Needed?") + linear head.\n'
                      '\n'
                      '    Per layer: z_t = sigmoid(W_z u_t), h_t = (1 - z_t) * h_{t-1} + z_t * g(W_h u_t). Gate and '
                      'candidate depend\n'
                      '    only on the layer input u_t (not on h_{t-1}), so training runs a parallel log-space scan '
                      'over each chunk\n'
                      '    (no step-by-step loop; long chunks are cheap), while inference stays an O(1) update per row '
                      'with ~1/3 of a GRU\n'
                      "    layer's multiply-adds. The nonlinear state interactions a GRU has come from stacking layers "
                      'instead.\n'
                      '    Gate biases start at log-spaced memory lengths of 2..2000 rows (chrono-style), so some '
                      'units remember long\n'
                      '    from the first step. State: one [B, H] tensor per layer."""\n'
                      '    BLOCK = 128  # scan block: keeps cumulative log-gates small enough for float32 (state is '
                      'carried across blocks)\n'
                      '\n'
                      '    def __init__(self, hidden=192, layers=3, use_diff=False, dropout=0.0, n_extra=0, '
                      'feat_kind=None):\n'
                      '        super().__init__()\n'
                      '        self.config = dict(arch="mingru", hidden=hidden, layers=layers, use_diff=use_diff, '
                      'dropout=dropout,\n'
                      '                           n_extra=n_extra, **({"feat_kind": feat_kind} if feat_kind else {}))\n'
                      '        self.use_diff, self.n_extra, self.hidden = use_diff, n_extra, hidden\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        self.proj = nn.ModuleList([nn.Linear(in_dim if l == 0 else hidden, 2 * hidden) for l in '
                      'range(layers)])\n'
                      '        self.drop = nn.Dropout(dropout)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '        with torch.no_grad():  # chrono init: 1 / z = memory length tau, log-uniform in [2, '
                      '2000]\n'
                      '            for p in self.proj:\n'
                      '                tau = torch.exp(torch.empty(hidden).uniform_(math.log(2.0), math.log(2000.0)))\n'
                      '                p.bias[:hidden] = -torch.log(tau - 1)\n'
                      '\n'
                      '    @classmethod\n'
                      '    def _scan(cls, k, c, h0):\n'
                      '        """h_t = a_t h_{t-1} + b_t with a = 1 - sigmoid(k), b = sigmoid(k) g(c), in log space '
                      'per block:\n'
                      '        h_t = A_t h0 + sum_{j<=t} (A_t / A_j) b_j, A_t = prod_{i<=t} a_i. k, c [B,T,H]; h0 '
                      '[B,H] or None (zeros)."""\n'
                      '        log_a = -F.softplus(k)\n'
                      '        log_b = -F.softplus(-k) + _log_g(c)\n'
                      '        outs = []\n'
                      '        for s in range(0, k.shape[1], cls.BLOCK):\n'
                      '            a_star = log_a[:, s:s + cls.BLOCK].cumsum(1)\n'
                      '            h = torch.exp(a_star + torch.logcumsumexp(log_b[:, s:s + cls.BLOCK] - a_star, 1))\n'
                      '            if h0 is not None:\n'
                      '                h = h + torch.exp(a_star) * h0[:, None]\n'
                      '            outs.append(h)\n'
                      '            h0 = h[:, -1]\n'
                      '        return torch.cat(outs, 1)\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h list of [B,H] per layer or None; x_prev as in GRUNet."""\n'
                      '        z = _model_inputs(x, x_prev, self.use_diff, self.n_extra)\n'
                      '        new = []\n'
                      '        for l, proj in enumerate(self.proj):\n'
                      '            k, c = proj(z).chunk(2, -1)\n'
                      '            z = self._scan(k, c, None if h is None else h[l])\n'
                      '            new.append(z[:, -1])\n'
                      '            if l < len(self.proj) - 1:\n'
                      '                z = self.drop(z)\n'
                      '        return self.head(z), new\n'
                      '\n'
                      '    def stream_states(self):\n'
                      '        return [(f"h{l}", (1, self.hidden)) for l in range(len(self.proj))]\n'
                      '\n'
                      '    def stream_step(self, z, states):\n'
                      '        """One row: z [1,1,in] (already built inputs), states list of [1,H] -> (pred [1,1,2], '
                      'new states)."""\n'
                      '        z, new = z[:, 0], []\n'
                      '        for proj, h in zip(self.proj, states):\n'
                      '            k, c = proj(z).chunk(2, -1)\n'
                      '            z = h + torch.sigmoid(k) * (_g(c) - h)\n'
                      '            new.append(z)\n'
                      '        return self.head(z)[:, None], new\n'
                      '\n'
                      '\n'
                      'class TCNNet(nn.Module):\n'
                      '    """Causal dilated temporal convolution network (Bai et al. 2018, WaveNet-style) + linear '
                      'head.\n'
                      '\n'
                      '    Input projection to `hidden` channels, then `layers` pre-norm residual blocks\n'
                      '    z <- z + W_mix relu(conv_{kernel, dilation}(LayerNorm(z))), dilation = kernel**l, so the '
                      'receptive field\n'
                      '    is kernel**layers rows (2**11 = 2048, 3**7 = 2187). Each output sees a fixed window of past '
                      'rows; unlike an\n'
                      '    RNN nothing older can leak in. State between chunks: the last (kernel - 1) * dilation '
                      'normalised inputs of\n'
                      '    each block ([B, pad_l, C]), zeros before the sequence start, so chunked training, '
                      'evaluation and row-by-row\n'
                      '    inference see exactly the same context."""\n'
                      '\n'
                      '    def __init__(self, hidden=64, layers=11, use_diff=False, dropout=0.0, n_extra=0, kernel=2, '
                      'feat_kind=None):\n'
                      '        super().__init__()\n'
                      '        assert kernel >= 2, "TCN kernel must be >= 2"\n'
                      '        self.config = dict(arch="tcn", hidden=hidden, layers=layers, use_diff=use_diff, '
                      'dropout=dropout,\n'
                      '                           n_extra=n_extra, kernel=kernel, **({"feat_kind": feat_kind} if '
                      'feat_kind else {}))\n'
                      '        self.use_diff, self.n_extra, self.hidden, self.kernel = use_diff, n_extra, hidden, '
                      'kernel\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        self.dilations = [kernel ** l for l in range(layers)]\n'
                      '        self.pads = [(kernel - 1) * d for d in self.dilations]\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        self.inp = nn.Linear(in_dim, hidden)\n'
                      '        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])\n'
                      '        self.convs = nn.ModuleList([nn.Conv1d(hidden, hidden, kernel, dilation=d) for d in '
                      'self.dilations])\n'
                      '        self.mix = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])\n'
                      '        self.drop = nn.Dropout(dropout)\n'
                      '        self.out_norm = nn.LayerNorm(hidden)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '\n'
                      '    @property\n'
                      '    def receptive_field(self):\n'
                      '        return 1 + sum(self.pads)\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h list of [B, pad_l, C] per block or None; x_prev as in '
                      'GRUNet."""\n'
                      '        z = self.inp(_model_inputs(x, x_prev, self.use_diff, self.n_extra))\n'
                      '        B, new = z.shape[0], []\n'
                      '        for l, (norm, conv, mix, pad) in enumerate(zip(self.norms, self.convs, self.mix, '
                      'self.pads)):\n'
                      '            u = norm(z)\n'
                      '            past = h[l] if h is not None else u.new_zeros(B, pad, u.shape[-1])\n'
                      '            full = torch.cat([past, u], 1)  # [B, pad + T, C]\n'
                      '            new.append(full[:, -pad:])\n'
                      '            y = conv(full.transpose(1, 2)).transpose(1, 2)  # [B, T, C]\n'
                      '            z = z + mix(self.drop(F.relu(y)))\n'
                      '        return self.head(self.out_norm(z)), new\n'
                      '\n'
                      '    def stream_states(self):\n'
                      '        return [(f"b{l}", (1, pad, self.hidden)) for l, pad in enumerate(self.pads)]\n'
                      '\n'
                      '    def stream_step(self, z, states):\n'
                      '        """One row: z [1,1,in], states list of [1, pad_l, C] -> (pred [1,1,2], new states). '
                      'Buffers are shifted\n'
                      '        with Concat + Slice (simple, but copies every buffer once per row: see the notes on '
                      'ring buffers)."""\n'
                      '        z, new = self.inp(z), []\n'
                      '        for norm, conv, mix, d, buf in zip(self.norms, self.convs, self.mix, self.dilations, '
                      'states):\n'
                      '            full = torch.cat([buf, norm(z)], 1)  # [1, pad + 1, C]\n'
                      '            new.append(full[:, 1:])\n'
                      "            taps = full[:, ::d]  # the kernel's inputs: rows t - (k-1)d, ..., t - d, t\n"
                      '            y = F.conv1d(taps.transpose(1, 2), conv.weight, conv.bias).transpose(1, 2)  # [1, '
                      '1, C]\n'
                      '            z = z + mix(F.relu(y))\n'
                      '        return self.head(self.out_norm(z)), new\n'
                      '\n'
                      '\n'
                      'class GroupGRUNet(nn.Module):\n'
                      '    """GRU conditioned on the instrument group. Input: the 112 features + the 4 detector '
                      'probabilities g\n'
                      '    (feature kind "grp"). The probabilities rescale and shift every input (FiLM: x * (1 + g '
                      'G_scale) + g G_shift)\n'
                      '    and mix 4 output heads (pred = sum_k g_k head_k(h)). G_scale / G_shift start at zero and '
                      'all heads equal, so\n'
                      '    `from_gru` gives exactly the plain GRU it starts from. The GRU layers are kept separate '
                      '(one state per layer,\n'
                      '    as in the exported plain GRU)."""\n'
                      '    N_GROUPS = 4\n'
                      '\n'
                      '    def __init__(self, hidden=128, layers=2, dropout=0.0, n_extra=4, feat_kind="grp", '
                      'use_diff=False):\n'
                      '        super().__init__()\n'
                      '        assert n_extra == self.N_GROUPS and feat_kind == "grp" and not use_diff\n'
                      '        self.config = dict(arch="ggru", hidden=hidden, layers=layers, dropout=dropout, '
                      'n_extra=n_extra,\n'
                      '                           feat_kind=feat_kind, use_diff=False)\n'
                      '        self.use_diff, self.n_extra, self.feat_kind, self.hidden = False, n_extra, feat_kind, '
                      'hidden\n'
                      '        self.film_scale = nn.Parameter(torch.zeros(self.N_GROUPS, N_FEATURES))\n'
                      '        self.film_shift = nn.Parameter(torch.zeros(self.N_GROUPS, N_FEATURES))\n'
                      '        self.layers = nn.ModuleList([nn.GRU(N_FEATURES if l == 0 else hidden, hidden, 1, '
                      'batch_first=True)\n'
                      '                                     for l in range(layers)])\n'
                      '        self.drop = nn.Dropout(dropout)\n'
                      '        self.heads = nn.Linear(hidden, 2 * self.N_GROUPS)\n'
                      '\n'
                      '    @torch.no_grad()\n'
                      '    def from_gru(self, gru_state):\n'
                      '        """Copy a plain GRUNet\'s weights (same hidden / layers, no extras): the result '
                      'predicts exactly the same."""\n'
                      '        for l, layer in enumerate(self.layers):\n'
                      '            for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):\n'
                      '                getattr(layer, f"{n}_l0").copy_(gru_state[f"gru.{n}_l{l}"])\n'
                      '        self.heads.weight.copy_(gru_state["head.weight"].repeat(self.N_GROUPS, 1))\n'
                      '        self.heads.bias.copy_(gru_state["head.bias"].repeat(self.N_GROUPS))\n'
                      '        return self\n'
                      '\n'
                      '    def _mix(self, out, g):\n'
                      '        heads = self.heads(out).unflatten(-1, (self.N_GROUPS, 2))  # [..., 4, 2]\n'
                      '        return (heads * g.unsqueeze(-1)).sum(-2)\n'
                      '\n'
                      '    def _film(self, x):\n'
                      '        base, g = x[..., :N_FEATURES], x[..., N_FEATURES:N_FEATURES + self.N_GROUPS]\n'
                      '        return base * (1 + g @ self.film_scale) + g @ self.film_shift, g\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        z, g = self._film(x)\n'
                      '        h = h if h is not None else [None] * len(self.layers)\n'
                      '        new = []\n'
                      '        for l, layer in enumerate(self.layers):\n'
                      '            z, hl = layer(self.drop(z) if l else z, h[l])\n'
                      '            new.append(hl)\n'
                      '        return self._mix(z, g), new\n'
                      '\n'
                      '    def stream_states(self):\n'
                      '        return [(f"h{l}", (1, 1, self.hidden)) for l in range(len(self.layers))]\n'
                      '\n'
                      '    def stream_step(self, x, states):\n'
                      '        z, g = self._film(x)\n'
                      '        new = []\n'
                      '        for layer, hl in zip(self.layers, states):\n'
                      '            z, hn = layer(z, hl)\n'
                      '            new.append(hn)\n'
                      '        return self._mix(z, g), new\n'
                      '\n'
                      '\n'
                      'ARCHS = {"gru": GRUNet, "mingru": MinGRUNet, "tcn": TCNNet, "ggru": GroupGRUNet}\n'
                      '\n'
                      '\n'
                      'def build_net(config):\n'
                      '    """Model from a saved config. GRU/LSTM configs predate the `arch` key, so it defaults to '
                      'gru."""\n'
                      '    cfg = dict(config)\n'
                      '    return ARCHS[cfg.pop("arch", "gru")](**cfg)\n'
                      '\n'
                      '\n'
                      'def needs_extra(net):\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    return max(getattr(m, "n_extra", 0) for m in members)\n'
                      '\n'
                      '\n'
                      'def feat_kind(net):\n'
                      '    """The feature kind a model (or every member of an ensemble) takes as extra inputs; None = '
                      'plain 112."""\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    kinds = {getattr(m, "feat_kind", None) for m in members}\n'
                      '    assert len(kinds - {None}) <= 1, f"members use different feature kinds: {kinds}"\n'
                      '    return max(kinds - {None}, default=None)\n'
                      '\n'
                      '\n'
                      'def detach_state(h):\n'
                      '    """Detach a recurrent state: tensor (GRU), (h, c) tuple (LSTM) or list of those '
                      '(ensemble)."""\n'
                      '    if isinstance(h, torch.Tensor):\n'
                      '        return h.detach()\n'
                      '    return type(h)(detach_state(x) for x in h)\n'
                      '\n'
                      '\n'
                      'def macs(config) -> int:\n'
                      '    """Multiply-adds per row (inference cost model). Ensembles: sum over members."""\n'
                      '    if "members" in config:\n'
                      '        return sum(macs(m) for m in config["members"])\n'
                      '    arch = config.get("arch", "gru")\n'
                      '    H, n_in = config["hidden"], N_FEATURES * (2 if config["use_diff"] else 1) + '
                      'config.get("n_extra", 0)\n'
                      '    if arch == "mingru":  # one [in -> 2H] projection per layer\n'
                      '        return sum(2 * H * (n_in if l == 0 else H) for l in range(config["layers"])) + 2 * H\n'
                      '    if arch == "tcn":  # input projection, per block a k-tap conv + 1x1 mix (LayerNorms '
                      'ignored)\n'
                      '        return n_in * H + config["layers"] * (config["kernel"] + 1) * H * H + 2 * H\n'
                      '    gates = 4 if config.get("cell", "gru") == "lstm" else 3\n'
                      '    total = 0\n'
                      '    if arch == "ggru":  # FiLM (2 x 4 x 112) + 4 heads on top of a plain GRU on the 112 inputs\n'
                      '        n_in = N_FEATURES\n'
                      '        total = 2 * 4 * N_FEATURES + 6 * H\n'
                      '    for _ in range(config["layers"]):\n'
                      '        total += gates * H * (n_in + H)\n'
                      '        n_in = H\n'
                      '    return total + 2 * H\n'
                      '\n'
                      '\n'
                      'BASELINE_MACS = macs(dict(hidden=128, layers=2, use_diff=False))  # 190,720\n'
                      '\n'
                      '\n'
                      'def _rw(w, rw):\n'
                      '    """Multiply metric weights [N,2] by optional per-row weights [N]."""\n'
                      '    return w if rw is None else w * rw[:, None]\n'
                      '\n'
                      '\n'
                      'def wmse_loss(p, y, rw=None):\n'
                      '    """Weighted MSE with the metric\'s clipping and |y| weights (times optional row weights). '
                      'p, y: [N, 2]."""\n'
                      '    y = y.clamp(-CLIP, CLIP)\n'
                      '    w = _rw(y.abs(), rw)\n'
                      '    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n'
                      '\n'
                      '\n'
                      'def distill_loss(p, teacher, y, weight="y", rw=None):\n'
                      '    """MSE to the teacher\'s predictions. Row weights: "y" = |clipped target| (like the '
                      'metric),\n'
                      '    "none" = uniform, "teacher" = |teacher prediction| (no dependence on the noisy target)."""\n'
                      '    if weight == "y":\n'
                      '        w = y.clamp(-CLIP, CLIP).abs()\n'
                      '    elif weight == "teacher":\n'
                      '        w = teacher.abs()\n'
                      '    elif weight == "none":\n'
                      '        w = torch.ones_like(teacher)\n'
                      '    else:\n'
                      '        raise ValueError(weight)\n'
                      '    w = _rw(w, rw)\n'
                      '    return ((w * (p - teacher) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n'
                      '\n'
                      '\n'
                      'def wpearson(p, y, rw=None):\n'
                      '    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> '
                      '[2]."""\n'
                      '    y = y.clamp(-CLIP, CLIP)\n'
                      '    w = _rw(y.abs(), rw)\n'
                      '    W = w.sum(0) + 1e-8\n'
                      '    yc = y - (w * y).sum(0) / W\n'
                      '    pc = p - (w * p).sum(0) / W\n'
                      '    cov = (w * yc * pc).sum(0)\n'
                      '    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)\n'
                      '\n'
                      '\n'
                      'def compute_loss(kind, p, y, alpha=0.1, rw=None):\n'
                      '    if kind == "wmse":\n'
                      '        return wmse_loss(p, y, rw)\n'
                      '    if kind == "pearson":\n'
                      '        return 1 - wpearson(p, y, rw).mean()\n'
                      '    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping\n'
                      '        return 1 - wpearson(p, y, rw).mean() + alpha * wmse_loss(p, y, rw)\n'
                      '    raise ValueError(kind)\n'
                      '\n'
                      '\n'
                      'class EnsembleNet(nn.Module):\n'
                      '    """Average of several GRUNets. Each member\'s output is multiplied by `scales[k]` (per '
                      'target)\n'
                      '    first, so members with different prediction spreads count equally. h is a list, one per '
                      'member."""\n'
                      '\n'
                      '    def __init__(self, nets, scales):\n'
                      '        super().__init__()\n'
                      '        self.nets = nn.ModuleList(nets)\n'
                      '        self.register_buffer("scales", torch.as_tensor(np.asarray(scales), '
                      'dtype=torch.float32))  # [K, 2]\n'
                      '        self.use_diff = any(n.use_diff for n in nets)\n'
                      '        self.config = {"members": [n.config for n in nets], "scales": '
                      'np.asarray(scales).tolist()}\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        h = h if h is not None else [None] * len(self.nets)\n'
                      '        preds, new_h = [], []\n'
                      '        for k, (net, hk) in enumerate(zip(self.nets, h)):\n'
                      '            p, hn = net(x, hk, x_prev if net.use_diff else None)\n'
                      '            preds.append(p * self.scales[k])\n'
                      '            new_h.append(hn)\n'
                      '        return torch.stack(preds).mean(0), new_h\n'
                      '\n'
                      '\n'
                      'def _split_layers(net):\n'
                      '    """Single-layer copies of a stacked GRU/LSTM, so the ONNX graph gets one state input per '
                      'layer\n'
                      '    (as in the baseline) instead of slicing/concatenating a stacked state on every row."""\n'
                      '    rnn, mods = net.gru, []\n'
                      '    for l in range(rnn.num_layers):\n'
                      '        m = type(rnn)(rnn.input_size if l == 0 else rnn.hidden_size, rnn.hidden_size, 1, '
                      'batch_first=True)\n'
                      '        for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):\n'
                      '            getattr(m, f"{n}_l0").data.copy_(getattr(rnn, f"{n}_l{l}").data)\n'
                      '        mods.append(m)\n'
                      '    return nn.ModuleList(mods)\n'
                      '\n'
                      '\n'
                      'def _member_states(m):\n'
                      '    """(name suffix, shape) of each streaming state of one model, in graph order."""\n'
                      '    if isinstance(m, GRUNet):\n'
                      '        H, lstm = m.config["hidden"], m.cell == "lstm"\n'
                      '        return [s for l in range(m.config["layers"])\n'
                      '                for s in [(f"h{l}", (1, 1, H))] + ([(f"c{l}", (1, 1, H))] if lstm else [])]\n'
                      '    return m.stream_states()\n'
                      '\n'
                      '\n'
                      'class _Step(nn.Module):\n'
                      '    """One-row step graph: (x [1,1,112], x_prev [1,112], *states) -> (pred [1,1,2], *new '
                      'states).\n'
                      '    States are flat, per member: GRU one h [1,1,H] per layer, LSTM h and c per layer, minGRU '
                      'one h [1,H] per\n'
                      '    layer, TCN one input buffer [1, pad, C] per block."""\n'
                      '\n'
                      '    def __init__(self, net):\n'
                      '        super().__init__()\n'
                      '        self.ensemble = isinstance(net, EnsembleNet)\n'
                      '        members = list(net.nets) if self.ensemble else [net]\n'
                      '        self.rnn = [isinstance(m, GRUNet) for m in members]\n'
                      '        self.layers = nn.ModuleList([_split_layers(m) if r else nn.ModuleList() for m, r in '
                      'zip(members, self.rnn)])\n'
                      '        self.heads = nn.ModuleList([m.head if r else nn.Identity() for m, r in zip(members, '
                      'self.rnn)])\n'
                      '        self.streams = nn.ModuleList([nn.Identity() if r else m for m, r in zip(members, '
                      'self.rnn)])\n'
                      '        self.n_states = [len(_member_states(m)) for m in members]\n'
                      '        self.diff = [m.use_diff for m in members]\n'
                      '        self.n_extra = [getattr(m, "n_extra", 0) for m in members]\n'
                      '        self.lstm = [r and m.cell == "lstm" for m, r in zip(members, self.rnn)]\n'
                      '        self.register_buffer("scales", net.scales if self.ensemble else torch.ones(1, 2))\n'
                      '\n'
                      '    def forward(self, x, x_prev, *states):\n'
                      '        states, new, preds, i = list(states), [], [], 0\n'
                      '        for k, layers in enumerate(self.layers):\n'
                      '            base = x[..., :N_FEATURES]\n'
                      '            parts = [base] + ([base - x_prev[:, None]] if self.diff[k] else []) + \\\n'
                      '                    ([x[..., N_FEATURES:N_FEATURES + self.n_extra[k]]] if self.n_extra[k] else '
                      '[])\n'
                      '            z = torch.cat(parts, -1) if len(parts) > 1 else base\n'
                      "            if not self.rnn[k]:  # minGRU / TCN: the model's own one-row update\n"
                      '                p, ns = self.streams[k].stream_step(z, states[i:i + self.n_states[k]])\n'
                      '                new += ns\n'
                      '                i += self.n_states[k]\n'
                      '            else:\n'
                      '                for layer in layers:\n'
                      '                    if self.lstm[k]:\n'
                      '                        z, (h, c) = layer(z, (states[i], states[i + 1]))\n'
                      '                        new += [h, c]\n'
                      '                        i += 2\n'
                      '                    else:\n'
                      '                        z, h = layer(z, states[i])\n'
                      '                        new.append(h)\n'
                      '                        i += 1\n'
                      '                p = self.heads[k](z)\n'
                      '            preds.append(p * self.scales[k] if self.ensemble else p)\n'
                      '        pred = preds[0] if len(preds) == 1 else torch.stack(preds).mean(0)\n'
                      '        return (pred, *new)\n'
                      '\n'
                      '\n'
                      'def state_names(net):\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    return [f"m{k}_{s}" for k, m in enumerate(members) for s, _ in _member_states(m)]\n'
                      '\n'
                      '\n'
                      'def export_onnx(net, path: str):\n'
                      '    """Export a model (GRUNet, MinGRUNet, TCNNet) or EnsembleNet as a one-row step graph. '
                      'Inputs: x, x_prev\n'
                      '    (dropped from the graph when no member uses diff features), then the states per member '
                      'named m{k}_<state>\n'
                      '    (h{l} / c{l} for GRU/LSTM and minGRU, b{l} for TCN buffers); outputs: pred, then the new '
                      'states in the same\n'
                      "    order. All states start at zero (solution.py zero-fills them from the graph's input "
                      'shapes)."""\n'
                      '    net = net.eval().cpu()\n'
                      '    feat_kind(net)  # raises if ensemble members need different extra features\n'
                      '    names = state_names(net)\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    states = [torch.zeros(*shape) for m in members for _, shape in _member_states(m)]\n'
                      '    args = (torch.zeros(1, 1, N_FEATURES + needs_extra(net)), torch.zeros(1, N_FEATURES), '
                      '*states)\n'
                      '    with torch.no_grad():\n'
                      '        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", *names],\n'
                      '                          output_names=["pred", *[n + "_out" for n in names]], '
                      'opset_version=17, dynamo=False)\n'
                      '\n'
                      '\n'
                      'def load_baseline(onnx_path: str) -> GRUNet:\n'
                      '    """Load the organisers\' 2-layer GRU ONNX weights into a GRUNet (for checks / warm '
                      'starts)."""\n'
                      '    import onnx\n'
                      '    from onnx import numpy_helper\n'
                      '\n'
                      '    g = onnx.load(onnx_path).graph\n'
                      '    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}\n'
                      '    grus = [n for n in g.node if n.op_type == "GRU"]\n'
                      '    matmul = next(n for n in g.node if n.op_type == "MatMul")\n'
                      '    H = 128\n'
                      '    net = GRUNet(hidden=H, layers=len(grus))\n'
                      '\n'
                      '    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n\n'
                      '        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)\n'
                      '\n'
                      '    sd = {}\n'
                      '    for l, n in enumerate(grus):\n'
                      '        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))\n'
                      '        sd[f"gru.weight_ih_l{l}"] = reorder(W)\n'
                      '        sd[f"gru.weight_hh_l{l}"] = reorder(R)\n'
                      '        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])\n'
                      '        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])\n'
                      '    sd["head.weight"] = init[matmul.input[1]].T\n'
                      '    sd["head.bias"] = init["reg_head.bias"]\n'
                      '    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in '
                      'sd.items()})\n'
                      '    return net\n',
 'pipeline/multi_train.py': '"""Train several experiments side by side from one data stream.\n'
                            '\n'
                            '    python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 '
                            '--cache-eval\n'
                            '\n'
                            'experiments.json is a list of ExpConfig dicts, e.g.\n'
                            '    [{"name": "wmse"}, {"name": "mix", "loss": "mix"}, {"name": "diff_h192", "diff": '
                            'true, "hidden": 192}]\n'
                            'Shared settings (epochs, batch, chunk, data/out dirs, ...) are the Common fields, as '
                            'flags.\n'
                            '"""\n'
                            'import argparse\n'
                            'import dataclasses\n'
                            'import json\n'
                            'import typing\n'
                            '\n'
                            'from .trainer import Common, ExpConfig, train_many\n'
                            '\n'
                            '\n'
                            'def field_type(f):\n'
                            '    """int | None -> int, str | None -> str, plain types unchanged."""\n'
                            '    args = [a for a in typing.get_args(f.type) if a is not type(None)]\n'
                            '    return args[0] if args else f.type\n'
                            '\n'
                            '\n'
                            'def add_flags(ap, cls, skip=("extra",)):\n'
                            '    for f in dataclasses.fields(cls):\n'
                            '        if f.name in skip:\n'
                            '            continue\n'
                            '        flag = "--" + f.name.replace("_", "-")\n'
                            '        if field_type(f) is bool:\n'
                            '            ap.add_argument(flag, action="store_true")\n'
                            '        else:\n'
                            '            ap.add_argument(flag, type=field_type(f), default=f.default)\n'
                            '\n'
                            '\n'
                            'def add_common_flags(ap):\n'
                            '    add_flags(ap, Common)\n'
                            '\n'
                            '\n'
                            'def main():\n'
                            '    ap = argparse.ArgumentParser()\n'
                            '    ap.add_argument("--config", required=True, help="JSON list of experiment configs")\n'
                            '    add_common_flags(ap)\n'
                            '    args = vars(ap.parse_args())\n'
                            '    configs = [ExpConfig(**c) for c in json.load(open(args.pop("config")))]\n'
                            '    names = [c.name for c in configs]\n'
                            '    assert len(set(names)) == len(names), "experiment names must be unique"\n'
                            '    train_many(configs, Common(**args))\n'
                            '\n'
                            '\n'
                            'if __name__ == "__main__":\n'
                            '    main()\n',
 'pipeline/routed.py': '"""One model per instrument group: score routings on validation and export a routed '
                       'submission.\n'
                       '\n'
                       '    python -m pipeline.routed --shared runs/sweep9/s9_h128/best.pt --cands '
                       '"runs/sweep10a/*/best.pt" \\\n'
                       '        --valid-split runs/mask_model/split.json --export runs/routed_10a\n'
                       '\n'
                       'Every model is run once over all validation sequences. Candidates trained on one group (their '
                       'saved ExpConfig\n'
                       'has `group`) compete for that group only; the others (and the shared model) for every group. '
                       'Besides the single\n'
                       'models, each group can take:\n'
                       "  * a blend: the average of the shared model and a candidate (2 models on that group's "
                       'sequences; costs add up);\n'
                       '  * a weight interpolation ("~wise"): weights halfway between the shared model and a fine-tune '
                       'of it (candidates\n'
                       "    whose ExpConfig `init` is the shared model; one model's cost).\n"
                       "Per group the route takes the option with the best scored-row WP on that group's half-A "
                       'sequences (within\n'
                       '--max-cost per sequence), so the half-B numbers stay an honest check. Routed scores use the '
                       "detector's groups\n"
                       '(features.detect, as solution.py does); the shared model takes sequences where the detector is '
                       'less sure than\n'
                       '--threshold. Also reported (not exported): a per-group affine calibration of the routed '
                       'predictions fitted on half A.\n'
                       '"""\n'
                       'import argparse\n'
                       'import copy\n'
                       'import glob\n'
                       'import json\n'
                       'import os\n'
                       'import shutil\n'
                       '\n'
                       'import numpy as np\n'
                       'import torch\n'
                       '\n'
                       'from utils import weighted_pearson\n'
                       '\n'
                       'from .data import SeqStore\n'
                       'from .fast_eval import load_ckpt, net_inputs, pick_device, predict_batch\n'
                       'from .features import detect\n'
                       'from .groups import label as group_label\n'
                       'from .model import BASELINE_MACS, EnsembleNet, GroupGRUNet, build_net, export_onnx, macs\n'
                       '\n'
                       'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                       '\n'
                       '\n'
                       'def wp(y, p):\n'
                       '    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))\n'
                       '\n'
                       '\n'
                       'def collect(nets, store, device):\n'
                       '    """-> Y [list per seq of scored rows], P [model][seq], true groups, detector probs '
                       '[n_seqs, 4]."""\n'
                       '    Y, P, G, D = [], [[] for _ in nets], [], []\n'
                       '    for X, Yb, M in store.batches(list(range(store.n_seqs)), 128):\n'
                       '        cache = {}\n'
                       '        G.append(group_label(Yb))\n'
                       '        D.append(detect(X))\n'
                       '        Y += [Yb[b][M[b]] for b in range(len(X))]\n'
                       '        for k, net in enumerate(nets):\n'
                       '            Pb = predict_batch(net, net_inputs(net, X, cache), device)\n'
                       '            P[k] += [Pb[b][M[b]] for b in range(len(X))]\n'
                       '    return Y, P, np.concatenate(G), np.concatenate(D)\n'
                       '\n'
                       '\n'
                       'def interpolate(shared, net, alpha=0.5):\n'
                       '    """Weights (1 - alpha) * shared + alpha * net, for a fine-tune of `shared` (a ggru '
                       'fine-tune: from the ggru copy\n'
                       '    of the shared GRU). None when the shapes do not match."""\n'
                       '    base = GroupGRUNet(**{k: v for k, v in net.config.items() if k != '
                       '"arch"}).from_gru(shared.state_dict()) \\\n'
                       '        if isinstance(net, GroupGRUNet) and not isinstance(shared, GroupGRUNet) else shared\n'
                       '    sa, sb = base.state_dict(), net.state_dict()\n'
                       '    if sa.keys() != sb.keys() or any(sa[k].shape != sb[k].shape for k in sa):\n'
                       '        return None\n'
                       '    out = build_net(net.config)\n'
                       '    out.load_state_dict({k: (1 - alpha) * sa[k].float() + alpha * sb[k].float() for k in sa})\n'
                       '    return out\n'
                       '\n'
                       '\n'
                       'def main():\n'
                       '    ap = argparse.ArgumentParser()\n'
                       '    ap.add_argument("--shared", required=True, help="best.pt of the shared model (also the '
                       'fallback)")\n'
                       '    ap.add_argument("--cands", nargs="+", required=True, help="best.pt paths/globs of '
                       'candidate models")\n'
                       '    ap.add_argument("--valid-split", required=True)\n'
                       '    ap.add_argument("--threshold", type=float, default=0.6, help="detector confidence below '
                       'this -> shared model")\n'
                       '    ap.add_argument("--max-cost", type=float, default=2.2, help="max cost (x baseline '
                       'multiply-adds) per sequence")\n'
                       '    ap.add_argument("--no-blends", action="store_true")\n'
                       '    ap.add_argument("--export", help="folder for the routed submission")\n'
                       '    ap.add_argument("--device", default="auto")\n'
                       '    ap.add_argument("--threads", type=int, default=8)\n'
                       '    args = ap.parse_args()\n'
                       '\n'
                       '    paths = [args.shared] + [p for c in args.cands for p in sorted(glob.glob(c))\n'
                       '                             if os.path.realpath(p) != os.path.realpath(args.shared)]\n'
                       '    names = [os.path.basename(os.path.dirname(p)) for p in paths]\n'
                       '    exps = [{}] + [torch.load(p, map_location="cpu", weights_only=False).get("exp") or {} for '
                       'p in paths[1:]]\n'
                       '    groups_of = [None] + [e.get("group") for e in exps[1:]]  # None = all groups\n'
                       '    nets = [load_ckpt(p).eval() for p in paths]\n'
                       '    for k in range(1, len(paths)):  # weight interpolations of fine-tunes of the shared model\n'
                       '        init = exps[k].get("init")\n'
                       '        if init and os.path.realpath(init) == os.path.realpath(args.shared):\n'
                       '            mid = interpolate(nets[0], nets[k])\n'
                       '            if mid is not None:\n'
                       '                nets.append(mid.eval()); names.append(names[k] + "~wise"); '
                       'groups_of.append(groups_of[k])\n'
                       '                paths.append(f"{paths[k]} (weights 0.5 / 0.5 with the shared model)")\n'
                       '    device = pick_device(args.device)\n'
                       '    nets = [n.to(device) for n in nets]\n'
                       '    cost = [macs(n.config) / BASELINE_MACS for n in nets]\n'
                       '    print(f"{len(nets)} models: " + ", ".join(f"{n}" + (f" (group {g + 1})" if g is not None '
                       'else "")\n'
                       '                                              for n, g in zip(names, groups_of)), flush=True)\n'
                       '\n'
                       '    Y, P, G, D = collect(nets, SeqStore("datasets/valid.parquet", args.threads), device)\n'
                       '    n_single = len(nets)\n'
                       '    # options: the single models, then blends shared + candidate (prediction average)\n'
                       '    opts = [{"name": n, "members": [k], "group": g, "cost": c} for k, (n, g, c) in '
                       'enumerate(zip(names, groups_of, cost))]\n'
                       '    if not args.no_blends:\n'
                       '        for k in range(1, n_single):\n'
                       '            if cost[0] + cost[k] <= args.max_cost:\n'
                       '                P.append([(a + b) / 2 for a, b in zip(P[0], P[k])])\n'
                       '                opts.append({"name": f"shared+{names[k]}", "members": [0, k], "group": '
                       'groups_of[k], "cost": cost[0] + cost[k]})\n'
                       '    split = json.load(open(args.valid_split))\n'
                       '    half = np.zeros(len(G), int)\n'
                       '    half[split["B"]] = 1  # 0 = A, 1 = B\n'
                       '    det, conf = D.argmax(1), D.max(1)\n'
                       '    print(f"detector: {(det == G).mean():.4f} of {len(G)} valid sequences right; "\n'
                       '          f"{int((conf < args.threshold).sum())} below the threshold {args.threshold} (-> '
                       'shared model)", flush=True)\n'
                       '\n'
                       '    def pooled(model_of_seq, seqs):\n'
                       '        """Pooled scored-row WP over `seqs`, each predicted by option model_of_seq[s]."""\n'
                       '        seqs = [s for s in seqs if len(Y[s])]\n'
                       '        if not seqs:  # no sequence of that kind in this half\n'
                       '            return float("nan")\n'
                       '        return wp(np.concatenate([Y[s] for s in seqs]), np.concatenate([P[model_of_seq[s]][s] '
                       'for s in seqs]))\n'
                       '\n'
                       '    idx = {h: {g: np.flatnonzero((half == h) & (G == g)) for g in range(4)} for h in (0, 1)}\n'
                       '    per = {}  # (option, half, group) -> WP\n'
                       '    print(f"\\nscored-row WP per group (A = selection half, B = honest half)")\n'
                       '    print(f"{\'option\':34s} {\'cost\':>5s} " + " ".join(f"A_g{g + 1:<5d} B_g{g + 1:<5d}" for '
                       'g in range(4)) + "  B_all")\n'
                       '    for k, o in enumerate(opts):\n'
                       '        cells = []\n'
                       '        for g in range(4):\n'
                       '            ok = o["group"] is None or o["group"] == g\n'
                       '            for h in (0, 1):\n'
                       '                per[k, h, g] = pooled(np.full(len(G), k), idx[h][g]) if ok else np.nan\n'
                       '                cells.append(f"{per[k, h, g]:.4f}" if ok else "   -  ")\n'
                       '        b_all = pooled(np.full(len(G), k), np.flatnonzero(half == 1)) if o["group"] is None '
                       'else np.nan\n'
                       '        print(f"{o[\'name\'][:34]:34s} {o[\'cost\']:5.2f} " + " ".join(f"{a:7s} {b:7s}" for a, '
                       'b in zip(cells[::2], cells[1::2]))\n'
                       '              + (f"  {b_all:.4f}" if o["group"] is None else ""), flush=True)\n'
                       '\n'
                       '    def best_route(max_cost):\n'
                       '        route = []\n'
                       '        for g in range(4):\n'
                       '            ok = [k for k, o in enumerate(opts) if (o["group"] is None or o["group"] == g) and '
                       'o["cost"] <= max_cost]\n'
                       '            route.append(max(ok, key=lambda k: np.nan_to_num(per[k, 0, g], nan=-1.0)))\n'
                       '        return route\n'
                       '\n'
                       '    def routed(route, use_detector=True):\n'
                       '        if not use_detector:\n'
                       '            return np.array([route[g] for g in G])\n'
                       '        return np.array([route[d] if c >= args.threshold else 0 for d, c in zip(det, conf)])\n'
                       '\n'
                       '    share = np.bincount(det, minlength=4) / len(det)  # valid mix; the train mix has more of '
                       'groups 3-4\n'
                       '    share_train = np.array([0.123, 0.119, 0.329, 0.430])\n'
                       '    avg_cost = lambda route: (float(share @ [opts[k]["cost"] for k in route]),\n'
                       '                              float(share_train @ [opts[k]["cost"] for k in route]))\n'
                       '    choice = best_route(args.max_cost)\n'
                       '    single = best_route(max(cost) + 1e-9) if args.no_blends else \\\n'
                       '        [max([k for k in range(n_single) if groups_of[k] in (None, g)], key=lambda k: '
                       'np.nan_to_num(per[k, 0, g], nan=-1.0))\n'
                       '         for g in range(4)]\n'
                       '    for title, r in (("route", choice), ("route without blends", single)):\n'
                       '        print(f"\\n{title} (best on half A per group): " + ", ".join(f"g{g + 1} -> '
                       '{opts[k][\'name\']}" for g, k in enumerate(r))\n'
                       '              + "; average cost %.2f (valid mix) / %.2f (train mix)" % avg_cost(r))\n'
                       '\n'
                       '    B = np.flatnonzero(half == 1)\n'
                       '    rows = [("shared alone", np.zeros(len(G), int))]\n'
                       '    rows += [(f"{o[\'name\']} alone", np.full(len(G), k)) for k, o in '
                       'enumerate(opts[1:n_single], 1) if o["group"] is None]\n'
                       '    rows += [("routed, true groups", routed(choice, False)), ("routed, detected groups", '
                       'routed(choice)),\n'
                       '             ("routed without blends, detected", routed(single))]\n'
                       '    for g in range(4):  # one group routed at a time, the shared model elsewhere\n'
                       '        only = [0] * 4\n'
                       '        only[g] = choice[g]\n'
                       '        if choice[g]:\n'
                       '            rows.append((f"route only g{g + 1} ({opts[choice[g]][\'name\'][:16]})", '
                       'routed(only)))\n'
                       '    print(f"\\n{\'combination\':44s} {\'valid_B\':>8s} {\'valid_A\':>8s} " + " '
                       '".join(f"{\'B_g\' + str(g + 1):>7s}" for g in range(4)))\n'
                       '    results = {}\n'
                       '    for name, m in rows:\n'
                       '        rb, ra = pooled(m, B), pooled(m, np.flatnonzero(half == 0))\n'
                       '        bg = [pooled(m, idx[1][g]) for g in range(4)]\n'
                       '        results[name] = {"valid_B": rb, "valid_A": ra, **{f"B_g{g + 1}": v for g, v in '
                       'enumerate(bg)}}\n'
                       '        print(f"{name[:44]:44s} {rb:8.5f} {ra:8.5f} " + " ".join(f"{v:7.4f}" for v in bg), '
                       'flush=True)\n'
                       '\n'
                       '    # per-group affine calibration of the routed predictions (fitted on half A, |y|-weighted '
                       'least squares)\n'
                       '    m = routed(choice)\n'
                       '    cal = {}\n'
                       '    for g in range(4):\n'
                       '        sa = [s for s in idx[0][g] if len(Y[s])]\n'
                       '        p, y = np.concatenate([P[m[s]][s] for s in sa]), np.clip(np.concatenate([Y[s] for s in '
                       'sa]), -2, 2)\n'
                       '        cal[g] = []\n'
                       '        for j in range(2):\n'
                       '            Z, w = np.stack([p[:, j], np.ones(len(p))], 1), np.abs(y[:, j])\n'
                       '            cal[g].append(np.linalg.solve((Z * w[:, None]).T @ Z, (Z * w[:, None]).T @ y[:, '
                       'j]))\n'
                       '    sb = [s for s in B if len(Y[s])]\n'
                       '    yb = np.concatenate([Y[s] for s in sb])\n'
                       '    pb = np.concatenate([np.stack([cal[det[s]][j][0] * P[m[s]][s][:, j] + cal[det[s]][j][1] '
                       'for j in range(2)], 1) for s in sb])\n'
                       '    results["routed + per-group calibration (not exported)"] = {"valid_B": wp(yb, pb)}\n'
                       '    print(f"{\'routed + per-group calibration (not exported)\':44s} {wp(yb, pb):8.5f}   slopes '
                       '"\n'
                       '          + ", ".join(f"g{g + 1} {cal[g][0][0]:.2f}/{cal[g][1][0]:.2f}" for g in range(4)), '
                       'flush=True)\n'
                       '\n'
                       '    if args.export:\n'
                       '        used = sorted(set(choice) | {0})  # the shared model is always there as the fallback\n'
                       '        os.makedirs(args.export, exist_ok=True)\n'
                       '        files = []\n'
                       '        for j, k in enumerate(used):\n'
                       '            mem = [nets[i].cpu() for i in opts[k]["members"]]\n'
                       '            net = mem[0] if len(mem) == 1 else EnsembleNet([copy.deepcopy(x) for x in mem], '
                       'np.ones((len(mem), 2)))\n'
                       '            files.append(f"model_{j}.onnx")\n'
                       '            export_onnx(net, os.path.join(args.export, files[-1]))\n'
                       '        routes = {"models": files, "route": [used.index(k) for k in choice], "fallback": '
                       'used.index(0),\n'
                       '                  "threshold": args.threshold}\n'
                       '        json.dump(routes, open(os.path.join(args.export, "routes.json"), "w"), indent=1)\n'
                       '        shutil.copy(os.path.join(HERE, "solution_routed.py"), os.path.join(args.export, '
                       '"solution.py"))\n'
                       '        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(args.export, '
                       '"wnn_runner.py"))\n'
                       '        shutil.copy(os.path.join(HERE, "features.py"), os.path.join(args.export, '
                       '"wnn_features.py"))\n'
                       '        json.dump({"models": {f: [paths[i] for i in opts[k]["members"]] for f, k in zip(files, '
                       'used)},\n'
                       '                   "route_names": [opts[k]["name"] for k in choice], "cost": {opts[k]["name"]: '
                       'round(opts[k]["cost"], 3) for k in used},\n'
                       '                   "average_cost_valid_train_mix": avg_cost(choice), "scores": results},\n'
                       '                  open(os.path.join(args.export, "routing.json"), "w"), indent=1)\n'
                       '        print(f"exported {len(files)} models -> {args.export} (route {routes[\'route\']}, '
                       'fallback {routes[\'fallback\']})")\n'
                       '\n'
                       '\n'
                       'if __name__ == "__main__":\n'
                       '    main()\n',
 'pipeline/solution_routed.py': '"""Routed submission: one model per instrument group. Copied as solution.py by '
                                'pipeline/routed.py, next to\n'
                                'routes.json, the model files, wnn_runner.py (= solution_template.py) and '
                                'wnn_features.py.\n'
                                '\n'
                                'During the warm-up rows (no prediction needed) the group detector (wnn_features '
                                '"grp") sees the rows and they are\n'
                                "buffered. At the first row that needs a prediction the detector picks the group's "
                                'model (or the fallback model when\n'
                                "it is less sure than routes.json's threshold), the buffered rows are replayed through "
                                'that model only, and from then\n'
                                'on only that model runs."""\n'
                                'import json\n'
                                'import sys\n'
                                'from pathlib import Path\n'
                                '\n'
                                'HERE = Path(__file__).resolve().parent\n'
                                'sys.path.insert(0, str(HERE))\n'
                                '\n'
                                'import numpy as np\n'
                                'import wnn_features\n'
                                'from wnn_runner import PredictionModel as Runner\n'
                                '\n'
                                '\n'
                                'class PredictionModel:\n'
                                '    def __init__(self):\n'
                                '        cfg = json.load(open(HERE / "routes.json"))\n'
                                '        self.runners = [Runner(str(HERE / f)) for f in cfg["models"]]  # absolute: '
                                'wnn_runner may come from another folder\n'
                                '        self.route, self.fallback, self.threshold = cfg["route"], cfg["fallback"], '
                                'cfg["threshold"]\n'
                                '        self.detector = wnn_features.Online("grp")\n'
                                '        self.seq_ix = None\n'
                                '\n'
                                '    def predict(self, data_point):\n'
                                '        if data_point.seq_ix != self.seq_ix:\n'
                                '            self.seq_ix = data_point.seq_ix\n'
                                '            for r in self.runners:\n'
                                '                r.reset()\n'
                                '            self.detector.reset()\n'
                                '            self.active, self.probs, self.buffer = None, None, []\n'
                                '        if self.active is None:\n'
                                '            self.probs = self.detector(data_point.state)\n'
                                '            if not data_point.need_prediction:\n'
                                '                self.buffer.append(np.array(data_point.state, dtype=np.float32))  # a '
                                'copy: the caller may reuse its array\n'
                                '                return None\n'
                                '            g = int(np.argmax(self.probs))\n'
                                '            self.active = self.runners[self.route[g] if self.probs[g] >= '
                                'self.threshold else self.fallback]\n'
                                '            for row in self.buffer:  # catch the chosen model up on the warm-up rows\n'
                                '                self.active.step(row)\n'
                                '            self.buffer = []\n'
                                '        pred = self.active.step(data_point.state)\n'
                                '        return pred if data_point.need_prediction else None\n',
 'pipeline/solution_template.py': '"""Generic stateful ONNX step runner. Copied as solution.py next to model.onnx by '
                                  'the pipeline.\n'
                                  'Works for single models and ensembles: every graph input other than x / x_prev is a '
                                  'recurrent\n'
                                  'state, and graph outputs[1:] are their updates in the same order. The routed '
                                  'submission (solution_routed.py)\n'
                                  'reuses this file as wnn_runner.py: one PredictionModel(model_file) per model, '
                                  'driven through reset() / step()."""\n'
                                  'from pathlib import Path\n'
                                  'import os\n'
                                  'import sys\n'
                                  '\n'
                                  'os.environ.setdefault("OMP_NUM_THREADS", "1")\n'
                                  'os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\n'
                                  'os.environ.setdefault("MKL_NUM_THREADS", "1")\n'
                                  '\n'
                                  'import numpy as np\n'
                                  'import onnxruntime as ort\n'
                                  '\n'
                                  '\n'
                                  'class PredictionModel:\n'
                                  '    def __init__(self, model_file="model.onnx"):\n'
                                  '        o = ort.SessionOptions()\n'
                                  '        o.intra_op_num_threads = 1\n'
                                  '        o.inter_op_num_threads = 1\n'
                                  '        o.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL\n'
                                  '        o.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n'
                                  '        o.add_session_config_entry("session.intra_op.allow_spinning", "0")\n'
                                  '        self.session = ort.InferenceSession(str(Path(__file__).resolve().parent / '
                                  'model_file),\n'
                                  '                                            sess_options=o, '
                                  'providers=["CPUExecutionProvider"])\n'
                                  '        inputs = self.session.get_inputs()\n'
                                  '        self.state_names = [i.name for i in inputs if i.name not in ("x", '
                                  '"x_prev")]\n'
                                  '        self.state_init = [np.zeros(i.shape, dtype=np.float32) for i in inputs if '
                                  'i.name in self.state_names]\n'
                                  '        self.use_prev = any(i.name == "x_prev" for i in inputs)  # absent when no '
                                  'model uses diff features\n'
                                  '        self.out_names = [o.name for o in self.session.get_outputs()]\n'
                                  '        x_width = next(i.shape[-1] for i in inputs if i.name == "x")\n'
                                  '        self.features = None\n'
                                  '        if x_width > 112:  # the model also takes the extra causal features '
                                  '(wnn_features.py, shipped alongside)\n'
                                  '            sys.path.insert(0, str(Path(__file__).resolve().parent))\n'
                                  '            import wnn_features\n'
                                  '            kinds = getattr(wnn_features, "KINDS", {"act": wnn_features.N_EXTRA})\n'
                                  '            kind = next((k for k, w in kinds.items() if 112 + w == x_width), None)\n'
                                  '            assert kind, "wnn_features.py does not match model.onnx"\n'
                                  '            self.features = wnn_features.Online(kind) if hasattr(wnn_features, '
                                  '"KINDS") else wnn_features.Online()\n'
                                  '        self.seq_ix = None\n'
                                  '\n'
                                  '    def reset(self):\n'
                                  '        self.feeds = dict(zip(self.state_names, [s.copy() for s in '
                                  'self.state_init]))\n'
                                  '        self.x_prev = None\n'
                                  '        if self.features is not None:\n'
                                  '            self.features.reset()\n'
                                  '\n'
                                  '    def predict(self, data_point):\n'
                                  '        if data_point.seq_ix != self.seq_ix:\n'
                                  '            self.seq_ix = data_point.seq_ix\n'
                                  '            self.reset()\n'
                                  '        pred = self.step(data_point.state)\n'
                                  '        if not data_point.need_prediction:\n'
                                  '            return None\n'
                                  '        return pred\n'
                                  '\n'
                                  '    def step(self, row):\n'
                                  '        """One row of the current sequence -> prediction [2]."""\n'
                                  '        feeds = self.feeds\n'
                                  '        state = np.asarray(row, dtype=np.float32)\n'
                                  '        x = state.reshape(1, 1, 112)\n'
                                  '        if self.use_prev:\n'
                                  '            feeds["x_prev"] = x[0] if self.x_prev is None else self.x_prev\n'
                                  '            self.x_prev = x[0]\n'
                                  '        if self.features is not None:\n'
                                  '            x = np.concatenate((state, self.features(state))).reshape(1, 1, -1)\n'
                                  '        feeds["x"] = x\n'
                                  '        out = self.session.run(self.out_names, feeds)\n'
                                  '        for name, value in zip(self.state_names, out[1:]):\n'
                                  '            feeds[name] = value\n'
                                  '        pred = out[0].reshape(2)\n'
                                  '        if not np.isfinite(pred).all():  # never seen; keep the submission valid '
                                  'and restart the state\n'
                                  '            self.reset()\n'
                                  '            pred = np.zeros(2, dtype=np.float32)\n'
                                  '        return pred\n',
 'pipeline/speed.py': '"""Time submission folders row by row on one CPU thread, next to the official baseline.\n'
                      '\n'
                      '    python -m pipeline.speed runs/NAME/submission runs/ens/submission\n'
                      '\n'
                      "The test set is 39.4M rows in 60 min (~91 us/row on the grader's CPU). The baseline is known "
                      'to\n'
                      'fit, so the ratio to the baseline is the number to watch; absolute times depend on this '
                      'machine.\n'
                      '"""\n'
                      'import os\n'
                      '\n'
                      'for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n'
                      '    os.environ[_v] = "1"\n'
                      '\n'
                      'import argparse\n'
                      'import importlib.util\n'
                      'import sys\n'
                      'import time\n'
                      '\n'
                      'import numpy as np\n'
                      'import pyarrow.parquet as pq\n'
                      '\n'
                      'from utils import FEATURE_COLUMNS, WARMUP, DataPoint\n'
                      '\n'
                      'TEST_ROWS = 39_400_000\n'
                      '\n'
                      '\n'
                      'def load_model(folder, i):\n'
                      '    folder = os.path.abspath(folder)\n'
                      '    sys.path.insert(0, folder)\n'
                      '    spec = importlib.util.spec_from_file_location(f"solution_{i}", os.path.join(folder, '
                      '"solution.py"))\n'
                      '    mod = importlib.util.module_from_spec(spec)\n'
                      '    spec.loader.exec_module(mod)\n'
                      '    return mod.PredictionModel()\n'
                      '\n'
                      '\n'
                      'def main():\n'
                      '    ap = argparse.ArgumentParser()\n'
                      '    ap.add_argument("folders", nargs="+")\n'
                      '    ap.add_argument("--rows", type=int, default=20000)\n'
                      '    args = ap.parse_args()\n'
                      '\n'
                      '    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, '
                      'columns=list(FEATURE_COLUMNS))\n'
                      '    X = np.column_stack([tbl[c].to_numpy() for c in '
                      'FEATURE_COLUMNS]).astype(np.float32)[:args.rows]\n'
                      '\n'
                      '    results = []\n'
                      '    for i, folder in enumerate(["baseline"] + args.folders):\n'
                      '        model = load_model(folder, i)\n'
                      '        t = time.perf_counter()\n'
                      '        for step in range(len(X)):\n'
                      '            model.predict(DataPoint(0, step, step >= WARMUP, X[step]))\n'
                      '        us = (time.perf_counter() - t) / len(X) * 1e6\n'
                      '        results.append((folder, us))\n'
                      '        base = results[0][1]\n'
                      '        print(f"{folder:50s} {us:7.1f} us/row  {us / base:5.2f}x baseline  "\n'
                      '              f"~{us * TEST_ROWS / 60e6:5.1f} min for the test set here", flush=True)\n'
                      '\n'
                      '\n'
                      'if __name__ == "__main__":\n'
                      '    main()\n',
 'pipeline/teacher.py': '"""Write a teacher\'s predictions on every training sequence, as the target for '
                        'distillation.\n'
                        '\n'
                        '    python -m pipeline.teacher --ensemble runs/ens/cost99/submission/ensemble.json --out '
                        'teacher.npy\n'
                        '    python -m pipeline.teacher --ckpts runs/sweep2/s1_wmse_h128/best.pt '
                        'runs/sweep2/c3_*/best.pt --out t_in2.npy\n'
                        '    python -m pipeline.teacher --ckpts "runs/sweep4/f*/best.pt" --oof --out t_oof.npy\n'
                        '\n'
                        'Output: float16 .npy of shape [n_train_seqs, 20000, 2] (~850 MB), indexed by train.parquet '
                        'row group,\n'
                        'read by the trainer (Common.teacher / ExpConfig.teacher).\n'
                        '\n'
                        'In-sample (default): every member predicts every sequence, including the ones it trained on.\n'
                        '--oof: members were trained with ExpConfig.fold; sequence i is predicted only by the members '
                        'whose\n'
                        'left-out fold is i % n_folds, so every training sequence gets honest (out-of-fold) '
                        'predictions.\n'
                        'With --ckpts, members are rescaled to a common prediction spread (measured on the first '
                        'batch).\n'
                        '"""\n'
                        'import argparse\n'
                        'import glob\n'
                        'import json\n'
                        'import time\n'
                        '\n'
                        'import numpy as np\n'
                        'import torch\n'
                        '\n'
                        'from .data import SeqStore\n'
                        'from .fast_eval import load_ckpt, net_inputs, pick_device, predict_batch\n'
                        '\n'
                        '\n'
                        'def main():\n'
                        '    ap = argparse.ArgumentParser()\n'
                        '    src = ap.add_mutually_exclusive_group(required=True)\n'
                        '    src.add_argument("--ensemble", help="ensemble.json written by pipeline.ensemble '
                        '--export")\n'
                        '    src.add_argument("--ckpts", nargs="+", help="best.pt paths/globs, averaged")\n'
                        '    ap.add_argument("--oof", action="store_true", help="out-of-fold: use each member only on '
                        'its left-out fold")\n'
                        '    ap.add_argument("--out", required=True)\n'
                        '    ap.add_argument("--data", default="datasets/train.parquet")\n'
                        '    ap.add_argument("--limit", type=int, default=None, help="only predict the first N '
                        'sequences (tests)")\n'
                        '    ap.add_argument("--threads", type=int, default=8)\n'
                        '    ap.add_argument("--device", default="auto")\n'
                        '    args = ap.parse_args()\n'
                        '    device = pick_device(args.device)\n'
                        '\n'
                        '    if args.ensemble:\n'
                        '        spec = json.load(open(args.ensemble))\n'
                        '        paths, scales = spec["members"], np.asarray(spec["scales"])\n'
                        '    else:\n'
                        '        paths = [p for c in args.ckpts for p in sorted(glob.glob(c))]\n'
                        '        scales = None\n'
                        '    cks = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]\n'
                        '    nets = [load_ckpt(p).eval().to(device) for p in paths]\n'
                        '    folds = [ck.get("exp", {}).get("fold") for ck in cks]\n'
                        '    if args.oof:\n'
                        '        assert all(f is not None for f in folds), "--oof needs members trained with a fold"\n'
                        '        n_folds = cks[0].get("n_folds") or max(folds) + 1\n'
                        '    print(f"{len(nets)} members" + (f", folds {sorted(set(folds))}" if args.oof else ""), '
                        'flush=True)\n'
                        '\n'
                        '    store = SeqStore(args.data, args.threads)\n'
                        '    n = store.n_seqs\n'
                        '    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.float16, shape=(n, 20000, '
                        '2))\n'
                        '    ids = list(range(args.limit or n))\n'
                        '    if args.limit:\n'
                        '        out[args.limit:] = 0\n'
                        '    t, done = time.time(), 0\n'
                        '    for X, _, _ in store.batches(ids, 128):\n'
                        '        cache = {}  # each member gets its own extra features, computed once per batch and '
                        'kind\n'
                        '        P = np.stack([predict_batch(net, net_inputs(net, X, cache), device) for net in '
                        'nets])  # [K, B, T, 2]\n'
                        '        if scales is None:  # common spread, fixed from the first batch\n'
                        '            sd = P[:, :, 99:].std(axis=(1, 2))  # [K, 2]\n'
                        '            scales = sd.mean(0, keepdims=True) / sd\n'
                        '        P = P * scales[:, None, None, :]\n'
                        '        if args.oof:\n'
                        '            batch_fold = np.arange(done, done + len(X)) % n_folds\n'
                        '            pred = np.zeros(P.shape[1:], np.float32)\n'
                        '            for f in range(n_folds):\n'
                        '                members = [k for k in range(len(nets)) if folds[k] == f]\n'
                        '                rows = batch_fold == f\n'
                        '                pred[rows] = P[members][:, rows].mean(0)\n'
                        '        else:\n'
                        '            pred = P.mean(0)\n'
                        '        out[done:done + len(X)] = pred.astype(np.float16)\n'
                        '        done += len(X)\n'
                        '        if done % 1280 == 0 or done == len(ids):\n'
                        '            print(f"{done}/{len(ids)} sequences, {time.time() - t:.0f}s", flush=True)\n'
                        '    out.flush()\n'
                        '    print(f"teacher ({len(nets)} models{\', out-of-fold\' if args.oof else \'\'}) -> '
                        '{args.out}")\n'
                        '\n'
                        '\n'
                        'if __name__ == "__main__":\n'
                        '    main()\n',
 'pipeline/test_export.py': '"""Check that exported ONNX step graphs + solution.py reproduce PyTorch predictions row '
                            'by row.\n'
                            '\n'
                            '    python -m pipeline.test_export            # random GRU / diff-GRU / LSTM / minGRU / '
                            'TCN / feature kinds / aux head /\n'
                            '                                              # group-conditioned GRU / ensembles / '
                            'routed submission\n'
                            '"""\n'
                            'import importlib.util\n'
                            'import os\n'
                            'import shutil\n'
                            'import json\n'
                            'import tempfile\n'
                            '\n'
                            'import numpy as np\n'
                            'import pyarrow.parquet as pq\n'
                            'import torch\n'
                            '\n'
                            'from utils import FEATURE_COLUMNS, WARMUP, DataPoint\n'
                            '\n'
                            'from .fast_eval import predict_batch\n'
                            'from .features import KINDS, N_EXTRA, detect\n'
                            'from .model import EnsembleNet, GRUNet, GroupGRUNet, MinGRUNet, TCNNet, export_onnx\n'
                            '\n'
                            'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                            '\n'
                            '\n'
                            'def run_rows(net, X):\n'
                            '    d = tempfile.mkdtemp()\n'
                            '    export_onnx(net, os.path.join(d, "model.onnx"))\n'
                            '    shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(d, '
                            '"solution.py"))\n'
                            '    shutil.copy(os.path.join(HERE, "features.py"), os.path.join(d, "wnn_features.py"))\n'
                            '    spec = importlib.util.spec_from_file_location("sol_test", os.path.join(d, '
                            '"solution.py"))\n'
                            '    mod = importlib.util.module_from_spec(spec)\n'
                            '    spec.loader.exec_module(mod)\n'
                            '    m = mod.PredictionModel()\n'
                            '    out = np.zeros((len(X), 2), np.float32)\n'
                            '    for seq in (0, 1):  # two sequences: also checks the state reset\n'
                            '        for t in range(len(X)):\n'
                            '            p = m.predict(DataPoint(seq, t, t >= WARMUP, X[t]))\n'
                            '            if p is not None:\n'
                            '                out[t] = p\n'
                            '    shutil.rmtree(d)\n'
                            '    return out\n'
                            '\n'
                            '\n'
                            'def run_routed(nets, route, fallback, threshold, X):\n'
                            '    """Routed submission (pipeline/solution_routed.py) row by row over two copies of '
                            'X."""\n'
                            '    d = tempfile.mkdtemp()\n'
                            '    files = []\n'
                            '    for k, net in enumerate(nets):\n'
                            '        files.append(f"model_{k}.onnx")\n'
                            '        export_onnx(net, os.path.join(d, files[-1]))\n'
                            '    json.dump({"models": files, "route": route, "fallback": fallback, "threshold": '
                            'threshold},\n'
                            '              open(os.path.join(d, "routes.json"), "w"))\n'
                            '    shutil.copy(os.path.join(HERE, "solution_routed.py"), os.path.join(d, '
                            '"solution.py"))\n'
                            '    shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(d, '
                            '"wnn_runner.py"))\n'
                            '    shutil.copy(os.path.join(HERE, "features.py"), os.path.join(d, "wnn_features.py"))\n'
                            '    spec = importlib.util.spec_from_file_location("sol_routed_test", os.path.join(d, '
                            '"solution.py"))\n'
                            '    mod = importlib.util.module_from_spec(spec)\n'
                            '    spec.loader.exec_module(mod)\n'
                            '    m = mod.PredictionModel()\n'
                            '    out = np.zeros((len(X), 2), np.float32)\n'
                            '    for seq in (0, 1):\n'
                            '        for t in range(len(X)):\n'
                            '            p = m.predict(DataPoint(seq, t, t >= WARMUP, X[t]))\n'
                            '            if p is not None:\n'
                            '                out[t] = p\n'
                            '    shutil.rmtree(d)\n'
                            '    return out\n'
                            '\n'
                            '\n'
                            'def main():\n'
                            '    torch.manual_seed(0)\n'
                            '    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, '
                            'columns=list(FEATURE_COLUMNS))\n'
                            '    X = np.column_stack([tbl[c].to_numpy() for c in '
                            'FEATURE_COLUMNS]).astype(np.float32)[:3000]\n'
                            '    a, b, c = GRUNet(96), GRUNet(64, use_diff=True), GRUNet(80, layers=3, cell="lstm")\n'
                            '    d, e = GRUNet(64, n_extra=N_EXTRA), GRUNet(48, use_diff=True, n_extra=N_EXTRA)\n'
                            '    # minGRU: 3000 rows cross several scan blocks and chunks; TCN 3**6 = 729-row '
                            'receptive field crosses chunks\n'
                            '    f, g = MinGRUNet(64, layers=3), MinGRUNet(48, layers=2, use_diff=True, '
                            'n_extra=N_EXTRA)\n'
                            '    t1, t2 = TCNNet(32, layers=6, kernel=3), TCNNet(24, layers=8, kernel=2, '
                            'use_diff=True, n_extra=N_EXTRA)\n'
                            '    nets = {"gru": a, "gru_diff": b, "lstm_3layer": c, "gru_feats": d, "gru_diff_feats": '
                            'e,\n'
                            '            "mingru": f, "mingru_diff_feats": g, "tcn_k3": t1, "tcn_k2_diff_feats": t2,\n'
                            '            "ensemble": EnsembleNet([a, b, c], np.array([[1.0, 1.1], [0.9, 1.2], [1.3, '
                            '0.7]])),\n'
                            '            "ensemble_feats": EnsembleNet([a, d, e], np.array([[1.0, 1.0], [0.8, 1.2], '
                            '[1.1, 0.9]])),\n'
                            '            "ensemble_mixed": EnsembleNet([a, f, t1, t2], np.array([[1.0, 1.0], [0.8, '
                            '1.2], [1.1, 0.9], [0.9, 1.0]]))}\n'
                            '    n1, n2 = GRUNet(64, n_extra=KINDS["norm"], feat_kind="norm"), GRUNet(48, '
                            'n_extra=KINDS["act+norm"], feat_kind="act+norm")\n'
                            '    n3 = GRUNet(56, use_diff=True, n_extra=KINDS["norm"], feat_kind="norm", n_aux=7)  # '
                            'aux head: not in the export\n'
                            '    nets.update({"gru_norm": n1, "gru_act+norm": n2, "gru_diff_norm_aux": n3,\n'
                            '                 "ensemble_norm": EnsembleNet([a, n1, n3], np.array([[1.0, 1.0], [0.8, '
                            '1.2], [1.1, 0.9]]))})\n'
                            '    base = GRUNet(64)  # group-conditioned GRU: starts as exactly the plain GRU it is '
                            'copied from\n'
                            '    gg = GroupGRUNet(64).from_gru(base.state_dict()).eval()\n'
                            '    same = np.abs(predict_batch(gg, X[None], torch.device("cpu"))[0] - '
                            'predict_batch(base.eval(), X[None], torch.device("cpu"))[0]).max()\n'
                            '    print(f"ggru from_gru vs gru: max diff {same:.2e}  {\'OK\' if same < 1e-5 else '
                            '\'MISMATCH\'}")\n'
                            '    gg2 = GroupGRUNet(48, layers=3)\n'
                            '    with torch.no_grad():\n'
                            '        gg2.film_scale.normal_(0, 0.3), gg2.film_shift.normal_(0, 0.3)\n'
                            '    nets.update({"ggru": gg2, "ensemble_ggru": EnsembleNet([n1, gg2], np.array([[1.0, '
                            '1.0], [0.8, 1.2]]))})\n'
                            '    nets = {k: v for k, v in nets.items() if k != "ensemble_ggru"}  # members with '
                            'different kinds: not exportable\n'
                            '    for name, net in nets.items():\n'
                            '        net.eval()\n'
                            '        ref = predict_batch(net, X[None], torch.device("cpu"), chunk=700)[0]\n'
                            '        got = run_rows(net, X)\n'
                            '        err = np.abs(ref[WARMUP:] - got[WARMUP:]).max()\n'
                            '        print(f"{name:18s} max |torch - onnx| = {err:.2e}  {\'OK\' if err < 1e-4 else '
                            '\'MISMATCH\'}")\n'
                            "    # routed submission: 3 models, the detector's group picks one; threshold above 1 -> "
                            'always the fallback\n'
                            '    g = int(detect(X[None]).argmax())\n'
                            '    members = [a, n1, gg2, EnsembleNet([a, gg2], np.ones((2, 2)))]  # last: a blend as '
                            'routed.py exports it\n'
                            '    refs = [predict_batch(net.eval(), X[None], torch.device("cpu"))[0] for net in '
                            'members]\n'
                            '    for route, fallback, thr in (([0, 1, 2, 1], 0, 0.0), ([3, 3, 3, 3], 1, 0.0), ([1, 2, '
                            '1, 2], 0, 1.01)):\n'
                            '        want = refs[fallback] if thr > 1 else refs[route[g]]\n'
                            '        got = run_routed(members, route, fallback, thr, X)\n'
                            '        err = np.abs(want[WARMUP:] - got[WARMUP:]).max()\n'
                            '        print(f"routed {route} fb {fallback} thr {thr}: group {g + 1}, max |torch - onnx| '
                            '= {err:.2e}  "\n'
                            '              f"{\'OK\' if err < 1e-4 else \'MISMATCH\'}")\n'
                            '\n'
                            '\n'
                            'if __name__ == "__main__":\n'
                            '    main()\n',
 'pipeline/train.py': '"""Train one stateful GRU (thin wrapper over trainer.train_many; see multi_train.py for '
                      'several).\n'
                      '\n'
                      '    python -m pipeline.train --name quick --train-seqs 1000 --epochs 2\n'
                      '    python -m pipeline.train --name gru128_wmse --epochs 4\n'
                      '\n'
                      'Outputs in runs/NAME/: best.pt, log.jsonl, results.json, submission/{solution.py, model.onnx}.\n'
                      '"""\n'
                      'import argparse\n'
                      'import dataclasses\n'
                      '\n'
                      'from .multi_train import add_common_flags, add_flags\n'
                      'from .trainer import Common, ExpConfig, train_many\n'
                      '\n'
                      '\n'
                      'def main():\n'
                      '    ap = argparse.ArgumentParser()\n'
                      '    shared = ("seed", "chunk")  # one flag each for a single model: used for the model and the '
                      'data stream\n'
                      '    exp_fields = {f.name for f in dataclasses.fields(ExpConfig)} - set(shared)\n'
                      '    ap.add_argument("--name", required=True)\n'
                      '    add_flags(ap, ExpConfig, skip=("name", *shared))\n'
                      '    add_common_flags(ap)\n'
                      '    args = vars(ap.parse_args())\n'
                      '    exp = ExpConfig(**{k: args.pop(k) for k in list(args) if k in exp_fields}, '
                      'seed=args["seed"])\n'
                      '    train_many([exp], Common(**args))\n'
                      '\n'
                      '\n'
                      'if __name__ == "__main__":\n'
                      '    main()\n',
 'pipeline/trainer.py': '"""Shared training loop: one data stream feeds any number of experiments.\n'
                        '\n'
                        'Every batch of whole sequences is read from parquet once and then used to train each model '
                        'in\n'
                        'turn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= '
                        '99).\n'
                        'Evaluation is shared the same way: one pass over the eval data scores all models.\n'
                        '\n'
                        'Optional per model: EMA of the weights (evaluated and saved instead of the raw weights),\n'
                        "distillation towards a teacher's predictions on the training set (see pipeline/teacher.py), "
                        'its own\n'
                        'number of passes, and a data fold to leave out (out-of-fold teachers: sequences with\n'
                        'id % n_folds == fold are skipped by that model), extra input features of a kind '
                        '(pipeline/features.py),\n'
                        'auxiliary targets (aux_targets) learned by a training-only head, one instrument group only '
                        '(`group`: the model\n'
                        "trains on that group's sequences and is selected on that group's validation rows), and a "
                        'starting checkpoint\n'
                        '(`init`).\n'
                        '"""\n'
                        'import copy\n'
                        'import json\n'
                        'import math\n'
                        'import os\n'
                        'import shutil\n'
                        'import time\n'
                        'from dataclasses import asdict, dataclass, field\n'
                        '\n'
                        'import numpy as np\n'
                        'import torch\n'
                        '\n'
                        'from utils import WARMUP\n'
                        '\n'
                        'from .data import SeqStore, split_train\n'
                        'from .groups import label as group_label\n'
                        'from .features import offline as extra_features, width as feat_width\n'
                        'from .fast_eval import evaluate_many\n'
                        'from .model import BASELINE_MACS, GRUNet, build_net, compute_loss, detach_state, '
                        'distill_loss, export_onnx, load_baseline, macs\n'
                        '\n'
                        'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                        'AUX_CHANGES = [(0, 10), (0, 50), (0, 200), (0, 1000), (1, 50), (1, 200)]  # (instrument, rows '
                        'ahead)\n'
                        "AUX_VOL = 100  # plus: mean |change| of i0's average price over the next AUX_VOL rows\n"
                        'N_AUX = len(AUX_CHANGES) + 1\n'
                        '\n'
                        '\n'
                        'def aux_targets(X):\n'
                        '    """Auxiliary targets from the raw inputs X [B,T,>=112] (torch): future change of each '
                        "instrument's average\n"
                        "    price column (i0: 0-21, i1: 52-73) at several horizons, and i0's near-future activity. -> "
                        'A, V [B,T,N_AUX]:\n'
                        '    targets scaled to unit spread over the batch (clipped to +-5) and where they exist (V, '
                        'rows with enough future)."""\n'
                        '    B, T = X.shape[:2]\n'
                        '    mids = [X[..., 0:22].mean(-1), X[..., 52:74].mean(-1)]\n'
                        '    A, V = X.new_zeros(B, T, N_AUX), torch.zeros(B, T, N_AUX, dtype=torch.bool, '
                        'device=X.device)\n'
                        '    for k, (i, h) in enumerate(AUX_CHANGES):\n'
                        '        A[:, :T - h, k] = mids[i][:, h:] - mids[i][:, :-h]\n'
                        '        V[:, :T - h, k] = True\n'
                        '    step = (mids[0][:, 1:] - mids[0][:, :-1]).abs()  # step[t] = |m[t+1] - m[t]|\n'
                        '    c = torch.cat([step.new_zeros(B, 1), step.cumsum(1)], 1)  # c[t] = sum of step[:t]\n'
                        '    n = T - AUX_VOL\n'
                        '    A[:, :n, -1] = (c[:, AUX_VOL:AUX_VOL + n] - c[:, :n]) / AUX_VOL\n'
                        '    V[:, :n, -1] = True\n'
                        '    for k in range(N_AUX):\n'
                        '        a = A[..., k][V[..., k]]\n'
                        '        A[..., k] = ((A[..., k] - a.mean()) / (a.std() + 1e-8)).clamp(-5, 5) * V[..., k]\n'
                        '    return A, V\n'
                        '\n'
                        '\n'
                        '@dataclass\n'
                        'class ExpConfig:\n'
                        '    """Per-model settings. Everything about the data stream is shared (see Common)."""\n'
                        '    name: str\n'
                        '    hidden: int = 128\n'
                        '    layers: int = 2\n'
                        '    diff: bool = False\n'
                        '    dropout: float = 0.0\n'
                        '    init_baseline: bool = False\n'
                        '    loss: str = "wmse"          # wmse | pearson | mix\n'
                        '    alpha: float = 0.1          # wmse weight in mix\n'
                        '    lr: float = 2e-3\n'
                        '    wd: float = 1e-4\n'
                        '    seed: int = 0\n'
                        '    cell: str = "gru"           # gru | lstm\n'
                        '    chunk: int | None = None    # TBPTT length for this model (default: Common.chunk)\n'
                        '    ema: float = 0.0            # >0: keep an EMA of the weights with this decay; eval/save '
                        'the EMA\n'
                        '    distill: float = 0.0        # weight of the teacher loss (0 = off, 1 = teacher only); '
                        'needs a teacher\n'
                        '    distill_weight: str = "y"   # teacher-loss row weights: y (|clipped target|, like the '
                        'metric) | none | teacher\n'
                        "    teacher: str | None = None  # this model's teacher .npy (default: Common.teacher)\n"
                        '    epochs: int | None = None   # passes for this model (default: Common.epochs)\n'
                        '    fold: int | None = None     # leave out training sequences with id % Common.n_folds == '
                        'fold\n'
                        '    mask_weight: float = 0.0    # lambda: row weight (1 - lambda) + lambda * p^mask_power / '
                        'mean(p^mask_power)\n'
                        '    mask_power: float = 1.0     # p = predicted p(scored) from Common.row_weights '
                        '(pipeline/mask_model.py)\n'
                        '    feats: bool | str = False   # extra causal input features: a kind from features.KINDS '
                        '(act | norm | act+norm);\n'
                        '                                # True = act (the 36 activity / EMA features)\n'
                        '    aux: float = 0.0            # >0: weight of the auxiliary-target loss (aux_targets; GRU '
                        'only, not exported)\n'
                        '    micro_batch: int | None = None  # sequences per optimizer step for this model (default: '
                        'the whole data batch)\n'
                        '    arch: str = "gru"           # gru (cell gru|lstm) | mingru | tcn (see pipeline/model.py)\n'
                        '    kernel: int = 2             # tcn only: kernel size; dilations kernel**l, receptive field '
                        'kernel**layers rows\n'
                        "    group: int | None = None    # 0..3: train only on this instrument group's sequences "
                        '(pipeline/groups.py)\n'
                        '    init: str | None = None     # start from this best.pt (same shape; a plain GRU also '
                        'starts an arch="ggru" model)\n'
                        '\n'
                        '\n'
                        '@dataclass\n'
                        'class Common:\n'
                        '    epochs: int = 4\n'
                        '    batch: int = 256            # sequences per batch\n'
                        '    chunk: int = 500            # TBPTT length\n'
                        '    clip_grad: float = 1.0\n'
                        '    train_seqs: int | None = None\n'
                        '    holdout: int = 500\n'
                        '    valid_seqs: int = 400\n'
                        '    evals_per_epoch: int = 2\n'
                        '    seed: int = 0               # data order + holdout split\n'
                        '    device: str = "auto"\n'
                        '    threads: int = 4            # parquet reader threads\n'
                        '    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)\n'
                        '    cache_train: bool = False   # keep training sequences in RAM after first read\n'
                        '    data_dir: str = "datasets"\n'
                        '    out_dir: str = "runs"\n'
                        '    teacher: str | None = None  # teacher predictions on train.parquet (.npy, [n_seqs, 20000, '
                        '2]) for distillation\n'
                        '    n_folds: int = 4            # for ExpConfig.fold\n'
                        '    row_weights: str | None = None  # p(scored) per train row (.npy [n_seqs, 20000]) for '
                        'ExpConfig.mask_weight\n'
                        '    valid_split: str | None = None  # split.json from mask_model: evaluate and select on '
                        'validation half B\n'
                        '    extra: dict = field(default_factory=dict)\n'
                        '\n'
                        '\n'
                        'class Experiment:\n'
                        '    def __init__(self, cfg: ExpConfig, common: Common, device, n_batches):\n'
                        '        self.cfg = cfg\n'
                        '        self.chunk = cfg.chunk or common.chunk\n'
                        '        self.epochs = cfg.epochs or common.epochs\n'
                        '        self.teacher_path = (cfg.teacher or common.teacher) if cfg.distill > 0 else None\n'
                        '        assert cfg.distill == 0 or self.teacher_path, f"{cfg.name}: distill > 0 needs a '
                        'teacher"\n'
                        '        self.micro = cfg.micro_batch or common.batch\n'
                        '        total_steps = (self.epochs * n_batches * math.ceil(common.batch / self.micro)  # '
                        'upper bound on steps\n'
                        '                       * math.ceil(20000 / self.chunk))\n'
                        '        self.kind = "act" if cfg.feats is True else (cfg.feats or None)\n'
                        '        n_extra = feat_width(self.kind)\n'
                        '        assert cfg.aux == 0 or cfg.arch == "gru", "auxiliary targets: GRU only"\n'
                        '        torch.manual_seed(cfg.seed)\n'
                        '        if cfg.arch == "ggru":\n'
                        '            assert self.kind == "grp", "arch ggru takes the group probabilities: '
                        'feats=\'grp\'"\n'
                        '        if cfg.init_baseline:\n'
                        '            assert cfg.arch == "gru" and not cfg.diff and cfg.hidden == 128 and cfg.layers == '
                        '2, "baseline shape is fixed"\n'
                        '            self.net = load_baseline("baseline/baseline.onnx")\n'
                        '        elif cfg.arch == "gru":\n'
                        '            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout, cfg.cell, '
                        'n_extra=n_extra,\n'
                        '                              feat_kind=self.kind, n_aux=N_AUX if cfg.aux > 0 else 0)\n'
                        '        else:\n'
                        '            self.net = build_net(dict(arch=cfg.arch, hidden=cfg.hidden, layers=cfg.layers, '
                        'use_diff=cfg.diff,\n'
                        '                                      dropout=cfg.dropout, n_extra=n_extra, '
                        'feat_kind=self.kind,\n'
                        '                                      **({"kernel": cfg.kernel} if cfg.arch == "tcn" else '
                        '{})))\n'
                        '        if cfg.init:\n'
                        '            ck = torch.load(cfg.init, map_location="cpu", weights_only=False)\n'
                        '            if cfg.arch == "ggru" and ck["config"].get("arch", "gru") == "gru":\n'
                        '                self.net.from_gru(ck["state_dict"])\n'
                        '            else:\n'
                        '                self.net.load_state_dict(ck["state_dict"])\n'
                        '        self.net.to(device)\n'
                        '        self.ema_net = copy.deepcopy(self.net) if cfg.ema > 0 else None\n'
                        '        self.n_updates = 0\n'
                        '        self.total_steps = total_steps\n'
                        '        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)\n'
                        '        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, '
                        'total_steps=total_steps,\n'
                        '                                                         pct_start=max(0.05, 3 / '
                        'total_steps))\n'
                        '        self.dir = os.path.join(common.out_dir, cfg.name)\n'
                        '        os.makedirs(self.dir, exist_ok=True)\n'
                        '        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")\n'
                        '        self.best = -1.0\n'
                        '        self.params = sum(p.numel() for p in self.net.parameters())\n'
                        '        self.cost = macs(self.net.config) / BASELINE_MACS\n'
                        '\n'
                        '    @property\n'
                        '    def eval_net(self):\n'
                        '        return self.ema_net if self.ema_net is not None else self.net\n'
                        '\n'
                        '    @torch.no_grad()\n'
                        '    def update_ema(self):\n'
                        '        self.n_updates += 1\n'
                        '        if self.ema_net is None:\n'
                        '            return\n'
                        '        decay = min(self.cfg.ema, (1 + self.n_updates) / (10 + self.n_updates))  # warm-up\n'
                        '        for pe, p in zip(self.ema_net.parameters(), self.net.parameters()):\n'
                        '            pe.lerp_(p, 1 - decay)\n'
                        '\n'
                        '    def log(self, **kw):\n'
                        '        line = json.dumps(kw)\n'
                        '        self.log_f.write(line + "\\n")\n'
                        '        self.log_f.flush()\n'
                        '        print(f"[{self.cfg.name}] {line}", flush=True)\n'
                        '\n'
                        '\n'
                        'def _store(common, name):\n'
                        '    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), '
                        'n_threads=common.threads)\n'
                        '\n'
                        '\n'
                        'def train_many(configs: list[ExpConfig], common: Common):\n'
                        '    from .fast_eval import pick_device\n'
                        '\n'
                        '    device = pick_device(common.device)\n'
                        '    if device.type == "cuda":\n'
                        '        torch.backends.cudnn.benchmark = True\n'
                        '    rng = np.random.default_rng(common.seed)\n'
                        '    train, valid = _store(common, "train"), _store(common, "valid")\n'
                        '    if common.cache_train:\n'
                        '        train.enable_cache()\n'
                        '    if common.cache_eval:\n'
                        '        valid.enable_cache()\n'
                        '    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)\n'
                        '    if common.train_seqs:\n'
                        '        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)\n'
                        '    valid_B = json.load(open(common.valid_split))["B"] if common.valid_split else None\n'
                        '    valid_ids = (valid_B or list(range(valid.n_seqs)))[:common.valid_seqs]\n'
                        '    hold_store = train\n'
                        '    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of '
                        'train\n'
                        '        hold_store = _store(common, "train")\n'
                        '        hold_store.enable_cache()\n'
                        '\n'
                        '    n_batches = math.ceil(len(train_ids) / common.batch)\n'
                        '    exps = [Experiment(c, common, device, n_batches) for c in configs]\n'
                        '    teachers = {}  # path -> memmap, loaded once and shared by the models that use it\n'
                        '    for path in sorted({e.teacher_path for e in exps if e.teacher_path}):\n'
                        '        teachers[path] = np.load(path, mmap_mode="r")\n'
                        '        assert teachers[path].shape[:2] == (train.n_seqs, 20000), f"teacher {path}: shape '
                        '{teachers[path].shape}"\n'
                        '    row_w, p_norm = None, {}\n'
                        '    if any(e.cfg.mask_weight > 0 for e in exps):\n'
                        '        assert common.row_weights, "mask_weight > 0 needs --row-weights (see '
                        'pipeline/mask_model.py)"\n'
                        '        row_w = np.load(common.row_weights, mmap_mode="r")\n'
                        '        assert row_w.shape == (train.n_seqs, 20000), f"row weights shape {row_w.shape}"\n'
                        '        sample = np.asarray(row_w[np.sort(np.asarray(train_ids))[::20], 99:], '
                        'dtype=np.float32)\n'
                        '        for pw in {e.cfg.mask_power for e in exps if e.cfg.mask_weight > 0}:\n'
                        '            p_norm[pw] = float((sample ** pw).mean())  # so the average row weight stays '
                        'about 1\n'
                        '            print(f"row weights: mean p^{pw:g} = {p_norm[pw]:.4f}", flush=True)\n'
                        '    n_epochs = max(e.epochs for e in exps)\n'
                        '    t0 = time.time()\n'
                        '    for e in exps:\n'
                        '        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), '
                        'params=e.params,\n'
                        '              cost=round(e.cost, 3), train_seqs=len(train_ids), holdout=len(hold_ids), '
                        'steps=e.total_steps)\n'
                        '    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)\n'
                        '\n'
                        '    def run_eval(epoch, bi, group):\n'
                        '        nets = [e.eval_net for e in group]\n'
                        '        vs = evaluate_many(nets, valid, valid_ids, device)\n'
                        '        hs = evaluate_many(nets, hold_store, hold_ids, device)\n'
                        '        for e, v, h in zip(group, vs, hs):\n'
                        "            # with a valid split: select on half B's scored rows (the mask model never saw "
                        'them);\n'
                        "            # otherwise don't select on the visible valid mask alone\n"
                        '            sel = v["scored"] if valid_B else (v["scored"] + h["all"]) / 2\n'
                        '            if e.cfg.group is not None and v[f"scored_g{e.cfg.group}"] == '
                        'v[f"scored_g{e.cfg.group}"]:\n'
                        '                sel = v[f"scored_g{e.cfg.group}"]  # a group model is selected on its '
                        "group's rows (if any)\n"
                        '            is_best = sel > e.best\n'
                        '            if is_best:\n'
                        '                e.best = sel\n'
                        '                torch.save({"config": e.net.config, "state_dict": e.eval_net.state_dict(), '
                        '"exp": asdict(e.cfg),\n'
                        '                            "n_folds": common.n_folds},\n'
                        '                           os.path.join(e.dir, "best.pt"))\n'
                        '            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), '
                        'valid_all=round(v["all"], 6),\n'
                        '                  valid_groups=[round(v[f"scored_g{g}"], 4) for g in range(4)],\n'
                        '                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, '
                        'time=round(time.time() - t0, 1))\n'
                        '            e.net.train()\n'
                        '\n'
                        '    for epoch in range(n_epochs):\n'
                        '        active = [e for e in exps if epoch < e.epochs]\n'
                        '        for e in active:\n'
                        '            e.net.train()\n'
                        '        ids = np.asarray(rng.permutation(train_ids))\n'
                        '        for bi, (X, Y, _) in enumerate(train.batches(list(ids), common.batch)):\n'
                        '            tb = time.time()\n'
                        '            batch_ids = ids[bi * common.batch:(bi + 1) * common.batch]\n'
                        '            groups = group_label(Y)\n'
                        '            Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy '
                        'per batch\n'
                        '            # extras once per batch and kind, shared by every model of that kind, joined on '
                        'the GPU (the joined\n'
                        '            # 256-seq batch would be another 3+ GB array in host RAM)\n'
                        '            Xk = {None: Xd, **{k: torch.cat([Xd, torch.from_numpy(extra_features(X, '
                        'k)).to(device)], -1)\n'
                        '                               for k in sorted({e.kind for e in active} - {None})}}\n'
                        '            Ad = Vd = None\n'
                        '            if any(e.cfg.aux > 0 for e in active):\n'
                        '                Ad, Vd = aux_targets(Xd)\n'
                        '            Yd = torch.from_numpy(Y).to(device, non_blocking=True)\n'
                        '            order = torch.as_tensor(np.argsort(np.argsort(batch_ids)), device=device)  # '
                        'memmap reads in file order\n'
                        '            Td = {path: torch.from_numpy(np.asarray(t[np.sort(batch_ids)], '
                        'dtype=np.float32)).to(device)[order]\n'
                        '                  for path, t in teachers.items() if any(e.teacher_path == path for e in '
                        'active)}\n'
                        '            Pd = None\n'
                        '            if row_w is not None and any(e.cfg.mask_weight > 0 for e in active):\n'
                        '                Pd = torch.from_numpy(np.asarray(row_w[np.sort(batch_ids)], '
                        'dtype=np.float32)).to(device)[order]\n'
                        '            for e in active:\n'
                        '                Xe, Ye, Te = Xk[e.kind], Yd, Td.get(e.teacher_path)\n'
                        '                Ae, Ve = (Ad, Vd) if e.cfg.aux > 0 else (None, None)\n'
                        '                We = None\n'
                        '                if e.cfg.mask_weight > 0:\n'
                        '                    lam, pw = e.cfg.mask_weight, e.cfg.mask_power\n'
                        '                    We = (1 - lam) + lam * Pd.clamp(min=0) ** pw / p_norm[pw]\n'
                        '                keep = np.ones(len(batch_ids), bool)\n'
                        "                if e.cfg.fold is not None:  # out-of-fold model: drop its fold's sequences "
                        'from the batch\n'
                        '                    keep &= batch_ids % common.n_folds != e.cfg.fold\n'
                        "                if e.cfg.group is not None:  # group model: only its group's sequences\n"
                        '                    keep &= groups == e.cfg.group\n'
                        '                if not keep.any():\n'
                        '                    continue\n'
                        '                if not keep.all():\n'
                        '                    rows = torch.as_tensor(np.flatnonzero(keep), device=device)\n'
                        '                    Xe, Ye = Xe[rows], Yd[rows]\n'
                        '                    Ae, Ve = (Ae[rows], Ve[rows]) if Ae is not None else (None, None)\n'
                        '                    Te = Te[rows] if Te is not None else None\n'
                        '                    We = We[rows] if We is not None else None\n'
                        '                losses = []\n'
                        '                for m0 in range(0, len(Xe), e.micro):  # mini-batches of whole sequences, '
                        'each its own TBPTT pass\n'
                        '                    Xm, Ym = Xe[m0:m0 + e.micro], Ye[m0:m0 + e.micro]\n'
                        '                    Tm = Te[m0:m0 + e.micro] if Te is not None else None\n'
                        '                    Wm = We[m0:m0 + e.micro] if We is not None else None\n'
                        '                    Am, Vm = (Ae[m0:m0 + e.micro], Ve[m0:m0 + e.micro]) if Ae is not None '
                        'else (None, None)\n'
                        '                    h = xp = None\n'
                        '                    for s in range(0, X.shape[1], e.chunk):\n'
                        '                        x, y = Xm[:, s:s + e.chunk], Ym[:, s:s + e.chunk]\n'
                        '                        p, h = e.net(x, h, xp)\n'
                        '                        lo = max(0, WARMUP - s)  # skip warm-up rows\n'
                        '                        if lo < x.shape[1]:\n'
                        '                            pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)\n'
                        '                            rw = Wm[:, s:s + e.chunk][:, lo:].reshape(-1) if Wm is not None '
                        'else None\n'
                        '                            lam = e.cfg.distill\n'
                        '                            loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha, rw) if lam < '
                        '1 else 0.0\n'
                        '                            if lam > 0:\n'
                        '                                tf = Tm[:, s:s + e.chunk][:, lo:].reshape(-1, 2)\n'
                        '                                loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, '
                        'e.cfg.distill_weight, rw)\n'
                        '                            if Am is not None:  # auxiliary targets on the same rows, from '
                        'the training-only head\n'
                        '                                pa = e.net.aux_head(e.net.last_out[:, lo:])\n'
                        '                                ta, va = Am[:, s:s + e.chunk][:, lo:], Vm[:, s:s + '
                        'e.chunk][:, lo:]\n'
                        '                                loss = loss + e.cfg.aux * ((pa - ta) ** 2 * va).sum() / '
                        'va.sum().clamp(min=1)\n'
                        '                            e.opt.zero_grad(set_to_none=True)\n'
                        '                            loss.backward()\n'
                        '                            torch.nn.utils.clip_grad_norm_(e.net.parameters(), '
                        'common.clip_grad)\n'
                        '                            e.opt.step()\n'
                        '                            e.sched.step()\n'
                        '                            e.update_ema()\n'
                        '                            losses.append(loss.detach())  # no GPU sync per step\n'
                        '                        h = detach_state(h)\n'
                        '                        xp = x[:, -1]\n'
                        '                e.log(event="batch", epoch=epoch, batch=bi, '
                        'loss=round(float(torch.stack(losses).mean()), 5),\n'
                        '                      lr=round(e.sched.get_last_lr()[0], 6), steps=len(losses))\n'
                        '            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{n_epochs}: {time.time() - '
                        'tb:.1f}s "\n'
                        '                  f"for {len(active)} models, elapsed {(time.time() - t0) / 60:.1f} min", '
                        'flush=True)\n'
                        '            del Xd, Xk, Yd, Td, Pd, Ad, Vd\n'
                        '            if bi in eval_at:\n'
                        '                run_eval(epoch, bi, active)\n'
                        '\n'
                        '    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary '
                        'table\n'
                        '    for e in exps:\n'
                        '        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", '
                        'weights_only=False)\n'
                        '        e.net.load_state_dict(ck["state_dict"])\n'
                        '        e.net.to(device)\n'
                        '    nets = [e.net for e in exps]\n'
                        '    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # '
                        'uncached: 17 GB\n'
                        '    full_hold = evaluate_many(nets, hold_store, hold_ids, device)\n'
                        '    full_B = evaluate_many(nets, _store(common, "valid"), valid_B, device) if valid_B else '
                        '[None] * len(exps)\n'
                        '    rows = []\n'
                        '    for e, v, h, b in zip(exps, full_valid, full_hold, full_B):\n'
                        '        sub = os.path.join(e.dir, "submission")\n'
                        '        os.makedirs(sub, exist_ok=True)\n'
                        '        export_onnx(e.net, os.path.join(sub, "model.onnx"))\n'
                        '        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, '
                        '"solution.py"))\n'
                        '        shutil.copy(os.path.join(HERE, "features.py"), os.path.join(sub, "wnn_features.py"))\n'
                        '        res = {"valid_full": v, "holdout": h, **({"valid_B": b} if b else {})}\n'
                        '        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)\n'
                        '        e.log(event="done", **res)\n'
                        '        rows.append({"name": e.cfg.name, "params": e.params, "cost": round(e.cost, 2), '
                        '"valid_scored": round(v["scored"], 6),\n'
                        '                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),\n'
                        '                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4),\n'
                        '                     **({"valid_B_scored": round(b["scored"], 6), "valid_B_all": '
                        'round(b["all"], 6),\n'
                        '                         **{f"B_g{g + 1}": round(b[f"scored_g{g}"], 4) for g in range(4)}} if '
                        'b else {}),\n'
                        '                     **({"group": e.cfg.group + 1} if e.cfg.group is not None else {})})\n'
                        '    rows.sort(key=lambda r: -r["valid_B_scored"] if "valid_B_scored" in r else '
                        '-(r["valid_scored"] + r["holdout_all"]))\n'
                        '    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:\n'
                        '        for r in rows:\n'
                        '            f.write(json.dumps(r) + "\\n")\n'
                        '    print("\\n" + " | ".join(rows[0].keys()))\n'
                        '    for r in rows:\n'
                        '        print(" | ".join(str(v) for v in r.values()))\n'
                        '    return rows\n'}
os.makedirs('/content/wnn/pipeline', exist_ok=True)
for rel, body in FILES.items():
    open(f'/content/wnn/{rel}', 'w').write(body)
sys.path.insert(0, '/content/wnn')
for m in [m for m in sys.modules if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[m]  # a previous run in this session may have loaded older code
from pipeline.trainer import ExpConfig
from pipeline.features import KINDS
assert 'group' in ExpConfig.__dataclass_fields__ and 'init' in ExpConfig.__dataclass_fields__ and 'grp' in KINDS
print(len(FILES), 'pipeline files written; feature kinds', KINDS, flush=True)
sh('cd /content/wnn && python -m pipeline.fast_eval --baseline --valid-seqs 300')

CACHE = f'{OUT_DIR}/cache'
ROWW, SPLIT = '/content/wnn/mask_train.npy', f'{OUT_DIR}/mask_model/split.json'
S9 = f'{OUT_DIR}/sweep9/s9_h128/best.pt'
for need in (S9, f'{CACHE}/t_9.npy', f'{CACHE}/mask_train.npy', SPLIT):
    assert os.path.exists(need), f'missing in Drive: {need} (made by sweeps 5-9)'
sh(f'cp {CACHE}/mask_train.npy {ROWW} && ls -la {ROWW}')
T9 = '/content/wnn/t_9.npy'
if not os.path.exists(T9):
    sh(f'cp {CACHE}/t_9.npy {T9}')
sh('cd /content/wnn && python -m pipeline.groups --check')  # group detector on all validation sequences
FLAGS = f'--evals-per-epoch 1 --threads 8 --cache-eval --row-weights {ROWW} --valid-split {SPLIT}'
import pandas as pd

def train(experiments, out, epochs):
    """Train what has no results.json in `out` yet (re-runnable)."""
    done = [e['name'] for e in experiments if os.path.exists(f"{out}/{e['name']}/results.json")]
    todo = [e for e in experiments if e['name'] not in done]
    print(f"\n{len(todo)} to train in {out}: {[e['name'] for e in todo]}" + (f"; already done: {done}" if done else ''), flush=True)
    if not todo:
        return
    drive_writable()
    json.dump(todo, open('experiments.json', 'w'), indent=1)
    sh(f"""set -o pipefail; cd /content/wnn && python -m pipeline.multi_train --config experiments.json --epochs {epochs} {FLAGS} --out-dir {out} 2>&1 | grep -v '"event": "batch"'""")

def table(names, dirs):
    rows = [json.loads(l) for d in dirs if os.path.exists(f'{OUT_DIR}/{d}/summary.jsonl') for l in open(f'{OUT_DIR}/{d}/summary.jsonl')]
    df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
    df = df[df.name.isin(names)].sort_values('valid_B_scored', ascending=False)
    cols = ['name', 'cost', 'valid_B_scored'] + [c for c in ('B_g1', 'B_g2', 'B_g3', 'B_g4', 'group') if c in df] + ['valid_B_all', 'holdout_all']
    print(df[cols].to_string(index=False), flush=True)

# ---- Phase A: per-group fine-tunes of the best student, the group-conditioned model, group 4 from scratch
def ft(name, **kw):  # fine-tune of s9_h128 (same shape), 6 passes at a lower peak lr, half teacher / half targets
    return {'name': name, 'init': S9, 'lr': 5e-4, 'epochs': 6, 'distill': 0.5, 'teacher': T9, 'mask_weight': 0.5, **kw}
def g4y(name, **kw):  # group 4 only, true targets only, 5 passes: the recipe of the sweep 9 teachers
    return {'name': name, 'group': 3, 'epochs': 5, 'mask_weight': 0.5, **kw}
PHASE_A = [ft('ft_all'), ft('ft_g1', group=0), ft('ft_g2', group=1), ft('ft_g3', group=2), ft('ft_g4', group=3),
           ft('ft_film', arch='ggru', feats='grp'),
           g4y('g4y_h128'), g4y('g4y_h192_drop', hidden=192, dropout=0.1)]
SWEEP = f'{OUT_DIR}/sweep10a'
train(PHASE_A, SWEEP, epochs=6)
sync_up()
table([e['name'] for e in PHASE_A] + ['s9_h128'], ['sweep9', 'sweep10a'])
print('B_g1..B_g4 = scored WP on each group of half B; group models are only meaningful on their own group. '
      'Reference s9_h128 half B 0.67824', flush=True)

# ---- Phase B: routing (per group, the best model on half A), routed submission, zips, speed
ROUTED = f'{OUT_DIR}/sweep10a_routed/submission'
drive_writable()
REFS = ' '.join(f'"{OUT_DIR}/sweep9_teachers/{n}/best.pt"' for n in   # shared models trained like g4y_* (5 passes, no teacher)
                ['t9_h128_s0', 't9_h128_s1', 't9_h128_s2', 't9_h128_s3', 't9_h192_drop_s0', 't9_h192_drop_s1'])
sh(f'cd /content/wnn && python -m pipeline.routed --shared {S9} --cands "{SWEEP}/*/best.pt" {REFS} --valid-split {SPLIT} '
   f'--export {ROUTED} 2>&1 | grep -vE "Warning|_generic_rnn|warnings.warn"')
os.makedirs(f'{OUT_DIR}/zips', exist_ok=True)
print(shutil.make_archive(f'{OUT_DIR}/zips/sweep10a_routed', 'zip', ROUTED), flush=True)
for n in ('ft_all', 'ft_film'):  # single models that could also win on their own
    d = f'{SWEEP}/{n}/submission'
    if os.path.isdir(d): print(shutil.make_archive(f'{OUT_DIR}/zips/sweep10a_{n}', 'zip', d), flush=True)
sync_up()  # zips and the routed submission to Drive before the speed test
print('decision (PLAN_sweep10.md): routed, detected groups, valid_B >= 0.6792 -> submit sweep10a_routed.zip. '
      'Group 4 learnable? compare B_g4 of g4y_h128 with t9_h128_s0..s3 and g4y_h192_drop with t9_h192_drop_s0/s1', flush=True)
sh(f'cd /content/wnn && python -m pipeline.speed {ROUTED} {OUT_DIR}/sweep9/s9_h128/submission {SWEEP}/ft_film/submission')  # grader ~ 0.52 x this
sync_up()
print(f"done; results in Drive: {globals().get('REMOTE', OUT_DIR)}", flush=True)
